# Patient-level DINOv2 PET representation training: frozen baseline, supervised frozen classifier, DINO-SSL and DINO-JEPA branches

This patched notebook keeps the slice-level normalisation mitigation and adds an ablation-ready branch structure:

- **Branch A — `frozen_dino_baseline`**: no training; extract patient embeddings from frozen DINOv2 + patient pooling.
- **Branch B — `supervised_frozen_classifier`**: supervised linear/classification head on top of a frozen DINOv2 patient encoder.
- **Branch C — `dino_ssl`**: existing DINO-style student-teacher patient-level SSL.
- **Branch D — `dino_jepa_slice`**: JEPA-style latent prediction from masked PET slice-triplets.
- **Branch E — `dino_jepa_cross_slice`**: JEPA-style latent prediction of a held-out slice representation from neighbouring context slices.

The branch list is controlled by `CFG.branches_to_run`. For a quick run, start with Branch A and Branch B. For a full ablation, include all five branches.


In [ ]:

import os
import sys
import math
import copy
import requests
import time
import json
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


## Environment compatibility check

The official DINOv2 repository documents that its training/evaluation stack expects **PyTorch 2.0**, **xFormers 0.0.18**, and a **Linux** environment for the tested training path.

Because you are using the pretrained backbone locally rather than reproducing Meta-scale pretraining from scratch, this notebook is more forgiving than the full repo training stack, but it is still best to keep:

- PyTorch / torchvision versions compatible with each other
- the local DINOv2 repo on disk if using `source="local"`
- the official checkpoint available locally if using `manual_local_weights`

In [ ]:

# -----------------------------
# Environment check
# -----------------------------
import platform
import importlib

def safe_version(pkg_name):
    try:
        mod = importlib.import_module(pkg_name)
        return getattr(mod, "__version__", "unknown")
    except Exception as e:
        return f"NOT_IMPORTABLE ({e})"

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("PyTorch     :", safe_version("torch"))
print("torchvision :", safe_version("torchvision"))
print("nibabel     :", safe_version("nibabel"))
print("sklearn     :", safe_version("sklearn"))

if isinstance(safe_version("torchvision"), str) and safe_version("torchvision").startswith("NOT_IMPORTABLE"):
    print("\n[Warning] torchvision is not importing cleanly in this environment.")
    print("Please fix the local torch/torchvision installation before running the notebook.")

## Local setup notes

This notebook now defaults to the **same loading pattern as your working code**:

1. directly construct the DINOv2 ViT backbone from `dinov2.models.vision_transformer`
2. load the official pretrained `.pth` weights manually
3. then run patient-level DINO-style SSL on top

This avoids reliance on `torch.hub.load(..., source="local")`, which is fragile in local/forked DINOv2 checkouts.

### Recommended local run
Set:
```python
CFG.backbone_source = "manual_local_weights"
CFG.dinov2_repo_dir = "/mnt/c/Users/uqrvashi/Downloads/DINOv2ForRadiology-main"
CFG.backbone_name = "dinov2_vitb14_reg"
CFG.manual_checkpoint_path = None  # auto-download official checkpoint if missing
```

If you already have the checkpoint downloaded, set `CFG.manual_checkpoint_path` explicitly.

### Optional alternatives
- `CFG.backbone_source = "local_hub"` only if your local repo really exposes the needed hub entrypoint.
- `CFG.backbone_source = "remote_hub"` if you want to pull from the official online Hub.
- `CFG.backbone_source = "mock_smoke"` only for a very fast structural smoke test.


In [ ]:

# -----------------------------
# Reproducibility / device
# -----------------------------
SEED = 1024
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
@dataclass
class Config:
    # -----------------------------
    # Paths
    # -----------------------------
    data_root: str = "/mnt/d/data/data_ssl_patient"
    train_dir_name: str = "train"
    val_dir_name: str = "val"
    oac_dir_name: str = "oac"

    output_root: str = "/mnt/d/data/patient_dinov2_ssl_outputs_branches_7"

    # -----------------------------
    # DINOv2 loading
    # -----------------------------
    backbone_source: str = "manual_local_weights"   # recommended: direct ctor + manual pretrained weights
    dinov2_repo_dir: str = "/mnt/c/Users/uqrvashi/Downloads/DINOv2ForRadiology-main"
    backbone_name: str = "dinov2_vitb14_reg"
    manual_checkpoint_path: Optional[str] = None
    checkpoint_cache_dir: str = "./pretrained_dinov2_weights"
    auto_download_checkpoint: bool = True
    use_register_backbone: bool = True
    backbone_native_img_size: int = 518   # build pretrained DINOv2 at native checkpoint size; training views can still be smaller

    # -----------------------------
    # Image / bag settings
    # -----------------------------
    # Conservative defaults to avoid OOM with DINOv2-B + student/teacher SSL.
    base_resize: int = 336
    pad_multiple: int = 14
    bag_size_train: int = 6
    bag_size_eval: int = 8
    n_global_crops: int = 2
    n_local_crops: int = 0               # keep 0 for the diagnostic/stabilisation run; increase later only after stable losses
    global_crop_scale: Tuple[float, float] = (0.85, 1.0)  # slightly less aggressive than 0.80 for small medical datasets
    local_crop_scale: Tuple[float, float] = (0.60, 0.85)

    # -----------------------------
    # Slice-level normalization
    # -----------------------------
    # Use "slice" for the current mitigation: each axial slice is clipped/scaled independently.
    # Keep "patient" available only for ablation/reproducibility.
    normalization_scope: str = "patient"  # options: "slice", "patient"
    clip_percentiles: Tuple[float, float] = (1.0, 99.0)

    # -----------------------------
    # Optimization: patched conservative defaults
    # -----------------------------
    train_batch_size: int = 1
    eval_batch_size: int = 1
    num_workers: int = 2
    epochs: int = 50
    lr: float = 1e-5                     # reduced from 1e-4 to avoid late divergence on small patient-level SSL
    min_lr: float = 1e-6
    lr_warmup_epochs: int = 5
    use_lr_schedule: bool = True
    weight_decay: float = 1e-2
    final_weight_decay: float = 1e-2
    grad_clip_norm: float = 1.0
    grad_accum_steps: int = 4            # now implemented across batches; improves effective batch for DINO centre statistics
    freeze_last_layer_epochs: int = 1    # DINO-style stabilisation for the projection head

    # -----------------------------
    # Early stopping / validation hygiene
    # -----------------------------
    early_stopping_patience: int = 25
    early_stopping_min_delta: float = 1e-4
    stop_on_nonfinite_loss: bool = True
    update_center_on_val: bool = False   # critical: validation must not mutate the DINO centre buffer
    deterministic_val_ssl: bool = True   # use deterministic validation views so val loss is interpretable

    # -----------------------------
    # Memory / precision
    # -----------------------------
    use_amp: bool = True
    amp_dtype: str = "bfloat16"          # "bfloat16" preferred on recent GPUs; fallback to float16 automatically
    empty_cache_each_epoch: bool = True
    frozen_backbone_chunk_size: int = 4
    eval_chunk_size: int = 4

    # -----------------------------
    # Backbone tuning
    # -----------------------------
    # Stage 1: head/pool-only SSL on pretrained DINOv2 features.
    # Stage 2 only after stable curves: unfreeze_last_n_blocks = 1, lr <= 1e-5.
    freeze_backbone: bool = False
    unfreeze_last_n_blocks: int = 1

    # -----------------------------
    # DINO-style SSL head
    # -----------------------------
    head_out_dim: int = 8192
    head_hidden_dim: int = 2048
    head_bottleneck_dim: int = 256
    head_nlayers: int = 4
    norm_last_layer: bool = True

    # -----------------------------
    # DINO temperatures / EMA
    # -----------------------------
    student_temp: float = 0.1
    warmup_teacher_temp: float = 0.04
    teacher_temp: float = 0.06           # slightly lower than 0.07 for small-batch medical SSL
    warmup_teacher_temp_epochs: int = 10
    center_momentum: float = 0.95        # smoother centre update than 0.9 for tiny effective batches
    momentum_teacher_base: float = 0.998 # slower teacher drift than 0.996

    # -----------------------------
    # KoLeo regularizer
    # -----------------------------
    use_koleo: bool = False              # enable only when effective batch > 2 and curves are stable
    koleo_weight: float = 0.01

    # -----------------------------
    # Diagnostic thresholds
    # -----------------------------
    diagnostic_n_patients: int = 4
    collapse_entropy_floor_fraction: float = 0.15  # warn if entropy < this fraction of log(head_out_dim)
    collapse_std_floor: float = 1e-3

    # -----------------------------
    # Progress / logging
    # -----------------------------
    show_batch_progress: bool = True
    progress_update_every: int = 1
    show_cuda_memory: bool = True
    print_first_batch_shapes: bool = True

    # -----------------------------
    # Clustering / export
    # -----------------------------
    n_clusters: int = 4
    n_attention_slices_to_plot: int = 4

    # -----------------------------
    # Data balancing
    # -----------------------------
    balance_patients_across_classes: bool = True


    # -----------------------------
    # Branch controller
    # -----------------------------
    # Available branches:
    #   "frozen_dino_baseline"         : no training; frozen DINOv2 patient embeddings only
    #   "supervised_frozen_classifier" : supervised classification head on frozen DINOv2 features
    #   "dino_ssl"                     : DINO-style student-teacher patient SSL
    #   "dino_jepa_slice"              : JEPA-style masked slice-triplet latent prediction
    #   "dino_jepa_cross_slice"        : JEPA-style cross-slice latent prediction
    # For the fastest diagnostic run, use only ("frozen_dino_baseline", "supervised_frozen_classifier").
    branches_to_run: Tuple[str, ...] = (
        "dino_ssl",
        "dino_jepa_cross_slice"

    )

    # -----------------------------
    # Branch B: supervised frozen-DINO classifier
    # -----------------------------
    supervised_epochs: int = 26
    supervised_lr: float = 1e-3
    supervised_min_lr: float = 1e-5
    supervised_weight_decay: float = 1e-4
    supervised_dropout: float = 0.20
    supervised_grad_accum_steps: int = 4
    supervised_early_stopping_patience: int = 15
    supervised_early_stopping_min_delta: float = 1e-4
    supervised_train_pool: bool = False       # train patient attention pooling + classifier; DINOv2 remains frozen
    supervised_use_class_weights: bool = False

    # -----------------------------
    # Branches D/E: DINO-JEPA-style latent prediction
    # -----------------------------
    jepa_epochs: int = 50
    jepa_lr: float = 3e-4
    jepa_min_lr: float = 1e-5
    jepa_weight_decay: float = 1e-4
    jepa_grad_accum_steps: int = 4
    jepa_ema_momentum: float = 0.996
    jepa_predictor_hidden_dim: int = 2048
    jepa_dropout: float = 0.10
    jepa_loss: str = "cosine"               # "cosine" or "smooth_l1"
    jepa_num_mask_blocks: int = 2
    jepa_slice_mask_scale: Tuple[float, float] = (0.15, 0.35)
    jepa_fill_value: float = 0.0             # zero in normalised tensor space is approximately ImageNet-mean fill
    jepa_freeze_backbone: bool = False        # start with frozen DINOv2; later try False + unfreeze_last_n_blocks=1
    jepa_unfreeze_last_n_blocks: int = 1
    jepa_patient_consistency_weight: float = 0.05

CFG = Config()
OUT_DIR = Path(CFG.output_root)
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "artifacts").mkdir(parents=True, exist_ok=True)

print(CFG)


## Branch controller

Use `CFG.branches_to_run` to decide which branches execute when the notebook is run top-to-bottom. The code keeps each branch independent and saves outputs under `patient_dinov2_ssl_outputs/artifacts/branches/<branch_name>/`.


In [ ]:

# -----------------------------
# Branch controller utilities
# -----------------------------
SUPPORTED_BRANCHES = (
    "frozen_dino_baseline",
    "supervised_frozen_classifier",
    "dino_ssl",
    "dino_jepa_slice",
    "dino_jepa_cross_slice",
)

BRANCH_ROOT = OUT_DIR / "artifacts" / "branches"
BRANCH_ROOT.mkdir(parents=True, exist_ok=True)

def _normalise_branch_names(branches):
    if branches is None:
        return set()
    if isinstance(branches, str):
        branches = (branches,)
    branches = tuple(str(b).strip() for b in branches if str(b).strip())
    if "all" in branches:
        return set(SUPPORTED_BRANCHES)
    unknown = sorted(set(branches) - set(SUPPORTED_BRANCHES))
    if unknown:
        raise ValueError(f"Unknown branch names: {unknown}. Supported: {SUPPORTED_BRANCHES}")
    return set(branches)

ACTIVE_BRANCHES = _normalise_branch_names(CFG.branches_to_run)

def branch_enabled(name: str) -> bool:
    return name in ACTIVE_BRANCHES

print("Active branches:")
for b in SUPPORTED_BRANCHES:
    print(f"  {b}: {branch_enabled(b)}")


## Patch: slice-level normalization active

This version shifts intensity preprocessing from patient-volume normalization back to **slice-level percentile normalization**. The patient-level SSL architecture is unchanged: slices are still sampled as a patient bag and pooled into one patient embedding, but each 2D slice is clipped/scaled independently before neighbouring-slice triplets are created.

Use `CFG.normalization_scope = "slice"` for the mitigation run. Set it to `"patient"` only for an ablation against the previous behaviour.


In [ ]:

# -----------------------------
# Path resolution
# -----------------------------
DATA_ROOT = Path(CFG.data_root)
TRAIN_DIR = DATA_ROOT / CFG.train_dir_name
VAL_DIR   = DATA_ROOT / CFG.val_dir_name
OAC_DIR   = DATA_ROOT / CFG.oac_dir_name
OUT_DIR   = Path(CFG.output_root)

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "plots").mkdir(exist_ok=True, parents=True)
(OUT_DIR / "artifacts").mkdir(exist_ok=True, parents=True)
(OUT_DIR / "checkpoints").mkdir(exist_ok=True, parents=True)

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR  :", VAL_DIR)
print("OAC_DIR  :", OAC_DIR)
print("OUT_DIR  :", OUT_DIR)

In [ ]:

# -----------------------------
# Helper utilities
# -----------------------------
IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD  = (0.229, 0.224, 0.225)
VALID_EXTS = (".nii", ".nii.gz")

def is_nifti_file(path: Path) -> bool:
    s = path.name.lower()
    return s.endswith(".nii") or s.endswith(".nii.gz")

def strip_nii_suffix(name: str) -> str:
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return Path(name).stem

def _percentile_to_uint8(x: np.ndarray, p_low=1.0, p_high=99.0) -> np.ndarray:
    """Clip finite values by percentiles and rescale to uint8 [0, 255]."""
    x = np.asarray(x, dtype=np.float32)
    out = np.zeros_like(x, dtype=np.uint8)
    finite = np.isfinite(x)
    if not finite.any():
        return out

    vals = x[finite]
    lo, hi = np.percentile(vals, [p_low, p_high])
    if not np.isfinite(lo):
        lo = np.min(vals)
    if not np.isfinite(hi):
        hi = np.max(vals)

    if hi <= lo:
        return out

    y = np.clip(x, lo, hi)
    y = (y - lo) / (hi - lo + 1e-8)
    y = np.clip(y, 0.0, 1.0)
    out[finite] = (y[finite] * 255.0).astype(np.uint8)
    return out


def patient_level_percentile_normalize(volume: np.ndarray, p_low=1.0, p_high=99.0) -> np.ndarray:
    """Legacy/ablation option: one percentile range for the whole patient volume."""
    return _percentile_to_uint8(volume, p_low=p_low, p_high=p_high)


def slice_level_percentile_normalize(volume: np.ndarray, p_low=1.0, p_high=99.0) -> np.ndarray:
    """
    Current mitigation: normalize each axial slice independently.

    Rationale:
    - the SSL bag contains multiple lesion slices from the same patient;
    - patient-level clipping can let a single high-uptake slice dominate the dynamic range;
    - slice-level clipping preserves within-slice lesion contrast before DINO augmentations.
    """
    vol = np.asarray(volume, dtype=np.float32)
    if vol.ndim != 3:
        raise ValueError(f"Expected 3D volume for slice-level normalization, got shape {vol.shape}")
    out = np.zeros_like(vol, dtype=np.uint8)
    for z in range(vol.shape[2]):
        out[:, :, z] = _percentile_to_uint8(vol[:, :, z], p_low=p_low, p_high=p_high)
    return out


def normalize_volume_for_triplets(volume: np.ndarray, cfg) -> np.ndarray:
    """Dispatch intensity normalisation used before 3-channel neighbouring-slice triplets."""
    scope = str(getattr(cfg, "normalization_scope", "slice")).lower().strip()
    p_low, p_high = getattr(cfg, "clip_percentiles", (1.0, 99.0))
    if scope in {"slice", "slice_level", "per_slice"}:
        return slice_level_percentile_normalize(volume, p_low=p_low, p_high=p_high)
    if scope in {"patient", "patient_level", "volume", "volume_level"}:
        return patient_level_percentile_normalize(volume, p_low=p_low, p_high=p_high)
    raise ValueError(
        f"Unknown normalization_scope={scope!r}. Use 'slice' for per-slice normalization or 'patient' for volume-level normalization."
    )

def load_nifti_volume(path: Path) -> np.ndarray:
    vol = nib.load(str(path)).get_fdata()
    if vol.ndim != 3:
        raise ValueError(f"Expected 3D NIfTI at {path}, got shape {vol.shape}")
    return np.asarray(vol, dtype=np.float32)

def choose_slice_centers(z: int, bag_size: int, random_sample: bool = True) -> List[int]:
    if z <= 0:
        raise ValueError("Volume has zero z-slices.")
    if z == 1:
        return [0] * bag_size

    eligible = list(range(z))
    if len(eligible) >= bag_size:
        if random_sample:
            centers = sorted(random.sample(eligible, bag_size))
        else:
            centers = np.linspace(0, z - 1, bag_size)
            centers = [int(round(x)) for x in centers]
    else:
        if random_sample:
            centers = sorted(random.choices(eligible, k=bag_size))
        else:
            centers = [eligible[i % len(eligible)] for i in range(bag_size)]

    return centers

def make_triplet_from_center(volume_u8: np.ndarray, center_z: int) -> np.ndarray:
    z = volume_u8.shape[2]
    z0 = max(center_z - 1, 0)
    z1 = center_z
    z2 = min(center_z + 1, z - 1)
    triplet = np.stack([volume_u8[:, :, z0], volume_u8[:, :, z1], volume_u8[:, :, z2]], axis=-1)
    return triplet

def volume_to_triplet_pils(volume_u8: np.ndarray, centers: List[int]) -> List[Image.Image]:
    triplets = [Image.fromarray(make_triplet_from_center(volume_u8, c)).convert("RGB") for c in centers]
    return triplets

class ResizePadToPatchMultiple:
    def __init__(self, base_resize=512, multiple=14):
        self.base_resize = base_resize
        self.multiple = multiple

    def __call__(self, img):
        img = transforms.Resize((self.base_resize, self.base_resize))(img)
        w, h = img.size
        pad_w = (self.multiple - (w % self.multiple)) % self.multiple
        pad_h = (self.multiple - (h % self.multiple)) % self.multiple
        if pad_w == 0 and pad_h == 0:
            return img
        left = pad_w // 2
        right = pad_w - left
        top = pad_h // 2
        bottom = pad_h - top
        return transforms.functional.pad(img, [left, top, right, bottom], fill=0)

class RandomIntensityJitter:
    def __init__(self, scale=(0.9, 1.1), shift=(-0.05, 0.05), p=0.5):
        self.scale = scale
        self.shift = shift
        self.p = p

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        s = random.uniform(*self.scale)
        b = random.uniform(*self.shift)
        x = tensor * s + b
        return torch.clamp(x, 0.0, 1.0)

def denormalize_tensor_image(x: torch.Tensor, mean=IMG_MEAN, std=IMG_STD) -> np.ndarray:
    x = x.detach().cpu().float().clone()
    if x.ndim != 3:
        raise ValueError(f"Expected CHW tensor, got {tuple(x.shape)}")
    mean_t = torch.tensor(mean, dtype=x.dtype).view(3, 1, 1)
    std_t = torch.tensor(std, dtype=x.dtype).view(3, 1, 1)
    x = x * std_t + mean_t
    x = torch.clamp(x, 0.0, 1.0)
    return x.permute(1, 2, 0).numpy()

In [ ]:
# -----------------------------
# DINO-style transforms
# -----------------------------
def _padded_size(cfg: Config) -> int:
    return cfg.base_resize + ((cfg.pad_multiple - cfg.base_resize % cfg.pad_multiple) % cfg.pad_multiple)

def build_global_transform(cfg: Config):
    size = _padded_size(cfg)
    return transforms.Compose([
        ResizePadToPatchMultiple(cfg.base_resize, cfg.pad_multiple),
        transforms.RandomResizedCrop(
            size=(size, size),
            scale=cfg.global_crop_scale,
            ratio=(0.95, 1.05),
            antialias=True,
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
        RandomIntensityJitter(p=0.7),
        transforms.Normalize(IMG_MEAN, IMG_STD),
    ])

def build_local_transform(cfg: Config):
    size = _padded_size(cfg)
    return transforms.Compose([
        ResizePadToPatchMultiple(cfg.base_resize, cfg.pad_multiple),
        transforms.RandomResizedCrop(
            size=(size, size),
            scale=cfg.local_crop_scale,
            ratio=(0.90, 1.10),
            antialias=True,
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=12),
        transforms.ToTensor(),
        RandomIntensityJitter(p=0.7),
        transforms.Normalize(IMG_MEAN, IMG_STD),
    ])

def build_ssl_validation_transform(cfg: Config):
    # Deterministic SSL validation view.
    # This makes val loss a reproducible health signal instead of a moving target caused by random crops/jitter.
    return transforms.Compose([
        ResizePadToPatchMultiple(cfg.base_resize, cfg.pad_multiple),
        transforms.ToTensor(),
        transforms.Normalize(IMG_MEAN, IMG_STD),
    ])

def build_eval_transform(cfg: Config):
    return transforms.Compose([
        ResizePadToPatchMultiple(cfg.base_resize, cfg.pad_multiple),
        transforms.ToTensor(),
        transforms.Normalize(IMG_MEAN, IMG_STD),
    ])

GLOBAL_TRANSFORM = build_global_transform(CFG)
LOCAL_TRANSFORM  = build_local_transform(CFG)
VAL_SSL_TRANSFORM = build_ssl_validation_transform(CFG) if CFG.deterministic_val_ssl else GLOBAL_TRANSFORM
VAL_LOCAL_TRANSFORM = VAL_SSL_TRANSFORM if CFG.deterministic_val_ssl else LOCAL_TRANSFORM
EVAL_TRANSFORM   = build_eval_transform(CFG)


In [ ]:

# -----------------------------
# Patient discovery
# -----------------------------
def discover_labeled_patients(root: Path) -> List[Dict]:
    entries = []
    if not root.exists():
        warnings.warn(f"Directory does not exist: {root}")
        return entries

    class_dirs = [d for d in sorted(root.iterdir()) if d.is_dir()]
    class_to_idx = {d.name: i for i, d in enumerate(class_dirs)}

    for class_dir in class_dirs:
        for p in sorted(class_dir.rglob("*")):
            if p.is_file() and is_nifti_file(p):
                entries.append({
                    "path": str(p),
                    "patient_id": strip_nii_suffix(p.name),
                    "class_name": class_dir.name,
                    "label": class_to_idx[class_dir.name],
                })
    return entries

def discover_unlabeled_patients(root: Path) -> List[Dict]:
    entries = []
    if not root.exists():
        warnings.warn(f"Directory does not exist: {root}")
        return entries
    for p in sorted(root.rglob("*")):
        if p.is_file() and is_nifti_file(p):
            entries.append({
                "path": str(p),
                "patient_id": strip_nii_suffix(p.name),
                "class_name": None,
                "label": None,
            })
    return entries

train_entries = discover_labeled_patients(TRAIN_DIR)
val_entries   = discover_labeled_patients(VAL_DIR)
oac_entries   = discover_unlabeled_patients(OAC_DIR)

print("Train patients:", len(train_entries))
print("Val patients  :", len(val_entries))
print("OAC patients  :", len(oac_entries))

if train_entries:
    print(pd.DataFrame(train_entries).groupby("class_name").size())
if val_entries:
    print(pd.DataFrame(val_entries).groupby("class_name").size())

In [ ]:

# -----------------------------
# Patient-level datasets
# -----------------------------
class PatientBagMultiCropDataset(Dataset):
    def __init__(
        self,
        entries: List[Dict],
        cfg: Config,
        global_transform,
        local_transform,
        random_centers: bool = True,
    ):
        self.entries = entries
        self.cfg = cfg
        self.global_transform = global_transform
        self.local_transform = local_transform
        self.random_centers = random_centers

    def __len__(self):
        return len(self.entries)

    def _load_triplets(self, path: str, bag_size: int, random_centers: bool):
        vol = load_nifti_volume(Path(path))
        vol_u8 = normalize_volume_for_triplets(vol, self.cfg)
        centers = choose_slice_centers(z=vol_u8.shape[2], bag_size=bag_size, random_sample=random_centers)
        triplets = volume_to_triplet_pils(vol_u8, centers)
        return triplets, centers

    def __getitem__(self, idx):
        entry = self.entries[idx]
        triplets, centers = self._load_triplets(entry["path"], self.cfg.bag_size_train, self.random_centers)

        global_crops = []
        local_crops = []

        for _ in range(self.cfg.n_global_crops):
            global_crops.append(torch.stack([self.global_transform(img) for img in triplets], dim=0))

        for _ in range(self.cfg.n_local_crops):
            local_crops.append(torch.stack([self.local_transform(img) for img in triplets], dim=0))

        return {
            "patient_id": entry["patient_id"],
            "path": entry["path"],
            "label": -1 if entry["label"] is None else int(entry["label"]),
            "class_name": entry["class_name"],
            "centers": torch.tensor(centers, dtype=torch.long),
            "global_crops": global_crops,
            "local_crops": local_crops,
        }

class PatientBagInferenceDataset(Dataset):
    def __init__(self, entries: List[Dict], cfg: Config, eval_transform):
        self.entries = entries
        self.cfg = cfg
        self.eval_transform = eval_transform

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        vol = load_nifti_volume(Path(entry["path"]))
        vol_u8 = normalize_volume_for_triplets(vol, self.cfg)
        centers = choose_slice_centers(z=vol_u8.shape[2], bag_size=self.cfg.bag_size_eval, random_sample=False)
        triplets = volume_to_triplet_pils(vol_u8, centers)
        bag = torch.stack([self.eval_transform(img) for img in triplets], dim=0)
        return {
            "patient_id": entry["patient_id"],
            "path": entry["path"],
            "label": -1 if entry["label"] is None else int(entry["label"]),
            "class_name": entry["class_name"],
            "centers": torch.tensor(centers, dtype=torch.long),
            "bag": bag,
        }

def patient_multicrop_collate(batch):
    n_global = len(batch[0]["global_crops"])
    n_local = len(batch[0]["local_crops"])

    global_crops = [torch.stack([b["global_crops"][i] for b in batch], dim=0) for i in range(n_global)]
    local_crops = [torch.stack([b["local_crops"][i] for b in batch], dim=0) for i in range(n_local)]

    return {
        "patient_id": [b["patient_id"] for b in batch],
        "path": [b["path"] for b in batch],
        "label": torch.tensor([b["label"] for b in batch], dtype=torch.long),
        "class_name": [b["class_name"] for b in batch],
        "centers": torch.stack([b["centers"] for b in batch], dim=0),
        "global_crops": global_crops,
        "local_crops": local_crops,
    }

def patient_inference_collate(batch):
    return {
        "patient_id": [b["patient_id"] for b in batch],
        "path": [b["path"] for b in batch],
        "label": torch.tensor([b["label"] for b in batch], dtype=torch.long),
        "class_name": [b["class_name"] for b in batch],
        "centers": torch.stack([b["centers"] for b in batch], dim=0),
        "bag": torch.stack([b["bag"] for b in batch], dim=0),
    }

In [ ]:
# -----------------------------
# Optional balancing across labeled train patients
# -----------------------------
def make_patient_balanced_sampler(entries: List[Dict]):
    labels = np.array([e["label"] for e in entries], dtype=int)
    counts = np.bincount(labels)
    class_weights = 1.0 / np.maximum(counts, 1)
    weights = class_weights[labels]
    weights = torch.as_tensor(weights, dtype=torch.double)
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)
    return sampler, counts

train_ssl_dataset = PatientBagMultiCropDataset(
    train_entries, CFG, GLOBAL_TRANSFORM, LOCAL_TRANSFORM, random_centers=True
)
val_ssl_dataset = PatientBagMultiCropDataset(
    val_entries, CFG, VAL_SSL_TRANSFORM, VAL_LOCAL_TRANSFORM, random_centers=False
)

train_eval_dataset = PatientBagInferenceDataset(train_entries, CFG, EVAL_TRANSFORM)
val_eval_dataset   = PatientBagInferenceDataset(val_entries, CFG, EVAL_TRANSFORM)
oac_eval_dataset   = PatientBagInferenceDataset(oac_entries, CFG, EVAL_TRANSFORM)

train_sampler = None
if CFG.balance_patients_across_classes and train_entries and all(e["label"] is not None for e in train_entries):
    train_sampler, train_class_counts = make_patient_balanced_sampler(train_entries)
    print("Train class counts:", train_class_counts)

train_ssl_loader = DataLoader(
    train_ssl_dataset,
    batch_size=CFG.train_batch_size,
    shuffle=(train_sampler is None),
    sampler=train_sampler,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=patient_multicrop_collate,
    drop_last=len(train_ssl_dataset) >= CFG.train_batch_size,
)

val_ssl_loader = DataLoader(
    val_ssl_dataset,
    batch_size=CFG.train_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=patient_multicrop_collate,
    drop_last=False,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=CFG.eval_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=patient_inference_collate,
)

val_eval_loader = DataLoader(
    val_eval_dataset,
    batch_size=CFG.eval_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=patient_inference_collate,
)

oac_eval_loader = DataLoader(
    oac_eval_dataset,
    batch_size=CFG.eval_batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=patient_inference_collate,
)

print("train_ssl_loader batches:", len(train_ssl_loader))
print("val_ssl_loader   batches:", len(val_ssl_loader))
print("train_eval_loader batches:", len(train_eval_loader))
print("oac_eval_loader   batches:", len(oac_eval_loader))
print("Validation SSL transform is deterministic:", CFG.deterministic_val_ssl)
print("Effective SSL batch via accumulation:", CFG.train_batch_size * max(int(CFG.grad_accum_steps), 1))


## Diagnostic-first training strategy

Before changing the model, this cell checks the most common causes of a late rise in training/validation loss for this setup:

1. **validation leakage into training state** — validation must not update the DINO centre buffer;
2. **stochastic validation loss** — validation should use deterministic views;
3. **tiny effective batch size** — DINO centre/teacher statistics are noisy when patient-level batch size is 1;
4. **excessive learning rate or aggressive augmentation** — especially when only a small patient cohort is available;
5. **collapse/instability indicators** — very low teacher entropy, very low embedding variance, non-finite losses, or large gradient norms.


In [ ]:
# -----------------------------
# Pre-training diagnostics
# -----------------------------
def _entry_class_counts(entries: List[Dict]) -> pd.DataFrame:
    if not entries:
        return pd.DataFrame(columns=["class_name", "n"])
    return (
        pd.DataFrame(entries)
        .assign(class_name=lambda d: d["class_name"].fillna("unlabeled") if "class_name" in d else "unknown")
        .groupby("class_name")
        .size()
        .reset_index(name="n")
        .sort_values("class_name")
    )

def summarize_training_risk_flags(cfg: Config, train_entries: List[Dict], val_entries: List[Dict]) -> pd.DataFrame:
    flags = []
    eff_batch = int(cfg.train_batch_size) * max(int(cfg.grad_accum_steps), 1)

    def add(item, status, action):
        flags.append({"check": item, "status": status, "recommended_action": action})

    add(
        "Validation updates DINO centre",
        "OK" if not cfg.update_center_on_val else "HIGH_RISK",
        "Keep CFG.update_center_on_val=False so validation does not mutate training state.",
    )
    add(
        "Validation transform randomness",
        "OK" if cfg.deterministic_val_ssl else "HIGH_RISK",
        "Use deterministic validation views for interpretable val loss.",
    )
    add(
        "Effective SSL batch size",
        "OK" if eff_batch >= 4 else "WARN",
        f"Current effective batch={eff_batch}. Use grad_accum_steps>=4 if memory requires train_batch_size=1.",
    )
    add(
        "Learning rate",
        "OK" if cfg.lr <= 5e-5 else "WARN",
        "For head/pool-only SSL on small medical cohorts start around 1e-5 to 5e-5.",
    )
    add(
        "Backbone update scope",
        "OK" if cfg.freeze_backbone and cfg.unfreeze_last_n_blocks == 0 else "WARN",
        "First stabilise head/pool-only training. Then optionally unfreeze only the last 1 block with a smaller LR.",
    )
    add(
        "Local crops",
        "OK" if cfg.n_local_crops == 0 else "WARN",
        "Keep local crops at 0 during the diagnostic run; add local crops only after stable curves.",
    )
    add(
        "KoLeo regularizer",
        "OK" if not cfg.use_koleo else "WARN",
        "Disable KoLeo until effective batch size and loss curves are stable.",
    )
    add(
        "Train/val patient availability",
        "OK" if len(train_entries) > 0 and len(val_entries) > 0 else "WARN",
        "Use a held-out val split where possible; if no val split exists, monitor train loss and embedding diagnostics only.",
    )
    return pd.DataFrame(flags)

def inspect_first_ssl_batch(loader: DataLoader, split_name: str = "train") -> pd.DataFrame:
    if len(loader) == 0:
        return pd.DataFrame([{"split": split_name, "message": "empty loader"}])
    batch = next(iter(loader))
    rows = []
    for view_kind in ["global_crops", "local_crops"]:
        for i, x in enumerate(batch[view_kind]):
            rows.append({
                "split": split_name,
                "view": f"{view_kind}[{i}]",
                "shape": tuple(x.shape),
                "finite": bool(torch.isfinite(x).all().item()),
                "mean": float(x.mean().item()),
                "std": float(x.std().item()),
                "min": float(x.min().item()),
                "max": float(x.max().item()),
            })
    return pd.DataFrame(rows)

print("Train class counts:")
display(_entry_class_counts(train_entries))
print("Validation class counts:")
display(_entry_class_counts(val_entries))

risk_df = summarize_training_risk_flags(CFG, train_entries, val_entries)
display(risk_df)

try:
    print("First train SSL batch tensor statistics:")
    display(inspect_first_ssl_batch(train_ssl_loader, "train"))
    print("First validation SSL batch tensor statistics:")
    display(inspect_first_ssl_batch(val_ssl_loader, "val"))
except Exception as e:
    print("Batch sanity check skipped/failed:", e)


In [ ]:

# -----------------------------
# Visual sanity check: slice-level-normalized triplets
# -----------------------------
def show_random_patient_triplets(entries: List[Dict], n_patients=2, bag_size=6):
    if len(entries) == 0:
        print("No entries available.")
        return

    chosen = random.sample(entries, k=min(n_patients, len(entries)))
    for entry in chosen:
        vol = load_nifti_volume(Path(entry["path"]))
        vol_u8 = normalize_volume_for_triplets(vol, CFG)
        centers = choose_slice_centers(vol_u8.shape[2], bag_size=bag_size, random_sample=False)
        triplets = volume_to_triplet_pils(vol_u8, centers)

        fig, axes = plt.subplots(1, len(triplets), figsize=(3 * len(triplets), 3))
        if len(triplets) == 1:
            axes = [axes]
        for ax, img, c in zip(axes, triplets, centers):
            ax.imshow(np.asarray(img))
            ax.set_title(f"z={c}")
            ax.axis("off")
        plt.suptitle(f"{entry['patient_id']} | class={entry['class_name']}")
        plt.tight_layout()
        plt.show()

show_random_patient_triplets(train_entries, n_patients=1, bag_size=min(6, CFG.bag_size_eval))

## DINOv2 backbone loading

This cell now follows the same structure as your working local code:
- build the register ViT directly from `dinov2.models.vision_transformer`
- load official pretrained weights manually
- avoid fragile local `torch.hub` assumptions unless explicitly requested


In [ ]:

# -----------------------------
# Mock backbone for smoke test
# -----------------------------
class MockAttention(nn.Module):
    def __init__(self, dim=384, num_heads=6):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q * self.scale) @ k.transpose(-2, -1)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        return out

class MockBlock(nn.Module):
    def __init__(self, dim=384, num_heads=6, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MockAttention(dim=dim, num_heads=num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))
        self.ls1 = nn.Identity()
        self.ls2 = nn.Identity()

    def forward(self, x):
        x = x + self.ls1(self.attn(self.norm1(x)))
        x = x + self.ls2(self.mlp(self.norm2(x)))
        return x

class MockDinoBackbone(nn.Module):
    def __init__(self, img_size=518, patch_size=14, embed_dim=384, depth=6, num_heads=6, num_register_tokens=4):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_register_tokens = num_register_tokens
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.register_tokens = nn.Parameter(torch.zeros(1, num_register_tokens, embed_dim))
        self.blocks = nn.ModuleList([MockBlock(embed_dim, num_heads=num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)

    def prepare_tokens_with_masks(self, x, masks=None):
        B, C, H, W = x.shape
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        regs = self.register_tokens.expand(B, -1, -1)
        x = torch.cat([cls, regs, x], dim=1)
        return x

    def forward_features(self, x, masks=None):
        x = self.prepare_tokens_with_masks(x, masks)
        for blk in self.blocks:
            x = blk(x)
        x_norm = self.norm(x)
        return {
            "x_norm_clstoken": x_norm[:, 0],
            "x_norm_regtokens": x_norm[:, 1 : 1 + self.num_register_tokens],
            "x_norm_patchtokens": x_norm[:, 1 + self.num_register_tokens :],
            "x_prenorm": x,
            "masks": masks,
        }

    def forward(self, x):
        return self.forward_features(x)

# -----------------------------
# Real DINOv2 loading helpers
# -----------------------------
def download_file(url: str, local_path: Path):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {local_path}")
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    with open(local_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


def _ensure_repo_on_syspath(repo_dir: Optional[Path]):
    if repo_dir is None:
        return
    repo_dir = Path(repo_dir)
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))


def _get_vit_constructors(repo_dir: Optional[Path] = None):
    last_err = None

    # try repo injection first if user gave a local repo path
    if repo_dir is not None and Path(repo_dir).exists():
        _ensure_repo_on_syspath(Path(repo_dir))
        try:
            from dinov2.models.vision_transformer import vit_small, vit_base, vit_large, vit_giant2
            return vit_small, vit_base, vit_large, vit_giant2
        except Exception as e:
            last_err = e

    # fallback to installed package import if available
    try:
        from dinov2.models.vision_transformer import vit_small, vit_base, vit_large, vit_giant2
        return vit_small, vit_base, vit_large, vit_giant2
    except Exception as e:
        last_err = e

    raise ImportError(
        "Could not import DINOv2 vision transformer constructors. "
        "Set CFG.dinov2_repo_dir to a valid local DINOv2 repo root or install/import dinov2 locally."
    ) from last_err


def _official_weights_maps():
    weights_map = {
        "dinov2_vits14": "dinov2_vits14_pretrain.pth",
        "dinov2_vitb14": "dinov2_vitb14_pretrain.pth",
        "dinov2_vitl14": "dinov2_vitl14_pretrain.pth",
        "dinov2_vitg14": "dinov2_vitg14_pretrain.pth",
        "dinov2_vits14_reg": "dinov2_vits14_reg4_pretrain.pth",
        "dinov2_vitb14_reg": "dinov2_vitb14_reg4_pretrain.pth",
        "dinov2_vitl14_reg": "dinov2_vitl14_reg4_pretrain.pth",
        "dinov2_vitg14_reg": "dinov2_vitg14_reg4_pretrain.pth",
    }
    urls_map = {
        "dinov2_vits14": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth",
        "dinov2_vitb14": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth",
        "dinov2_vitl14": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth",
        "dinov2_vitg14": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitg14/dinov2_vitg14_pretrain.pth",
        "dinov2_vits14_reg": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_reg4_pretrain.pth",
        "dinov2_vitb14_reg": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_reg4_pretrain.pth",
        "dinov2_vitl14_reg": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth",
        "dinov2_vitg14_reg": "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitg14/dinov2_vitg14_reg4_pretrain.pth",
    }
    return weights_map, urls_map


def _resolve_checkpoint_path(cfg: Config) -> Path:
    if cfg.manual_checkpoint_path is not None:
        ckpt = Path(cfg.manual_checkpoint_path)
        if not ckpt.exists():
            raise FileNotFoundError(f"CFG.manual_checkpoint_path does not exist: {ckpt}")
        return ckpt

    weights_map, urls_map = _official_weights_maps()
    if cfg.backbone_name not in weights_map:
        raise ValueError(f"No official checkpoint mapping found for backbone_name={cfg.backbone_name}")

    cache_dir = Path(cfg.checkpoint_cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    ckpt = cache_dir / weights_map[cfg.backbone_name]
    if not ckpt.exists():
        if not cfg.auto_download_checkpoint:
            raise FileNotFoundError(
                f"Checkpoint not found at {ckpt}. Set CFG.manual_checkpoint_path or enable auto download."
            )
        download_file(urls_map[cfg.backbone_name], ckpt)
    return ckpt


def _infer_native_img_size_from_state(state: dict, patch_size: int = 14):
    pos = state.get("pos_embed", None)
    if pos is None or not torch.is_tensor(pos) or pos.ndim != 3:
        return None
    n_tokens = pos.shape[1]
    if n_tokens <= 1:
        return None
    n_patches = n_tokens - 1  # DINOv2 pos_embed covers cls + patch tokens; register tokens are separate
    grid = int(round(n_patches ** 0.5))
    if grid * grid != n_patches:
        return None
    return grid * patch_size


def build_manual_local_dinov2(cfg: Config):
    vit_small, vit_base, vit_large, vit_giant2 = _get_vit_constructors(
        Path(cfg.dinov2_repo_dir) if cfg.dinov2_repo_dir else None
    )

    manual_ctor = {
        "dinov2_vits14_reg": (vit_small, 384, 4),
        "dinov2_vitb14_reg": (vit_base, 768, 4),
        "dinov2_vitl14_reg": (vit_large, 1024, 4),
        "dinov2_vitg14_reg": (vit_giant2, 1536, 4),
        "dinov2_vits14": (vit_small, 384, 0),
        "dinov2_vitb14": (vit_base, 768, 0),
        "dinov2_vitl14": (vit_large, 1024, 0),
        "dinov2_vitg14": (vit_giant2, 1536, 0),
    }
    if cfg.backbone_name not in manual_ctor:
        raise ValueError(f"Unsupported backbone_name for manual loading: {cfg.backbone_name}")

    ctor, embed_dim, num_regs = manual_ctor[cfg.backbone_name]

    ckpt_path = _resolve_checkpoint_path(cfg)
    state = torch.load(ckpt_path, map_location="cpu")
    if isinstance(state, dict) and "model" in state:
        state = state["model"]

    native_img_size = _infer_native_img_size_from_state(state, patch_size=14)
    if native_img_size is None:
        native_img_size = cfg.backbone_native_img_size

    model = ctor(
        patch_size=14,
        img_size=native_img_size,
        init_values=1.0,
        num_register_tokens=num_regs,
        block_chunks=0,
    )

    missing, unexpected = model.load_state_dict(state, strict=True)
    print(f"Loaded pretrained weights from: {ckpt_path}")
    print(f"Backbone instantiated at native pretrained img_size={native_img_size}; training/eval views may still use cfg.base_resize={cfg.base_resize}.")
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    return model, embed_dim


def _hub_load_local(repo_dir: Path, backbone_name: str):
    return torch.hub.load(str(repo_dir), backbone_name, source="local")


def load_backbone(cfg: Config):
    if cfg.backbone_source == "mock_smoke":
        backbone = MockDinoBackbone(
            img_size=cfg.base_resize + ((cfg.pad_multiple - cfg.base_resize % cfg.pad_multiple) % cfg.pad_multiple),
            patch_size=14,
            embed_dim=384,
            depth=6,
            num_heads=6,
            num_register_tokens=4 if "reg" in cfg.backbone_name else 0,
        )
        return backbone, 384

    if cfg.backbone_source == "manual_local_weights":
        return build_manual_local_dinov2(cfg)

    if cfg.backbone_source == "local_hub":
        repo_dir = Path(cfg.dinov2_repo_dir)
        if not repo_dir.exists():
            raise FileNotFoundError(f"Local hub repo path not found: {repo_dir}")
        backbone = _hub_load_local(repo_dir, cfg.backbone_name)
        embed_dim = backbone.embed_dim if hasattr(backbone, "embed_dim") else None
        return backbone, embed_dim

    if cfg.backbone_source == "remote_hub":
        backbone = torch.hub.load("facebookresearch/dinov2", cfg.backbone_name)
        embed_dim = backbone.embed_dim if hasattr(backbone, "embed_dim") else None
        return backbone, embed_dim

    raise ValueError(f"Unsupported backbone_source: {cfg.backbone_source}")



### Local DINOv2 register-backbone troubleshooting

If `torch.hub.load(..., "dinov2_vitb14_reg", source="local")` fails with **`Cannot find callable ... in hubconf`**, your local repo checkout is usually older than the official register-backbone update, or it is a fork whose `hubconf.py` does not expose the register entrypoints. The current official `hubconf.py` imports `dinov2_vitb14_reg`, and the official `dinov2/hub/backbones.py` defines the register constructors directly. citeturn222847view0turn380601view0

This notebook now falls back to importing the backbone directly from `dinov2.hub.backbones` when local `hubconf.py` is incomplete. If your repo truly lacks the register implementation, set `CFG.backbone_name = "dinov2_vitb14"` temporarily or point `CFG.dinov2_repo_dir` to an updated official DINOv2 checkout. The official README also documents `torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14_reg')` as a supported pretrained backbone. citeturn830609search2turn830609search7


In [ ]:

# -----------------------------
# Patient-level pooling + DINO head
# -----------------------------
class PatientAttentionPool(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: Optional[int] = None):
        super().__init__()
        hidden_dim = hidden_dim or max(128, embed_dim // 2)
        self.score_net = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, slice_tokens: torch.Tensor, mask: Optional[torch.Tensor] = None):
        # slice_tokens: [B, S, D]
        scores = self.score_net(slice_tokens).squeeze(-1)  # [B, S]
        if mask is not None:
            scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        patient_token = torch.einsum("bs,bsd->bd", weights, slice_tokens)
        return patient_token, weights

class DINOProjectionHead(nn.Module):
    def __init__(
        self,
        in_dim: int,
        out_dim: int = 8192,
        hidden_dim: int = 2048,
        bottleneck_dim: int = 256,
        nlayers: int = 3,
        norm_last_layer: bool = True,
    ):
        super().__init__()
        nlayers = max(nlayers, 1)

        if nlayers == 1:
            self.mlp = nn.Linear(in_dim, bottleneck_dim)
        else:
            layers = [nn.Linear(in_dim, hidden_dim), nn.GELU()]
            for _ in range(nlayers - 2):
                layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU()]
            layers += [nn.Linear(hidden_dim, bottleneck_dim)]
            self.mlp = nn.Sequential(*layers)

        self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1.0)
        if norm_last_layer:
            self.last_layer.weight_g.requires_grad = False

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1, p=2)
        x = self.last_layer(x)
        return x

class PatientDinoWrapper(nn.Module):
    def __init__(self, backbone: nn.Module, embed_dim: int, cfg: Config):
        super().__init__()
        self.backbone = backbone
        self.embed_dim = embed_dim
        self.cfg = cfg
        self.pool = PatientAttentionPool(embed_dim)
        self.head = DINOProjectionHead(
            in_dim=embed_dim,
            out_dim=cfg.head_out_dim,
            hidden_dim=cfg.head_hidden_dim,
            bottleneck_dim=cfg.head_bottleneck_dim,
            nlayers=cfg.head_nlayers,
            norm_last_layer=cfg.norm_last_layer,
        )

    def backbone_is_trainable(self) -> bool:
        return any(p.requires_grad for p in self.backbone.parameters())

    def _extract_cls_tokens(self, x4d: torch.Tensor, chunk_size: Optional[int] = None, requires_grad: bool = True):
        cls_chunks = []
        if chunk_size is None or chunk_size <= 0:
            feats = self.backbone.forward_features(x4d)
            return feats["x_norm_clstoken"]

        splits = x4d.split(chunk_size, dim=0)
        ctx = torch.enable_grad if requires_grad else torch.no_grad
        with ctx():
            for xs in splits:
                feats = self.backbone.forward_features(xs)
                cls_chunks.append(feats["x_norm_clstoken"])
        return torch.cat(cls_chunks, dim=0)

    def forward_bag_features(self, bag_5d: torch.Tensor, need_patch_tokens: bool = False):
        # bag_5d: [B, S, 3, H, W]
        B, S, C, H, W = bag_5d.shape
        x = bag_5d.reshape(B * S, C, H, W)

        backbone_trainable = self.backbone_is_trainable()
        patch_tokens = None

        if need_patch_tokens:
            # Used only for interpretability / visualization; keep simple and no-grad outside caller.
            feats = self.backbone.forward_features(x)
            cls_tokens = feats["x_norm_clstoken"]
            patch_tokens = feats["x_norm_patchtokens"]
        else:
            if backbone_trainable:
                cls_tokens = self._extract_cls_tokens(x, chunk_size=None, requires_grad=True)
            else:
                cls_tokens = self._extract_cls_tokens(
                    x,
                    chunk_size=max(1, int(self.cfg.frozen_backbone_chunk_size)),
                    requires_grad=False,
                )

        cls_tokens = cls_tokens.reshape(B, S, -1)
        patient_tokens, slice_weights = self.pool(cls_tokens)
        return {
            "patient_tokens": patient_tokens,              # [B, D]
            "slice_tokens": cls_tokens,                    # [B, S, D]
            "slice_weights": slice_weights,                # [B, S]
            "patch_tokens": patch_tokens,                  # [B*S, N, D] or None
        }

    def forward(self, bag_5d: torch.Tensor):
        out = self.forward_bag_features(bag_5d, need_patch_tokens=False)
        out["proj"] = self.head(out["patient_tokens"])
        return out


In [ ]:
# -----------------------------
# DINO-style loss + schedules
# -----------------------------
def cosine_scheduler(base_value, final_value, epochs, niter_per_ep, warmup_epochs=0, start_warmup_value=0.0):
    """Robust cosine scheduler with optional warm-up."""
    epochs = max(int(epochs), 1)
    niter_per_ep = max(int(niter_per_ep), 1)
    total_iters = epochs * niter_per_ep
    warmup_iters = min(max(int(warmup_epochs), 0) * niter_per_ep, total_iters)

    warmup_schedule = np.array([], dtype=np.float32)
    if warmup_iters > 0:
        warmup_schedule = np.linspace(start_warmup_value, base_value, warmup_iters, dtype=np.float32)

    remaining_iters = total_iters - warmup_iters
    if remaining_iters <= 0:
        return warmup_schedule.astype(np.float32)

    iters = np.arange(remaining_iters, dtype=np.float32)
    schedule = final_value + 0.5 * (base_value - final_value) * (1 + np.cos(np.pi * iters / max(remaining_iters, 1)))
    schedule = np.concatenate((warmup_schedule, schedule.astype(np.float32)))
    return schedule.astype(np.float32)

class DINOLoss(nn.Module):
    def __init__(
        self,
        out_dim: int,
        n_student_views: int,
        warmup_teacher_temp: float,
        teacher_temp: float,
        warmup_teacher_temp_epochs: int,
        nepochs: int,
        student_temp: float = 0.1,
        center_momentum: float = 0.9,
    ):
        super().__init__()
        self.student_temp = student_temp
        self.center_momentum = center_momentum
        self.n_student_views = n_student_views
        self.register_buffer("center", torch.zeros(1, out_dim))

        nepochs = max(int(nepochs), 1)
        warm_epochs = min(max(int(warmup_teacher_temp_epochs), 0), nepochs)
        warm = np.linspace(warmup_teacher_temp, teacher_temp, warm_epochs, dtype=np.float32) if warm_epochs > 0 else np.array([], dtype=np.float32)
        rest = np.ones(nepochs - warm_epochs, dtype=np.float32) * float(teacher_temp)
        self.teacher_temp_schedule = np.concatenate((warm, rest)).astype(np.float32)

    @torch.no_grad()
    def update_center(self, teacher_output):
        teacher_output = teacher_output.detach().float()
        batch_center = torch.mean(teacher_output, dim=0, keepdim=True).to(self.center.device)
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)

    def forward(self, student_output_list: List[torch.Tensor], teacher_output_list: List[torch.Tensor], epoch: int):
        student_out = [F.log_softmax(s.float() / self.student_temp, dim=-1) for s in student_output_list]

        temp = float(self.teacher_temp_schedule[min(epoch, len(self.teacher_temp_schedule) - 1)])
        teacher_out = [F.softmax((t.float() - self.center.float()) / temp, dim=-1).detach() for t in teacher_output_list]

        total_loss = 0.0
        n_loss_terms = 0

        n_teacher = len(teacher_out)
        for iq, q in enumerate(teacher_out):
            for v, s in enumerate(student_out):
                if v < n_teacher and v == iq:
                    continue
                loss = torch.sum(-q * s, dim=-1).mean()
                total_loss += loss
                n_loss_terms += 1

        total_loss /= max(n_loss_terms, 1)
        self.update_center(torch.cat(teacher_output_list, dim=0))
        return total_loss

def probability_entropy(probs: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    probs = probs.float().clamp_min(eps)
    return -(probs * probs.log()).sum(dim=-1)

def koleo_loss(x: torch.Tensor, eps: float = 1e-8):
    # x: [B, D]
    x = F.normalize(x, dim=-1)
    dist = torch.pdist(x, p=2)
    if dist.numel() == 0:
        return torch.tensor(0.0, device=x.device)
    return -torch.log(dist + eps).mean()


In [ ]:

# -----------------------------
# Backbone freezing strategy
# -----------------------------
def set_backbone_trainability(backbone: nn.Module, freeze_backbone: bool, unfreeze_last_n_blocks: int = 0):
    for p in backbone.parameters():
        p.requires_grad = not freeze_backbone

    if freeze_backbone:
        for p in backbone.parameters():
            p.requires_grad = False

        if hasattr(backbone, "blocks") and unfreeze_last_n_blocks > 0:
            for blk in list(backbone.blocks)[-unfreeze_last_n_blocks:]:
                for p in blk.parameters():
                    p.requires_grad = True

        # keep norm trainable if present
        if hasattr(backbone, "norm"):
            for p in backbone.norm.parameters():
                p.requires_grad = True

def copy_model_weights(student: nn.Module, teacher: nn.Module):
    teacher.load_state_dict(student.state_dict(), strict=True)
    for p in teacher.parameters():
        p.requires_grad = False

In [ ]:

# -----------------------------
# Build student / teacher
# -----------------------------
student_backbone, embed_dim = load_backbone(CFG)
if embed_dim is None:
    embed_dim = getattr(student_backbone, "embed_dim", None)
if embed_dim is None:
    raise ValueError("Could not infer backbone embed_dim.")

teacher_backbone, _ = load_backbone(CFG)

student = PatientDinoWrapper(student_backbone, embed_dim, CFG)
teacher = PatientDinoWrapper(teacher_backbone, embed_dim, CFG)

copy_model_weights(student, teacher)
set_backbone_trainability(student.backbone, CFG.freeze_backbone, CFG.unfreeze_last_n_blocks)

student = student.to(device)
teacher = teacher.to(device)

trainable_params = [p for p in student.parameters() if p.requires_grad]
print("Trainable parameter tensors:", len(trainable_params))
print("Embedding dimension:", embed_dim)

## Branch A/B: frozen DINO baseline and supervised frozen-DINO classifier

Branch A exports frozen-DINO patient embeddings without changing any weights. Branch B trains a supervised classification head on top of frozen DINOv2 features. The DINOv2 backbone remains frozen; only the patient attention pooling layer and classifier head are trainable by default.


In [ ]:
# ============================================================
# Branch A/B helper code: frozen baseline and supervised classifier
# Patched version:
#   - Branch A remains fully frozen DINO baseline.
#   - Branch B supports user-defined DINOv2 layer unfreezing.
#   - Supports last-N block unfreezing, pattern-based unfreezing, norm tuning.
#   - Uses separate LR groups for classifier/pool and selected backbone layers.
# ============================================================

import math
import json
import fnmatch
from typing import Optional, List, Dict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)

# ------------------------------------------------------------
# Supervised branch tuning defaults
# ------------------------------------------------------------
# These attributes are added only if not already defined in CFG.
# You can override them before running this cell if desired.

def _cfg_set_default(name, value):
    if not hasattr(CFG, name):
        setattr(CFG, name, value)

# Enable selective DINOv2 fine-tuning for Branch B.
# If False, Branch B behaves like the original frozen-DINO classifier.
_cfg_set_default("supervised_enable_backbone_tuning", True)

# Recommended first test: unfreeze only the last DINOv2 block.
_cfg_set_default("supervised_unfreeze_last_n_blocks", 1)

# Optional user-defined name patterns.
# Examples:
#   CFG.supervised_unfreeze_patterns = ("blocks.11",)
#   CFG.supervised_unfreeze_patterns = ("blocks.10", "blocks.11")
#   CFG.supervised_unfreeze_patterns = ("norm",)
#   CFG.supervised_unfreeze_patterns = ("blocks.11", "norm")
_cfg_set_default("supervised_unfreeze_patterns", ())

# Optional norm-layer tuning. Keep False initially unless needed.
_cfg_set_default("supervised_train_norm_layers", False)

# Backbone LR multiplier relative to classifier/pool LR.
# Example: supervised_lr=1e-4 and multiplier=0.05 gives backbone LR=5e-6.
_cfg_set_default("supervised_backbone_lr_mult", 0.05)

# Optional warmup epochs for supervised branch.
_cfg_set_default("supervised_warmup_epochs", 1)

print("Supervised branch backbone-tuning settings:")
print("  supervised_enable_backbone_tuning:", getattr(CFG, "supervised_enable_backbone_tuning"))
print("  supervised_unfreeze_last_n_blocks:", getattr(CFG, "supervised_unfreeze_last_n_blocks"))
print("  supervised_unfreeze_patterns:", getattr(CFG, "supervised_unfreeze_patterns"))
print("  supervised_train_norm_layers:", getattr(CFG, "supervised_train_norm_layers"))
print("  supervised_backbone_lr_mult:", getattr(CFG, "supervised_backbone_lr_mult"))
print("  supervised_warmup_epochs:", getattr(CFG, "supervised_warmup_epochs"))


# ------------------------------------------------------------
# General branch helpers
# ------------------------------------------------------------

def _branch_autocast_context():
    enabled = bool(CFG.use_amp and torch.cuda.is_available())
    if not enabled:
        return torch.cuda.amp.autocast(enabled=False)

    dtype = (
        torch.bfloat16
        if (
            str(CFG.amp_dtype).lower() == "bfloat16"
            and torch.cuda.is_bf16_supported()
        )
        else torch.float16
    )

    return torch.cuda.amp.autocast(dtype=dtype)


@torch.no_grad()
def extract_patient_embeddings_from_model(
    model: nn.Module,
    loader: DataLoader,
    split_name: str,
    embed_dim_fallback: Optional[int] = None,
):
    """
    Generic extractor for:
      - PatientDinoWrapper
      - supervised DINO classifier
      - JEPA models

    Exports the patient-level embedding returned by forward_bag_features().
    """
    model.eval()

    rows = []
    all_embeddings = []

    for batch in loader:
        bag = batch["bag"].to(device, non_blocking=True)

        out = model.forward_bag_features(bag)

        if "patient_tokens" not in out:
            raise KeyError(
                "forward_bag_features must return a dictionary containing 'patient_tokens'. "
                f"Available keys: {list(out.keys())}"
            )

        patient_tokens = F.normalize(
            out["patient_tokens"],
            dim=-1,
        ).detach().float().cpu().numpy()

        for i in range(patient_tokens.shape[0]):
            rows.append({
                "split": split_name,
                "patient_id": batch["patient_id"][i],
                "path": batch["path"][i],
                "label": int(batch["label"][i].item()),
                "class_name": batch["class_name"][i],
            })
            all_embeddings.append(patient_tokens[i])

    if len(rows) == 0:
        d = int(embed_dim_fallback or globals().get("embed_dim", 0) or 0)
        return pd.DataFrame(), np.zeros((0, d), dtype=np.float32)

    return pd.DataFrame(rows), np.vstack(all_embeddings).astype(np.float32)


def export_branch_embeddings(
    model: nn.Module,
    branch_name: str,
    embed_dim_fallback: Optional[int] = None,
):
    """
    Saves train/val/OAC embeddings for a branch.

    Output:
      patient_dinov2_ssl_outputs/artifacts/branches/<branch_name>/
          train_embeddings.npy
          val_embeddings.npy
          oac_embeddings.npy
          train_embeddings_metadata.csv
          val_embeddings_metadata.csv
          oac_embeddings_metadata.csv
          train_patient_ids.npy
          val_patient_ids.npy
          oac_patient_ids.npy
          train_labels.npy
          val_labels.npy
          oac_labels.npy
    """
    export_dir = BRANCH_ROOT / branch_name
    export_dir.mkdir(parents=True, exist_ok=True)

    outputs = {}

    for split_name, loader in [
        ("train", train_eval_loader),
        ("val", val_eval_loader),
        ("oac", oac_eval_loader),
    ]:
        if len(loader) == 0:
            outputs[split_name] = (
                pd.DataFrame(),
                np.zeros((0, int(embed_dim_fallback or 0)), dtype=np.float32),
            )
            continue

        df, X = extract_patient_embeddings_from_model(
            model,
            loader,
            split_name,
            embed_dim_fallback,
        )

        df["branch"] = branch_name

        df.to_csv(export_dir / f"{split_name}_embeddings_metadata.csv", index=False)
        np.save(export_dir / f"{split_name}_embeddings.npy", X)
        np.save(export_dir / f"{split_name}_patient_ids.npy", df["patient_id"].astype(str).values)
        np.save(export_dir / f"{split_name}_labels.npy", df["class_name"].astype(str).values)

        outputs[split_name] = (df, X)

        print(f"[{branch_name}] {split_name} embeddings: {X.shape}")

    with open(export_dir / "manifest.json", "w") as f:
        json.dump(
            {
                "branch": branch_name,
                "cfg": asdict(CFG),
            },
            f,
            indent=2,
            default=str,
        )

    return outputs


# ------------------------------------------------------------
# Branch A: frozen DINO encoder
# ------------------------------------------------------------

def build_frozen_patient_dino_encoder(cfg: Config):
    """
    Fully frozen DINOv2 baseline.

    Branch A is intentionally not trained. It measures how separable
    the pretrained DINOv2 patient embeddings are before adaptation.
    """
    backbone, local_embed_dim = load_backbone(cfg)

    if local_embed_dim is None:
        local_embed_dim = getattr(backbone, "embed_dim", None)

    if local_embed_dim is None:
        raise ValueError("Could not infer backbone embed_dim for frozen DINO encoder.")

    model = PatientDinoWrapper(backbone, local_embed_dim, cfg)

    for p in model.backbone.parameters():
        p.requires_grad = False

    for p in model.head.parameters():
        p.requires_grad = False

    # In Branch A the pool is not trained.
    for p in model.pool.parameters():
        p.requires_grad = False

    return model.to(device), int(local_embed_dim)


# ------------------------------------------------------------
# Supervised selective unfreezing helpers
# ------------------------------------------------------------

def _as_tuple(x):
    if x is None:
        return tuple()

    if isinstance(x, str):
        if x.strip() == "":
            return tuple()
        return tuple(v.strip() for v in x.split(",") if v.strip())

    if isinstance(x, (list, tuple)):
        return tuple(x)

    return tuple([x])


def _matches_any_pattern(name: str, patterns):
    """
    Supports:
      - substring matching: 'blocks.11'
      - wildcard matching: '*blocks.11*'
    """
    for pat in patterns:
        pat = str(pat)

        if pat == "":
            continue

        if pat in name:
            return True

        if fnmatch.fnmatch(name, pat):
            return True

        if fnmatch.fnmatch(name, f"*{pat}*"):
            return True

    return False


def configure_supervised_backbone_trainability(backbone: nn.Module, cfg: Config):
    """
    Freezes the DINOv2 backbone, then selectively unfreezes user-defined layers.

    Controls:
      CFG.supervised_enable_backbone_tuning
      CFG.supervised_unfreeze_last_n_blocks
      CFG.supervised_unfreeze_patterns
      CFG.supervised_train_norm_layers

    Recommended first run:
      CFG.supervised_enable_backbone_tuning = True
      CFG.supervised_unfreeze_last_n_blocks = 1
      CFG.supervised_unfreeze_patterns = ()
      CFG.supervised_train_norm_layers = False
      CFG.supervised_backbone_lr_mult = 0.05
    """

    # Freeze everything first.
    for p in backbone.parameters():
        p.requires_grad = False

    enable_tuning = bool(getattr(cfg, "supervised_enable_backbone_tuning", False))

    if not enable_tuning:
        print("[supervised] backbone tuning disabled: full backbone frozen.")
        return

    n_last = int(getattr(cfg, "supervised_unfreeze_last_n_blocks", 0))
    patterns = _as_tuple(getattr(cfg, "supervised_unfreeze_patterns", ()))
    train_norms = bool(getattr(cfg, "supervised_train_norm_layers", False))

    # --------------------------------------------------------
    # Option 1: unfreeze last N transformer blocks.
    # DINOv2 usually exposes blocks as backbone.blocks.
    # --------------------------------------------------------
    if n_last > 0:
        if hasattr(backbone, "blocks"):
            blocks = getattr(backbone, "blocks")
            n_blocks = len(blocks)
            start = max(0, n_blocks - n_last)

            for idx in range(start, n_blocks):
                for p in blocks[idx].parameters():
                    p.requires_grad = True

            print(
                f"[supervised] unfroze last {n_last} DINO blocks: "
                f"blocks.{start} to blocks.{n_blocks - 1}"
            )
        else:
            print(
                "[supervised][WARN] backbone has no attribute 'blocks'; "
                "cannot unfreeze last-N blocks."
            )

    # --------------------------------------------------------
    # Option 2: pattern-based unfreezing.
    # Example:
    #   CFG.supervised_unfreeze_patterns = ('blocks.11', 'norm')
    # --------------------------------------------------------
    if len(patterns) > 0:
        matched = []

        for name, p in backbone.named_parameters():
            if _matches_any_pattern(name, patterns):
                p.requires_grad = True
                matched.append(name)

        print(f"[supervised] pattern-unfroze {len(matched)} backbone tensors.")

        if len(matched) > 0:
            print("  first matched tensors:")
            for x in matched[:12]:
                print("   ", x)

    # --------------------------------------------------------
    # Option 3: optionally unfreeze all norm layers.
    # --------------------------------------------------------
    if train_norms:
        norm_types = (
            nn.LayerNorm,
            nn.BatchNorm1d,
            nn.BatchNorm2d,
            nn.BatchNorm3d,
            nn.GroupNorm,
            nn.InstanceNorm1d,
            nn.InstanceNorm2d,
            nn.InstanceNorm3d,
        )

        norm_count = 0

        for module_name, module in backbone.named_modules():
            if isinstance(module, norm_types):
                for p in module.parameters(recurse=False):
                    p.requires_grad = True
                    norm_count += 1

        print(f"[supervised] unfroze norm-layer tensors: {norm_count}")

    total = sum(p.numel() for p in backbone.parameters())
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)

    print(
        f"[supervised] backbone trainable params: "
        f"{trainable:,} / {total:,} ({100.0 * trainable / max(total, 1):.3f}%)"
    )


def supervised_backbone_is_trainable(model: nn.Module) -> bool:
    return any(p.requires_grad for p in model.backbone.parameters())


def build_supervised_optimizer(model: nn.Module, cfg: Config):
    """
    Creates separate AdamW parameter groups.

    Group 1:
      classifier + pooling parameters
      LR = CFG.supervised_lr

    Group 2:
      selected DINOv2 backbone parameters
      LR = CFG.supervised_lr * CFG.supervised_backbone_lr_mult
    """

    head_params = []
    backbone_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if name.startswith("backbone."):
            backbone_params.append(p)
        else:
            head_params.append(p)

    param_groups = []

    if len(head_params) > 0:
        param_groups.append({
            "params": head_params,
            "lr": float(cfg.supervised_lr),
            "base_lr": float(cfg.supervised_lr),
            "name": "classifier_pool",
        })

    if len(backbone_params) > 0:
        backbone_lr = (
            float(cfg.supervised_lr)
            * float(getattr(cfg, "supervised_backbone_lr_mult", 0.05))
        )

        param_groups.append({
            "params": backbone_params,
            "lr": backbone_lr,
            "base_lr": backbone_lr,
            "name": "selected_backbone_layers",
        })

    print("[supervised] optimizer parameter groups:")

    for g in param_groups:
        n_params = sum(p.numel() for p in g["params"])
        print(
            f"  {g['name']}: tensors={len(g['params'])}, "
            f"params={n_params:,}, base_lr={g['base_lr']:.2e}"
        )

    if len(param_groups) == 0:
        raise RuntimeError("No trainable parameters found for supervised branch.")

    return torch.optim.AdamW(
        param_groups,
        weight_decay=float(cfg.supervised_weight_decay),
    )


def set_supervised_epoch_lr(optimizer, epoch: int, total_epochs: int):
    """
    Warmup + cosine decay.
    Keeps relative LR difference between classifier/pool and backbone groups.
    """

    warmup_epochs = max(0, int(getattr(CFG, "supervised_warmup_epochs", 0)))

    if warmup_epochs > 0 and epoch < warmup_epochs:
        scale = float(epoch + 1) / float(warmup_epochs)
    else:
        denom = max(int(total_epochs) - warmup_epochs - 1, 1)
        progress = (epoch - warmup_epochs) / denom
        progress = min(max(progress, 0.0), 1.0)
        scale = 0.5 * (1.0 + math.cos(math.pi * progress))

    min_scale = float(CFG.supervised_min_lr) / max(float(CFG.supervised_lr), 1e-12)
    scale = max(scale, min_scale)

    for g in optimizer.param_groups:
        g["lr"] = float(g["base_lr"]) * float(scale)

    return optimizer.param_groups[0]["lr"]


# ------------------------------------------------------------
# Branch B model: supervised classifier with optional DINO tuning
# ------------------------------------------------------------

class PatientFrozenDinoClassifier(nn.Module):
    """
    Supervised patient classifier on top of DINOv2 slice features.

    Historical name retained for notebook compatibility, but this patched
    version supports:
      1. fully frozen DINOv2 backbone
      2. user-defined partial unfreezing
      3. last-N DINO block fine-tuning
      4. optional norm-layer tuning
    """

    def __init__(
        self,
        backbone: nn.Module,
        embed_dim: int,
        num_classes: int,
        cfg: Config,
    ):
        super().__init__()

        self.backbone = backbone
        self.embed_dim = int(embed_dim)
        self.cfg = cfg

        self.pool = PatientAttentionPool(embed_dim)

        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(float(cfg.supervised_dropout)),
            nn.Linear(embed_dim, num_classes),
        )

        # Selective backbone tuning.
        configure_supervised_backbone_trainability(self.backbone, cfg)

        # Optional pool tuning.
        if not bool(cfg.supervised_train_pool):
            for p in self.pool.parameters():
                p.requires_grad = False

    def backbone_has_trainable_layers(self) -> bool:
        return any(p.requires_grad for p in self.backbone.parameters())

    def _encode_cls_tokens(self, bag_5d: torch.Tensor):
        """
        Critical patch:
          - If selected backbone layers are trainable and gradients are enabled,
            the backbone forward pass is performed with gradients.
          - If the backbone is frozen or model is in evaluation/no-grad context,
            the forward pass uses torch.no_grad().
        """

        B, S, C, H, W = bag_5d.shape
        x = bag_5d.reshape(B * S, C, H, W)

        chunks = []
        chunk_size = max(1, int(self.cfg.frozen_backbone_chunk_size))

        use_grad = (
            torch.is_grad_enabled()
            and self.backbone_has_trainable_layers()
        )

        ctx = torch.enable_grad() if use_grad else torch.no_grad()

        with ctx:
            for xs in x.split(chunk_size, dim=0):
                feats = self.backbone.forward_features(xs)
                chunks.append(feats["x_norm_clstoken"])

        cls_tokens = torch.cat(chunks, dim=0).reshape(B, S, -1)

        return cls_tokens

    def forward_bag_features(self, bag_5d: torch.Tensor):
        cls_tokens = self._encode_cls_tokens(bag_5d)
        patient_tokens, slice_weights = self.pool(cls_tokens)

        return {
            "patient_tokens": patient_tokens,
            "slice_tokens": cls_tokens,
            "slice_weights": slice_weights,
        }

    def forward(self, bag_5d: torch.Tensor):
        out = self.forward_bag_features(bag_5d)
        out["logits"] = self.classifier(out["patient_tokens"])
        return out


# ------------------------------------------------------------
# Supervised utilities
# ------------------------------------------------------------

def _num_classes_from_entries(entries: List[Dict]) -> int:
    labels = sorted({
        int(e["label"])
        for e in entries
        if e.get("label") is not None and int(e["label"]) >= 0
    })

    if not labels:
        raise ValueError("No labelled training entries found for supervised branch.")

    if labels != list(range(max(labels) + 1)):
        print(
            f"[WARN] Labels are not contiguous from 0: {labels}. "
            f"Classifier will still use num_classes={max(labels) + 1}."
        )

    return max(labels) + 1


def _bag_from_supervised_batch(batch: Dict, train: bool):
    if "bag" in batch:
        return batch["bag"]

    if "global_crops" in batch and len(batch["global_crops"]) > 0:
        return batch["global_crops"][0]

    raise KeyError("Batch does not contain 'bag' or 'global_crops'.")


def _classification_metrics_from_logits(
    logits: torch.Tensor,
    labels: torch.Tensor,
    num_classes: int,
):
    y_true = labels.detach().cpu().numpy().astype(int)
    y_prob = torch.softmax(logits.detach().float(), dim=-1).cpu().numpy()
    y_pred = y_prob.argmax(axis=1)

    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)) if len(y_true) else np.nan,
        "balanced_accuracy": (
            float(balanced_accuracy_score(y_true, y_pred))
            if len(np.unique(y_true)) > 1
            else np.nan
        ),
        "f1_macro": (
            float(f1_score(y_true, y_pred, average="macro", zero_division=0))
            if len(y_true)
            else np.nan
        ),
    }

    try:
        if num_classes == 2 and len(np.unique(y_true)) == 2:
            out["auroc"] = float(roc_auc_score(y_true, y_prob[:, 1]))
        elif len(np.unique(y_true)) > 1:
            out["auroc_ovr"] = float(
                roc_auc_score(y_true, y_prob, multi_class="ovr")
            )
        else:
            out["auroc"] = np.nan
    except Exception:
        out["auroc"] = np.nan

    return out


def run_supervised_epoch(
    model,
    loader,
    optimizer,
    criterion,
    num_classes: int,
    train: bool = True,
):
    model.train(train)

    losses = []
    logits_all = []
    labels_all = []

    if train:
        optimizer.zero_grad(set_to_none=True)

    iterator = (
        _make_progress_bar(loader, desc="sup train" if train else "sup val")
        if "_make_progress_bar" in globals()
        else loader
    )

    accum_steps = max(1, int(CFG.supervised_grad_accum_steps))

    for it, batch in enumerate(iterator):
        bag = _bag_from_supervised_batch(batch, train=train).to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True).long()

        valid = labels >= 0

        if not valid.any():
            continue

        bag = bag[valid]
        labels = labels[valid]

        with torch.set_grad_enabled(train):
            with _branch_autocast_context():
                out = model(bag)
                logits = out["logits"]
                loss = criterion(logits.float(), labels)

            if train:
                (loss / accum_steps).backward()

                if ((it + 1) % accum_steps == 0) or (it == len(loader) - 1):
                    if CFG.grad_clip_norm is not None and CFG.grad_clip_norm > 0:
                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(),
                            CFG.grad_clip_norm,
                        )

                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

        losses.append(float(loss.detach().cpu()))
        logits_all.append(logits.detach().float().cpu())
        labels_all.append(labels.detach().cpu())

    if not losses:
        return {
            "loss": np.nan,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "f1_macro": np.nan,
            "auroc": np.nan,
        }

    logits_cat = torch.cat(logits_all, dim=0)
    labels_cat = torch.cat(labels_all, dim=0)

    metrics = _classification_metrics_from_logits(
        logits_cat,
        labels_cat,
        num_classes,
    )

    metrics["loss"] = float(np.mean(losses))

    return metrics


@torch.no_grad()
def export_supervised_predictions(
    model,
    loader,
    split_name: str,
    num_classes: int,
    branch_name: str,
):
    model.eval()

    rows = []

    for batch in loader:
        bag = _bag_from_supervised_batch(batch, train=False).to(device, non_blocking=True)

        out = model(bag)
        prob = torch.softmax(out["logits"].float(), dim=-1).cpu().numpy()
        pred = prob.argmax(axis=1)

        for i in range(prob.shape[0]):
            row = {
                "split": split_name,
                "patient_id": batch["patient_id"][i],
                "path": batch["path"][i],
                "label": int(batch["label"][i].item()),
                "class_name": batch["class_name"][i],
                "pred_label": int(pred[i]),
            }

            for k in range(num_classes):
                row[f"prob_class_{k}"] = float(prob[i, k])

            rows.append(row)

    df = pd.DataFrame(rows)

    out_dir = BRANCH_ROOT / branch_name
    out_dir.mkdir(parents=True, exist_ok=True)

    df.to_csv(out_dir / f"{split_name}_predictions.csv", index=False)

    return df


# ============================================================
# Branch A: frozen DINO baseline embeddings
# ============================================================

if branch_enabled("frozen_dino_baseline"):
    print("\n=== Branch A: frozen_dino_baseline ===")

    baseline_model, baseline_embed_dim = build_frozen_patient_dino_encoder(CFG)

    frozen_baseline_exports = export_branch_embeddings(
        baseline_model,
        branch_name="frozen_dino_baseline",
        embed_dim_fallback=baseline_embed_dim,
    )

    del baseline_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

else:
    print("Skipping Branch A: frozen_dino_baseline")


# ============================================================
# Branch B: supervised DINO classifier with optional unfreezing
# ============================================================

if branch_enabled("supervised_frozen_classifier"):
    print("\n=== Branch B: supervised_frozen_classifier ===")

    if len(train_entries) == 0 or len(val_entries) == 0:
        print("[WARN] Missing labelled train/val entries. Skipping supervised branch.")

    else:
        sup_num_classes = _num_classes_from_entries(train_entries)

        sup_backbone, sup_embed_dim = load_backbone(CFG)

        if sup_embed_dim is None:
            sup_embed_dim = getattr(sup_backbone, "embed_dim", None)

        if sup_embed_dim is None:
            raise ValueError("Could not infer sup_embed_dim from DINO backbone.")

        supervised_model = PatientFrozenDinoClassifier(
            sup_backbone,
            sup_embed_dim,
            sup_num_classes,
            CFG,
        ).to(device)

        sup_params = [p for p in supervised_model.parameters() if p.requires_grad]
        sup_n_trainable = sum(p.numel() for p in sup_params)

        print(f"Supervised DINO-classifier trainable tensors: {len(sup_params)}")
        print(f"Supervised DINO-classifier trainable params: {sup_n_trainable:,}")
        print(f"Supervised classes: {sup_num_classes}")

        label_counts = np.bincount(
            [int(e["label"]) for e in train_entries],
            minlength=sup_num_classes,
        )

        if CFG.supervised_use_class_weights:
            weights = label_counts.sum() / np.maximum(label_counts, 1)
            weights = weights / weights.mean()

            class_weight_tensor = torch.tensor(
                weights,
                dtype=torch.float32,
                device=device,
            )

            print("Supervised class counts:", label_counts.tolist())
            print("Supervised class weights:", np.round(weights, 3).tolist())

        else:
            class_weight_tensor = None

        criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)

        # Patched optimiser with separate classifier/pool and selected-backbone LR groups.
        sup_optimizer = build_supervised_optimizer(supervised_model, CFG)

        sup_history = []
        sup_best_metric = -float("inf")
        sup_bad_epochs = 0

        sup_ckpt_best = OUT_DIR / "checkpoints" / "supervised_frozen_classifier_best.pt"
        sup_ckpt_last = OUT_DIR / "checkpoints" / "supervised_frozen_classifier_last.pt"

        sup_ckpt_best.parent.mkdir(parents=True, exist_ok=True)

        for epoch in range(int(CFG.supervised_epochs)):
            lr_now = set_supervised_epoch_lr(
                sup_optimizer,
                epoch=epoch,
                total_epochs=int(CFG.supervised_epochs),
            )

            train_m = run_supervised_epoch(
                supervised_model,
                train_ssl_loader,
                sup_optimizer,
                criterion,
                sup_num_classes,
                train=True,
            )

            val_m = run_supervised_epoch(
                supervised_model,
                val_eval_loader,
                sup_optimizer,
                criterion,
                sup_num_classes,
                train=False,
            )

            row = {
                "epoch": epoch + 1,
                "lr": lr_now,
            }

            # Log all param-group LRs.
            for gi, group in enumerate(sup_optimizer.param_groups):
                row[f"lr_group_{gi}_{group.get('name', 'group')}"] = group["lr"]

            row.update({f"train_{k}": v for k, v in train_m.items()})
            row.update({f"val_{k}": v for k, v in val_m.items()})

            sup_history.append(row)

            monitor = val_m.get("balanced_accuracy", np.nan)

            if not np.isfinite(monitor):
                monitor = val_m.get("accuracy", -np.inf)

            print(
                f"[supervised] epoch {epoch+1:03d}/{CFG.supervised_epochs} | "
                f"train_loss={train_m['loss']:.4f} | "
                f"val_loss={val_m['loss']:.4f} | "
                f"val_acc={val_m.get('accuracy', np.nan):.3f} | "
                f"val_bacc={val_m.get('balanced_accuracy', np.nan):.3f} | "
                f"val_f1={val_m.get('f1_macro', np.nan):.3f} | "
                f"lr={lr_now:.2e}"
            )

            torch.save(
                {
                    "model": supervised_model.state_dict(),
                    "epoch": epoch,
                    "history": sup_history,
                    "cfg": asdict(CFG),
                    "num_classes": sup_num_classes,
                    "supervised_enable_backbone_tuning": getattr(
                        CFG,
                        "supervised_enable_backbone_tuning",
                        False,
                    ),
                    "supervised_unfreeze_last_n_blocks": getattr(
                        CFG,
                        "supervised_unfreeze_last_n_blocks",
                        0,
                    ),
                    "supervised_unfreeze_patterns": getattr(
                        CFG,
                        "supervised_unfreeze_patterns",
                        (),
                    ),
                    "supervised_train_norm_layers": getattr(
                        CFG,
                        "supervised_train_norm_layers",
                        False,
                    ),
                },
                str(sup_ckpt_last),
            )

            improved = (
                np.isfinite(monitor)
                and monitor
                > sup_best_metric + float(CFG.supervised_early_stopping_min_delta)
            )

            if improved:
                sup_best_metric = float(monitor)
                sup_bad_epochs = 0

                torch.save(
                    {
                        "model": supervised_model.state_dict(),
                        "epoch": epoch,
                        "history": sup_history,
                        "cfg": asdict(CFG),
                        "num_classes": sup_num_classes,
                        "supervised_enable_backbone_tuning": getattr(
                            CFG,
                            "supervised_enable_backbone_tuning",
                            False,
                        ),
                        "supervised_unfreeze_last_n_blocks": getattr(
                            CFG,
                            "supervised_unfreeze_last_n_blocks",
                            0,
                        ),
                        "supervised_unfreeze_patterns": getattr(
                            CFG,
                            "supervised_unfreeze_patterns",
                            (),
                        ),
                        "supervised_train_norm_layers": getattr(
                            CFG,
                            "supervised_train_norm_layers",
                            False,
                        ),
                    },
                    str(sup_ckpt_best),
                )

                print(
                    f"  -> saved new supervised best: {sup_ckpt_best} "
                    f"(monitor={sup_best_metric:.4f})"
                )

            else:
                sup_bad_epochs += 1
                print(
                    f"  -> no supervised improvement for "
                    f"{sup_bad_epochs}/{CFG.supervised_early_stopping_patience} epochs"
                )

            if (
                int(CFG.supervised_early_stopping_patience) > 0
                and sup_bad_epochs >= int(CFG.supervised_early_stopping_patience)
            ):
                print("[supervised] early stopping triggered.")
                break

        sup_out_dir = BRANCH_ROOT / "supervised_frozen_classifier"
        sup_out_dir.mkdir(parents=True, exist_ok=True)

        pd.DataFrame(sup_history).to_csv(
            sup_out_dir / "supervised_training_history.csv",
            index=False,
        )

        if sup_ckpt_best.exists():
            payload = torch.load(str(sup_ckpt_best), map_location=device)
            supervised_model.load_state_dict(payload["model"], strict=True)
            print(f"Loaded supervised best checkpoint for export: {sup_ckpt_best}")

        supervised_exports = export_branch_embeddings(
            supervised_model,
            branch_name="supervised_frozen_classifier",
            embed_dim_fallback=sup_embed_dim,
        )

        if len(train_eval_loader):
            export_supervised_predictions(
                supervised_model,
                train_eval_loader,
                "train",
                sup_num_classes,
                "supervised_frozen_classifier",
            )

        if len(val_eval_loader):
            export_supervised_predictions(
                supervised_model,
                val_eval_loader,
                "val",
                sup_num_classes,
                "supervised_frozen_classifier",
            )

        if len(oac_eval_loader):
            export_supervised_predictions(
                supervised_model,
                oac_eval_loader,
                "oac",
                sup_num_classes,
                "supervised_frozen_classifier",
            )

        del supervised_model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

else:
    print("Skipping Branch B: supervised_frozen_classifier")

In [ ]:
# -----------------------------
# Optimizer / schedules / checkpoint helpers
# -----------------------------
optimizer = torch.optim.AdamW(trainable_params, lr=CFG.lr, weight_decay=CFG.weight_decay)

niter_per_ep = max(len(train_ssl_loader), 1)
momentum_schedule = cosine_scheduler(
    CFG.momentum_teacher_base, 1.0, CFG.epochs, niter_per_ep
)

if CFG.use_lr_schedule:
    lr_schedule = cosine_scheduler(
        CFG.lr,
        CFG.min_lr,
        CFG.epochs,
        niter_per_ep,
        warmup_epochs=CFG.lr_warmup_epochs,
        start_warmup_value=min(CFG.lr * 0.1, CFG.lr),
    )
else:
    lr_schedule = np.ones(CFG.epochs * niter_per_ep, dtype=np.float32) * float(CFG.lr)

wd_schedule = cosine_scheduler(
    CFG.weight_decay,
    CFG.final_weight_decay,
    CFG.epochs,
    niter_per_ep,
    warmup_epochs=0,
    start_warmup_value=CFG.weight_decay,
)

dino_loss_fn = DINOLoss(
    out_dim=CFG.head_out_dim,
    n_student_views=CFG.n_global_crops + CFG.n_local_crops,
    warmup_teacher_temp=CFG.warmup_teacher_temp,
    teacher_temp=CFG.teacher_temp,
    warmup_teacher_temp_epochs=CFG.warmup_teacher_temp_epochs,
    nepochs=CFG.epochs,
    student_temp=CFG.student_temp,
    center_momentum=CFG.center_momentum,
).to(device)

amp_enabled = bool(CFG.use_amp and torch.cuda.is_available())
if amp_enabled and CFG.amp_dtype.lower() == "bfloat16" and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
else:
    amp_dtype = torch.float16

scaler = torch.cuda.amp.GradScaler(enabled=(amp_enabled and amp_dtype == torch.float16))

def autocast_context():
    if not amp_enabled:
        return torch.cuda.amp.autocast(enabled=False)
    return torch.cuda.amp.autocast(dtype=amp_dtype)

def set_optimizer_schedules(optimizer, schedule_index: int):
    schedule_index = min(max(int(schedule_index), 0), len(lr_schedule) - 1)
    lr = float(lr_schedule[schedule_index])
    wd = float(wd_schedule[min(schedule_index, len(wd_schedule) - 1)])
    for group in optimizer.param_groups:
        group["lr"] = lr
        group["weight_decay"] = wd
    return lr, wd

def current_optimizer_lr(optimizer) -> float:
    return float(optimizer.param_groups[0].get("lr", np.nan)) if optimizer.param_groups else np.nan

def cancel_gradients_last_layer(model: nn.Module, epoch: int):
    if int(CFG.freeze_last_layer_epochs) <= 0 or epoch >= int(CFG.freeze_last_layer_epochs):
        return
    if hasattr(model, "head") and hasattr(model.head, "last_layer"):
        for p in model.head.last_layer.parameters():
            p.grad = None

def save_ssl_checkpoint(path: Path, student, teacher, optimizer, epoch, history):
    payload = {
        "student": student.state_dict(),
        "teacher": teacher.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
        "history": history,
        "cfg": asdict(CFG),
    }
    torch.save(payload, str(path))


In [ ]:
# -----------------------------
# Training / validation loops
# -----------------------------
def _teacher_probs_from_global_views(teacher, global_views, epoch, update_center: bool = True):
    teacher_probs = []
    teacher_raw = []
    temp = float(dino_loss_fn.teacher_temp_schedule[min(epoch, len(dino_loss_fn.teacher_temp_schedule) - 1)])

    with torch.no_grad():
        for view in global_views:
            with autocast_context():
                out = teacher(view)
                proj = out["proj"]
            proj = proj.detach().float()
            teacher_raw.append(proj)
            probs = F.softmax((proj - dino_loss_fn.center.float()) / temp, dim=-1)
            teacher_probs.append(probs)

    if update_center and len(teacher_raw) > 0:
        dino_loss_fn.update_center(torch.cat(teacher_raw, dim=0))
    return teacher_probs, teacher_raw

def _student_view_loss(student_proj, teacher_probs, student_view_idx, n_teacher):
    s = F.log_softmax(student_proj.float() / dino_loss_fn.student_temp, dim=-1)
    total = None
    n_terms = 0
    for iq, q in enumerate(teacher_probs):
        if student_view_idx < n_teacher and student_view_idx == iq:
            continue
        term = torch.sum(-q.detach() * s, dim=-1).mean()
        total = term if total is None else (total + term)
        n_terms += 1
    if total is None:
        return student_proj.new_tensor(0.0)
    return total / max(n_terms, 1)

def _student_entropy(student_proj: torch.Tensor) -> float:
    with torch.no_grad():
        p = F.softmax(student_proj.detach().float() / dino_loss_fn.student_temp, dim=-1)
        return float(probability_entropy(p).mean().cpu())

def _teacher_entropy(teacher_probs: List[torch.Tensor]) -> float:
    if not teacher_probs:
        return np.nan
    vals = [probability_entropy(p.detach().float()).mean().detach().cpu().item() for p in teacher_probs]
    return float(np.mean(vals))

def _output_std(tensors: List[torch.Tensor]) -> float:
    if not tensors:
        return np.nan
    x = torch.cat([t.detach().float().cpu() for t in tensors], dim=0)
    return float(x.std(dim=0).mean().item())

def _cuda_mem_string():
    if not torch.cuda.is_available() or not CFG.show_cuda_memory:
        return ""
    alloc = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    max_alloc = torch.cuda.max_memory_allocated() / (1024 ** 3)
    return f"alloc={alloc:.2f}G reserved={reserved:.2f}G max={max_alloc:.2f}G"

def _make_progress_bar(loader, desc):
    total = len(loader) if hasattr(loader, "__len__") else None
    if CFG.show_batch_progress and tqdm is not None:
        return tqdm(loader, total=total, desc=desc, leave=False, dynamic_ncols=True)
    return loader

def _step_optimizer_and_teacher(student, teacher, optimizer, scaler, epoch, schedule_index):
    cancel_gradients_last_layer(student, epoch)

    grad_norm = np.nan
    if scaler.is_enabled():
        scaler.unscale_(optimizer)
        if CFG.grad_clip_norm is not None and CFG.grad_clip_norm > 0:
            grad_norm_tensor = torch.nn.utils.clip_grad_norm_(student.parameters(), CFG.grad_clip_norm)
            grad_norm = float(grad_norm_tensor.detach().cpu())
        scaler.step(optimizer)
        scaler.update()
    else:
        if CFG.grad_clip_norm is not None and CFG.grad_clip_norm > 0:
            grad_norm_tensor = torch.nn.utils.clip_grad_norm_(student.parameters(), CFG.grad_clip_norm)
            grad_norm = float(grad_norm_tensor.detach().cpu())
        optimizer.step()

    optimizer.zero_grad(set_to_none=True)

    momentum = float(momentum_schedule[min(schedule_index, len(momentum_schedule) - 1)])
    with torch.no_grad():
        for ps, pt in zip(student.parameters(), teacher.parameters()):
            pt.data.mul_(momentum).add_((1.0 - momentum) * ps.data)
    return grad_norm, momentum

def run_ssl_epoch(student, teacher, loader, optimizer, dino_loss_fn, epoch, train=True):
    if train:
        student.train()
        teacher.eval()
        optimizer.zero_grad(set_to_none=True)
        phase = "train"
    else:
        student.eval()
        teacher.eval()
        phase = "val"

    meter = {
        "loss_total": [],
        "loss_dino": [],
        "loss_koleo": [],
        "teacher_entropy": [],
        "student_entropy": [],
        "teacher_output_std": [],
        "center_norm": [],
        "grad_norm": [],
        "lr": [],
        "weight_decay": [],
        "teacher_momentum": [],
    }

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    iterator = _make_progress_bar(loader, desc=f"{phase} ep {epoch+1}")
    accum_steps = max(int(CFG.grad_accum_steps), 1)

    for it, batch in enumerate(iterator):
        schedule_index = epoch * niter_per_ep + min(it, niter_per_ep - 1)
        if train:
            lr_now, wd_now = set_optimizer_schedules(optimizer, schedule_index)
        else:
            lr_now, wd_now = current_optimizer_lr(optimizer), np.nan

        global_views = [x.to(device, non_blocking=True) for x in batch["global_crops"]]
        local_views  = [x.to(device, non_blocking=True) for x in batch["local_crops"]]
        all_student_views = global_views + local_views

        if CFG.print_first_batch_shapes and it == 0:
            print(f"[{phase}] epoch {epoch+1} first batch:")
            for j, gv in enumerate(global_views):
                print(f"  global[{j}] shape = {tuple(gv.shape)}")
            for j, lv in enumerate(local_views):
                print(f"  local[{j}]  shape = {tuple(lv.shape)}")
            if torch.cuda.is_available():
                print("  " + _cuda_mem_string())

        update_center = bool(train or CFG.update_center_on_val)
        teacher_probs, teacher_raw = _teacher_probs_from_global_views(
            teacher,
            global_views,
            epoch,
            update_center=update_center,
        )

        total_loss_value = 0.0
        total_dino_value = 0.0
        total_koleo_value = 0.0
        student_entropy_values = []
        n_student_loss_terms = 0

        # Process each student view separately so activations from one view
        # do not stay alive while the next view is forwarded.
        n_views = max(len(all_student_views), 1)
        for view_idx, view in enumerate(all_student_views):
            with torch.set_grad_enabled(train):
                with autocast_context():
                    out = student(view)
                    loss_dino_view = _student_view_loss(
                        out["proj"], teacher_probs, view_idx, n_teacher=len(teacher_probs)
                    )

                    loss_koleo_view = torch.tensor(0.0, device=device)
                    if CFG.use_koleo and out["patient_tokens"].shape[0] > 1:
                        loss_koleo_view = koleo_loss(out["patient_tokens"]) * CFG.koleo_weight

                    loss_view = loss_dino_view + loss_koleo_view

                student_entropy_values.append(_student_entropy(out["proj"]))

                if train:
                    # True gradient accumulation across batches and across student views.
                    loss_for_backward = loss_view / float(n_views * accum_steps)
                    if scaler.is_enabled():
                        scaler.scale(loss_for_backward).backward()
                    else:
                        loss_for_backward.backward()

            total_loss_value += float(loss_view.detach().cpu())
            total_dino_value += float(loss_dino_view.detach().cpu())
            total_koleo_value += float(loss_koleo_view.detach().cpu())
            n_student_loss_terms += 1

            del out
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        grad_norm = np.nan
        teacher_momentum = np.nan
        if train and (((it + 1) % accum_steps == 0) or (it == len(loader) - 1)):
            grad_norm, teacher_momentum = _step_optimizer_and_teacher(
                student,
                teacher,
                optimizer,
                scaler,
                epoch,
                schedule_index,
            )

        denom = max(n_student_loss_terms, 1)
        batch_total = total_loss_value / denom
        batch_dino = total_dino_value / denom
        batch_koleo = total_koleo_value / denom

        meter["loss_total"].append(batch_total)
        meter["loss_dino"].append(batch_dino)
        meter["loss_koleo"].append(batch_koleo)
        meter["teacher_entropy"].append(_teacher_entropy(teacher_probs))
        meter["student_entropy"].append(float(np.mean(student_entropy_values)) if student_entropy_values else np.nan)
        meter["teacher_output_std"].append(_output_std(teacher_raw))
        meter["center_norm"].append(float(dino_loss_fn.center.detach().float().norm().cpu()))
        meter["grad_norm"].append(grad_norm)
        meter["lr"].append(lr_now)
        meter["weight_decay"].append(wd_now)
        meter["teacher_momentum"].append(teacher_momentum)

        if CFG.show_batch_progress and tqdm is not None and hasattr(iterator, "set_postfix"):
            if (it % max(int(CFG.progress_update_every), 1) == 0) or (it == len(loader) - 1):
                postfix = {
                    "batch_loss": f"{batch_total:.4f}",
                    "avg_loss": f"{np.mean(meter['loss_total']):.4f}",
                    "lr": f"{lr_now:.2e}",
                    "H_t": f"{np.nanmean(meter['teacher_entropy']):.2f}",
                }
                mem_str = _cuda_mem_string()
                if mem_str:
                    postfix["mem"] = mem_str
                iterator.set_postfix(postfix)

        elif (it % max(int(CFG.progress_update_every), 1) == 0):
            msg = (
                f"[{phase}] epoch {epoch+1} iter {it+1}/{len(loader)} | "
                f"batch_loss={batch_total:.4f} | avg_loss={np.mean(meter['loss_total']):.4f} | lr={lr_now:.2e}"
            )
            mem_str = _cuda_mem_string()
            if mem_str:
                msg += " | " + mem_str
            print(msg)

    out = {}
    for k, v in meter.items():
        finite_vals = [x for x in v if np.isfinite(x)]
        out[k] = float(np.mean(finite_vals)) if len(finite_vals) else np.nan
    return out


In [ ]:
if not branch_enabled("dino_ssl"):
    print("Skipping Branch C: dino_ssl")
    history = {
        "train_loss_total": [], "train_loss_dino": [], "train_loss_koleo": [],
        "val_loss_total": [], "val_loss_dino": [], "val_loss_koleo": [],
        "train_teacher_entropy": [], "val_teacher_entropy": [],
        "train_student_entropy": [], "val_student_entropy": [],
        "train_teacher_output_std": [], "val_teacher_output_std": [],
        "center_norm": [], "grad_norm": [], "lr": [], "weight_decay": [],
        "teacher_momentum": [], "monitor_loss": [],
    }
else:
    # -----------------------------
    # Run SSL training with mitigation controls
    # -----------------------------
    history = {
        "train_loss_total": [],
        "train_loss_dino": [],
        "train_loss_koleo": [],
        "val_loss_total": [],
        "val_loss_dino": [],
        "val_loss_koleo": [],
        "train_teacher_entropy": [],
        "val_teacher_entropy": [],
        "train_student_entropy": [],
        "val_student_entropy": [],
        "train_teacher_output_std": [],
        "val_teacher_output_std": [],
        "center_norm": [],
        "grad_norm": [],
        "lr": [],
        "weight_decay": [],
        "teacher_momentum": [],
        "monitor_loss": [],
    }

    best_monitor = float("inf")
    bad_epochs = 0
    ckpt_best = OUT_DIR / "checkpoints" / "ssl_best.pt"
    ckpt_last = OUT_DIR / "checkpoints" / "ssl_last.pt"

    entropy_floor = float(CFG.collapse_entropy_floor_fraction) * math.log(max(int(CFG.head_out_dim), 2))

    if len(train_ssl_loader) == 0:
        print("No training data found. Skipping training.")
    else:
        print(f"Starting SSL training with {len(train_ssl_loader)} train batches and {len(val_ssl_loader)} val batches per epoch.")
        print(f"Batch progress enabled: {CFG.show_batch_progress}, update_every={CFG.progress_update_every}, cuda_mem={CFG.show_cuda_memory}")
        print("Mitigation active:")
        print(f"  normalization_scope={CFG.normalization_scope} | clip_percentiles={CFG.clip_percentiles}")
        print(f"  deterministic_val_ssl={CFG.deterministic_val_ssl}")
        print(f"  update_center_on_val={CFG.update_center_on_val}")
        print(f"  lr={CFG.lr:.2e}, min_lr={CFG.min_lr:.2e}, warmup_epochs={CFG.lr_warmup_epochs}")
        print(f"  grad_accum_steps={CFG.grad_accum_steps}, freeze_last_layer_epochs={CFG.freeze_last_layer_epochs}")
        print(f"  early_stopping_patience={CFG.early_stopping_patience}")

        for epoch in range(CFG.epochs):
            t0 = time.time()

            if torch.cuda.is_available() and CFG.empty_cache_each_epoch:
                torch.cuda.empty_cache()

            train_metrics = run_ssl_epoch(student, teacher, train_ssl_loader, optimizer, dino_loss_fn, epoch, train=True)
            val_metrics = run_ssl_epoch(student, teacher, val_ssl_loader, optimizer, dino_loss_fn, epoch, train=False) if len(val_ssl_loader) > 0 else {
                "loss_total": np.nan, "loss_dino": np.nan, "loss_koleo": np.nan,
                "teacher_entropy": np.nan, "student_entropy": np.nan, "teacher_output_std": np.nan,
                "center_norm": train_metrics.get("center_norm", np.nan), "grad_norm": np.nan,
                "lr": train_metrics.get("lr", np.nan), "weight_decay": np.nan, "teacher_momentum": np.nan,
            }

            history["train_loss_total"].append(train_metrics["loss_total"])
            history["train_loss_dino"].append(train_metrics["loss_dino"])
            history["train_loss_koleo"].append(train_metrics["loss_koleo"])
            history["val_loss_total"].append(val_metrics["loss_total"])
            history["val_loss_dino"].append(val_metrics["loss_dino"])
            history["val_loss_koleo"].append(val_metrics["loss_koleo"])
            history["train_teacher_entropy"].append(train_metrics.get("teacher_entropy", np.nan))
            history["val_teacher_entropy"].append(val_metrics.get("teacher_entropy", np.nan))
            history["train_student_entropy"].append(train_metrics.get("student_entropy", np.nan))
            history["val_student_entropy"].append(val_metrics.get("student_entropy", np.nan))
            history["train_teacher_output_std"].append(train_metrics.get("teacher_output_std", np.nan))
            history["val_teacher_output_std"].append(val_metrics.get("teacher_output_std", np.nan))
            history["center_norm"].append(train_metrics.get("center_norm", np.nan))
            history["grad_norm"].append(train_metrics.get("grad_norm", np.nan))
            history["lr"].append(train_metrics.get("lr", np.nan))
            history["weight_decay"].append(train_metrics.get("weight_decay", np.nan))
            history["teacher_momentum"].append(train_metrics.get("teacher_momentum", np.nan))

            monitor_loss = val_metrics["loss_total"] if np.isfinite(val_metrics["loss_total"]) else train_metrics["loss_total"]
            history["monitor_loss"].append(monitor_loss)

            dt = time.time() - t0
            print(
                f"Epoch {epoch+1:03d}/{CFG.epochs} | "
                f"train_total={train_metrics['loss_total']:.4f} | "
                f"val_total={val_metrics['loss_total']:.4f} | "
                f"H_t={train_metrics.get('teacher_entropy', np.nan):.3f} | "
                f"std_t={train_metrics.get('teacher_output_std', np.nan):.4e} | "
                f"grad={train_metrics.get('grad_norm', np.nan):.3f} | "
                f"lr={train_metrics.get('lr', np.nan):.2e} | "
                f"time={dt:.1f}s"
            )

            save_ssl_checkpoint(ckpt_last, student, teacher, optimizer, epoch, history)

            nonfinite = not np.isfinite(train_metrics["loss_total"]) or (len(val_ssl_loader) > 0 and not np.isfinite(val_metrics["loss_total"]))
            if CFG.stop_on_nonfinite_loss and nonfinite:
                print("[STOP] Non-finite loss detected. Last checkpoint saved; inspect input data and LR.")
                break

            # Soft warnings for possible collapse/instability.
            if np.isfinite(train_metrics.get("teacher_entropy", np.nan)) and train_metrics["teacher_entropy"] < entropy_floor:
                print(f"[WARN] Low teacher entropy ({train_metrics['teacher_entropy']:.3f} < {entropy_floor:.3f}). Consider lower LR, higher center_momentum, or stronger batch accumulation.")
            if np.isfinite(train_metrics.get("teacher_output_std", np.nan)) and train_metrics["teacher_output_std"] < CFG.collapse_std_floor:
                print(f"[WARN] Very low teacher output std ({train_metrics['teacher_output_std']:.4e}). Possible representation collapse.")

            improved = np.isfinite(monitor_loss) and (monitor_loss < best_monitor - float(CFG.early_stopping_min_delta))
            if improved:
                best_monitor = monitor_loss
                bad_epochs = 0
                save_ssl_checkpoint(ckpt_best, student, teacher, optimizer, epoch, history)
                print(f"  -> saved new best checkpoint: {ckpt_best} (monitor_loss={best_monitor:.4f})")
            else:
                bad_epochs += 1
                print(f"  -> no monitor improvement for {bad_epochs}/{CFG.early_stopping_patience} epochs")

            if int(CFG.early_stopping_patience) > 0 and bad_epochs >= int(CFG.early_stopping_patience):
                print(f"[EARLY STOP] monitor_loss did not improve for {CFG.early_stopping_patience} epochs. Best={best_monitor:.4f}")
                break

        # Always write the final history CSV for diagnostics.
        hist_df = pd.DataFrame(history)
        hist_path = OUT_DIR / "artifacts" / "training_history.csv"
        hist_df.to_csv(hist_path, index=False)
        print("Saved training history:", hist_path)


In [ ]:
# -----------------------------
# Loss curves and training health diagnostics
# -----------------------------
if len(history.get("train_loss_total", [])) > 0:
    df_hist = pd.DataFrame(history)
    hist_csv = OUT_DIR / "artifacts" / "training_history.csv"
    df_hist.to_csv(hist_csv, index=False)
    print("Saved:", hist_csv)

    plt.figure(figsize=(10, 5))
    plt.plot(df_hist["train_loss_total"], label="train total")
    if np.isfinite(df_hist["val_loss_total"]).any():
        plt.plot(df_hist["val_loss_total"], label="val total")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Patient-level SSL loss")
    plt.legend()
    plt.tight_layout()
    p = OUT_DIR / "plots" / "ssl_loss_curve.png"
    plt.savefig(p, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", p)

    plt.figure(figsize=(10, 5))
    if "lr" in df_hist:
        plt.plot(df_hist["lr"], label="learning rate")
    if "grad_norm" in df_hist:
        plt.plot(df_hist["grad_norm"], label="grad norm")
    plt.xlabel("Epoch")
    plt.title("Optimisation diagnostics")
    plt.legend()
    plt.tight_layout()
    p = OUT_DIR / "plots" / "ssl_optimisation_diagnostics.png"
    plt.savefig(p, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", p)

    plt.figure(figsize=(10, 5))
    for col, label in [
        ("train_teacher_entropy", "train teacher entropy"),
        ("val_teacher_entropy", "val teacher entropy"),
        ("train_student_entropy", "train student entropy"),
        ("val_student_entropy", "val student entropy"),
    ]:
        if col in df_hist and np.isfinite(df_hist[col]).any():
            plt.plot(df_hist[col], label=label)
    plt.axhline(CFG.collapse_entropy_floor_fraction * math.log(max(int(CFG.head_out_dim), 2)), linestyle="--", label="low-entropy warning")
    plt.xlabel("Epoch")
    plt.ylabel("Entropy")
    plt.title("DINO output entropy: collapse / instability check")
    plt.legend()
    plt.tight_layout()
    p = OUT_DIR / "plots" / "ssl_entropy_diagnostics.png"
    plt.savefig(p, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", p)

    plt.figure(figsize=(10, 5))
    if "center_norm" in df_hist:
        plt.plot(df_hist["center_norm"], label="DINO centre norm")
    for col, label in [("train_teacher_output_std", "train teacher output std"), ("val_teacher_output_std", "val teacher output std")]:
        if col in df_hist and np.isfinite(df_hist[col]).any():
            plt.plot(df_hist[col], label=label)
    plt.xlabel("Epoch")
    plt.title("DINO centre and output-variance diagnostics")
    plt.legend()
    plt.tight_layout()
    p = OUT_DIR / "plots" / "ssl_center_output_std_diagnostics.png"
    plt.savefig(p, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", p)


In [ ]:
# =============================
# Export embeddings from BOTH
# ssl_best.pt and ssl_last.pt
# =============================

from copy import deepcopy

CKPT_DIR = OUT_DIR / "checkpoints"
BEST_CKPT_PATH = CKPT_DIR / "ssl_best.pt"
LAST_CKPT_PATH = CKPT_DIR / "ssl_last.pt"

print("BEST_CKPT_PATH:", BEST_CKPT_PATH)
print("LAST_CKPT_PATH:", LAST_CKPT_PATH)
print("BEST exists:", BEST_CKPT_PATH.exists())
print("LAST exists:", LAST_CKPT_PATH.exists())

# -----------------------------
# Existing extractor (kept same logic)
# -----------------------------
@torch.no_grad()
def extract_embeddings(model: PatientDinoWrapper, loader: DataLoader, split_name: str):
    model.eval()
    rows = []
    all_embeddings = []

    for batch in loader:
        bag = batch["bag"].to(device, non_blocking=True)
        out = model.forward_bag_features(bag)
        patient_tokens = F.normalize(out["patient_tokens"], dim=-1).cpu().numpy()

        for i in range(patient_tokens.shape[0]):
            row = {
                "split": split_name,
                "patient_id": batch["patient_id"][i],
                "path": batch["path"][i],
                "label": int(batch["label"][i].item()),
                "class_name": batch["class_name"][i],
            }
            rows.append(row)
            all_embeddings.append(patient_tokens[i])

    if len(rows) == 0:
        return pd.DataFrame(), np.zeros((0, embed_dim), dtype=np.float32)

    emb = np.vstack(all_embeddings).astype(np.float32)
    df = pd.DataFrame(rows)
    return df, emb

# -----------------------------
# Helper: load teacher weights from checkpoint
# -----------------------------
def load_teacher_from_ckpt_inplace(model: PatientDinoWrapper, ckpt_path: Path):
    payload = torch.load(str(ckpt_path), map_location=device)
    if "teacher" not in payload:
        raise KeyError(f"'teacher' not found in checkpoint: {ckpt_path}")
    missing, unexpected = model.load_state_dict(payload["teacher"], strict=True)
    print(f"\nLoaded teacher from: {ckpt_path.name}")
    print(f"  epoch: {payload.get('epoch', 'NA')}")
    print(f"  missing keys: {len(missing)}")
    print(f"  unexpected keys: {len(unexpected)}")
    if "history" in payload:
        h = payload["history"]
        if len(h.get("val_loss_total", [])) > 0:
            print(f"  final recorded val_total in payload: {h['val_loss_total'][-1]}")
    return payload

# -----------------------------
# Helper: export one checkpoint
# -----------------------------
def export_embeddings_for_checkpoint(
    teacher_model: PatientDinoWrapper,
    ckpt_path: Path,
    ckpt_tag: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    oac_loader: DataLoader,
):
    if not ckpt_path.exists():
        print(f"[WARN] checkpoint not found: {ckpt_path}")
        return None

    payload = load_teacher_from_ckpt_inplace(teacher_model, ckpt_path)
    teacher_model.eval()

    export_dir = OUT_DIR / "artifacts" / ckpt_tag
    export_dir.mkdir(parents=True, exist_ok=True)

    results = {}

    if len(train_loader):
        df_train_emb, X_train = extract_embeddings(teacher_model, train_loader, "train")
        df_train_emb["checkpoint"] = ckpt_tag
        df_train_emb.to_csv(export_dir / "train_embeddings_metadata.csv", index=False)
        np.save(export_dir / "train_embeddings.npy", X_train)
        results["train"] = (df_train_emb, X_train)
        print(f"[{ckpt_tag}] train embeddings:", X_train.shape)
    else:
        results["train"] = (pd.DataFrame(), np.zeros((0, embed_dim), np.float32))

    if len(val_loader):
        df_val_emb, X_val = extract_embeddings(teacher_model, val_loader, "val")
        df_val_emb["checkpoint"] = ckpt_tag
        df_val_emb.to_csv(export_dir / "val_embeddings_metadata.csv", index=False)
        np.save(export_dir / "val_embeddings.npy", X_val)
        results["val"] = (df_val_emb, X_val)
        print(f"[{ckpt_tag}] val embeddings:", X_val.shape)
    else:
        results["val"] = (pd.DataFrame(), np.zeros((0, embed_dim), np.float32))

    if len(oac_loader):
        df_oac_emb, X_oac = extract_embeddings(teacher_model, oac_loader, "oac")
        df_oac_emb["checkpoint"] = ckpt_tag
        df_oac_emb.to_csv(export_dir / "oac_embeddings_metadata.csv", index=False)
        np.save(export_dir / "oac_embeddings.npy", X_oac)
        results["oac"] = (df_oac_emb, X_oac)
        print(f"[{ckpt_tag}] oac embeddings:", X_oac.shape)
    else:
        results["oac"] = (pd.DataFrame(), np.zeros((0, embed_dim), np.float32))

    # small manifest for traceability
    manifest = {
        "checkpoint_tag": ckpt_tag,
        "checkpoint_path": str(ckpt_path),
        "saved_dir": str(export_dir),
        "epoch": payload.get("epoch", None),
    }
    with open(export_dir / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    return results

# -----------------------------
# IMPORTANT:
# use a copy of current teacher so you do not
# accidentally overwrite your in-memory model
# -----------------------------
teacher_export = deepcopy(teacher).to(device)

exports = {}
exports["best"] = export_embeddings_for_checkpoint(
    teacher_model=teacher_export,
    ckpt_path=BEST_CKPT_PATH,
    ckpt_tag="best",
    train_loader=train_eval_loader,
    val_loader=val_eval_loader,
    oac_loader=oac_eval_loader,
)

exports["last"] = export_embeddings_for_checkpoint(
    teacher_model=teacher_export,
    ckpt_path=LAST_CKPT_PATH,
    ckpt_tag="last",
    train_loader=train_eval_loader,
    val_loader=val_eval_loader,
    oac_loader=oac_eval_loader,
)

print("\nSaved embedding exports to:")
print("  ", OUT_DIR / "artifacts" / "best")
print("  ", OUT_DIR / "artifacts" / "last")

# Optional convenience handles
if exports["best"] is not None:
    df_train_best, X_train_best = exports["best"]["train"]
    df_val_best,   X_val_best   = exports["best"]["val"]
    df_oac_best,   X_oac_best   = exports["best"]["oac"]

if exports["last"] is not None:
    df_train_last, X_train_last = exports["last"]["train"]
    df_val_last,   X_val_last   = exports["last"]["val"]
    df_oac_last,   X_oac_last   = exports["last"]["oac"]

In [ ]:
# =============================
# Select which checkpoint export
# to use downstream
# =============================
USE_CKPT = "best"   # change to "last" to compare

if USE_CKPT == "best":
    df_train_emb, X_train = df_train_best, X_train_best
    df_val_emb,   X_val   = df_val_best,   X_val_best
    df_oac_emb,   X_oac   = df_oac_best,   X_oac_best
elif USE_CKPT == "last":
    df_train_emb, X_train = df_train_last, X_train_last
    df_val_emb,   X_val   = df_val_last,   X_val_last
    df_oac_emb,   X_oac   = df_oac_last,   X_oac_last
else:
    raise ValueError("USE_CKPT must be 'best' or 'last'")

print(f"Using checkpoint export: {USE_CKPT}")
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_oac:", X_oac.shape)

In [ ]:
# ============================================================
# PCA comparison for BOTH checkpoints: best vs last
# ============================================================
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def _safe_concat_embeddings(dfs, Xs):
    keep_df = []
    keep_X = []
    for df, X in zip(dfs, Xs):
        if df is not None and len(df) > 0 and X is not None and len(X) > 0:
            keep_df.append(df.reset_index(drop=True))
            keep_X.append(X)
    if len(keep_df) == 0:
        return pd.DataFrame(), np.zeros((0, 2), dtype=np.float32)
    return pd.concat(keep_df, axis=0, ignore_index=True), np.vstack(keep_X).astype(np.float32)

def make_pca_for_checkpoint(df_train, X_train, df_val, X_val, df_oac, X_oac, tag, out_dir):
    df_all, X_all = _safe_concat_embeddings(
        [df_train, df_val, df_oac],
        [X_train, X_val, X_oac]
    )

    if len(df_all) == 0 or X_all.shape[0] < 2:
        print(f"[{tag}] Not enough embeddings for PCA.")
        return None, None

    pca = PCA(n_components=2, random_state=42)
    XY = pca.fit_transform(X_all)

    df_plot = df_all.copy()
    df_plot["PC1"] = XY[:, 0]
    df_plot["PC2"] = XY[:, 1]

    # save CSV + PCA object arrays
    out_dir.mkdir(parents=True, exist_ok=True)
    df_plot.to_csv(out_dir / f"pca_points_{tag}.csv", index=False)
    np.save(out_dir / f"pca_coords_{tag}.npy", XY)

    fig, ax = plt.subplots(figsize=(8, 6))

    split_styles = {
        "train": dict(marker="o", alpha=0.75, s=45),
        "val":   dict(marker="^", alpha=0.80, s=60),
        "oac":   dict(marker="*", alpha=0.95, s=120),
    }

    # color by class_name
    class_names_unique = list(df_plot["class_name"].astype(str).unique())
    cmap = plt.get_cmap("tab10")
    class_to_color = {c: cmap(i % 10) for i, c in enumerate(class_names_unique)}

    for split in ["train", "val", "oac"]:
        df_split = df_plot[df_plot["split"] == split]
        if len(df_split) == 0:
            continue
        for cname in df_split["class_name"].astype(str).unique():
            sub = df_split[df_split["class_name"].astype(str) == cname]
            ax.scatter(
                sub["PC1"], sub["PC2"],
                c=[class_to_color[cname]],
                label=f"{split}:{cname}",
                **split_styles[split]
            )

    ax.set_title(
        f"PCA of patient embeddings ({tag})\n"
        f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, "
        f"PC2={pca.explained_variance_ratio_[1]:.3f}"
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

    # deduplicate legend
    handles, labels = ax.get_legend_handles_labels()
    uniq = {}
    for h, l in zip(handles, labels):
        if l not in uniq:
            uniq[l] = h
    ax.legend(uniq.values(), uniq.keys(), loc="best", fontsize=8)

    plt.tight_layout()
    fig_path = out_dir / f"pca_plot_{tag}.png"
    plt.savefig(fig_path, dpi=220, bbox_inches="tight")
    plt.show()

    print(f"[{tag}] PCA plot saved to: {fig_path}")
    return df_plot, pca

# output folder
pca_compare_dir = OUT_DIR / "artifacts" / "pca_compare"
pca_compare_dir.mkdir(parents=True, exist_ok=True)

# BEST
if exports.get("best", None) is not None:
    df_train_best, X_train_best = exports["best"]["train"]
    df_val_best,   X_val_best   = exports["best"]["val"]
    df_oac_best,   X_oac_best   = exports["best"]["oac"]

    df_pca_best, pca_best = make_pca_for_checkpoint(
        df_train_best, X_train_best,
        df_val_best,   X_val_best,
        df_oac_best,   X_oac_best,
        tag="best",
        out_dir=pca_compare_dir
    )
else:
    df_pca_best, pca_best = None, None

# LAST
if exports.get("last", None) is not None:
    df_train_last, X_train_last = exports["last"]["train"]
    df_val_last,   X_val_last   = exports["last"]["val"]
    df_oac_last,   X_oac_last   = exports["last"]["oac"]

    df_pca_last, pca_last = make_pca_for_checkpoint(
        df_train_last, X_train_last,
        df_val_last,   X_val_last,
        df_oac_last,   X_oac_last,
        tag="last",
        out_dir=pca_compare_dir
    )
else:
    df_pca_last, pca_last = None, None

# Optional side-by-side plot
if (df_pca_best is not None) and (df_pca_last is not None):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=False, sharey=False)

    for ax, df_plot, title in zip(
        axes,
        [df_pca_best, df_pca_last],
        ["BEST checkpoint", "LAST checkpoint"]
    ):
        class_names_unique = list(df_plot["class_name"].astype(str).unique())
        cmap = plt.get_cmap("tab10")
        class_to_color = {c: cmap(i % 10) for i, c in enumerate(class_names_unique)}

        split_styles = {
            "train": dict(marker="o", alpha=0.75, s=40),
            "val":   dict(marker="^", alpha=0.80, s=55),
            "oac":   dict(marker="*", alpha=0.95, s=110),
        }

        for split in ["train", "val", "oac"]:
            df_split = df_plot[df_plot["split"] == split]
            if len(df_split) == 0:
                continue
            for cname in df_split["class_name"].astype(str).unique():
                sub = df_split[df_split["class_name"].astype(str) == cname]
                ax.scatter(
                    sub["PC1"], sub["PC2"],
                    c=[class_to_color[cname]],
                    label=f"{split}:{cname}",
                    **split_styles[split]
                )

        ax.set_title(title)
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")

    plt.tight_layout()
    cmp_path = pca_compare_dir / "pca_side_by_side_best_vs_last.png"
    plt.savefig(cmp_path, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved side-by-side PCA comparison:", cmp_path)

In [ ]:
# ============================================================
# Attention / focus map utilities for BOTH best and last ckpts
# ============================================================
from copy import deepcopy
CKPT_DIR = OUT_DIR / "checkpoints"
BEST_CKPT_PATH = CKPT_DIR / "ssl_best.pt"
LAST_CKPT_PATH = CKPT_DIR / "ssl_last.pt"

@torch.no_grad()
def get_last_selfattention(backbone: nn.Module, image_4d: torch.Tensor):
    if not hasattr(backbone, "prepare_tokens_with_masks") or not hasattr(backbone, "blocks"):
        raise AttributeError("Backbone does not expose DINOv2-style prepare_tokens_with_masks / blocks.")

    x = backbone.prepare_tokens_with_masks(image_4d)
    for blk in backbone.blocks[:-1]:
        x = blk(x)

    last_blk = backbone.blocks[-1]
    x_norm = last_blk.norm1(x)

    qkv = last_blk.attn.qkv(x_norm)
    B, N, threeC = qkv.shape
    num_heads = last_blk.attn.num_heads
    head_dim = threeC // (3 * num_heads)

    qkv = qkv.reshape(B, N, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    attn = (q * last_blk.attn.scale) @ k.transpose(-2, -1)
    attn = attn.softmax(dim=-1)
    return attn

@torch.no_grad()
def cls_patch_attention_map(backbone: nn.Module, image_4d: torch.Tensor):
    attn = get_last_selfattention(backbone, image_4d)
    num_regs = int(getattr(backbone, "num_register_tokens", 0))
    patch_size = int(getattr(backbone, "patch_size", 14))
    if not isinstance(patch_size, int):
        patch_size = patch_size[0] if isinstance(patch_size, (tuple, list)) else 14

    cls_to_patches = attn[:, :, 0, 1 + num_regs:]  # [B, heads, patches]
    cls_to_patches = cls_to_patches.mean(dim=1)    # [B, patches]

    H = image_4d.shape[-2] // patch_size
    W = image_4d.shape[-1] // patch_size
    heat = cls_to_patches.reshape(image_4d.shape[0], H, W)
    heat = F.interpolate(
        heat.unsqueeze(1),
        size=image_4d.shape[-2:],
        mode="bilinear",
        align_corners=False
    ).squeeze(1)
    return heat

def overlay_heatmap_on_image(image_hwc: np.ndarray, heatmap_hw: np.ndarray, alpha: float = 0.45):
    image = np.asarray(image_hwc, dtype=np.float32)
    heat = np.asarray(heatmap_hw, dtype=np.float32)

    heat = heat - heat.min()
    if heat.max() > 0:
        heat = heat / heat.max()

    cmap = plt.get_cmap("jet")
    heat_rgb = cmap(heat)[..., :3].astype(np.float32)

    overlay = (1 - alpha) * image + alpha * heat_rgb
    overlay = np.clip(overlay, 0.0, 1.0)
    return overlay

def load_single_patient_eval_bag(entry: Dict, cfg: Config):
    vol = load_nifti_volume(Path(entry["path"]))
    vol_u8 = normalize_volume_for_triplets(vol, cfg)
    centers = choose_slice_centers(vol_u8.shape[2], bag_size=cfg.bag_size_eval, random_sample=False)
    triplets = volume_to_triplet_pils(vol_u8, centers)
    bag = torch.stack([EVAL_TRANSFORM(img) for img in triplets], dim=0)
    return triplets, centers, bag

@torch.no_grad()
def visualize_oac_patient_attention(model: PatientDinoWrapper, entry: Dict, cfg: Config, save_dir: Path, ckpt_tag: str):
    save_dir.mkdir(parents=True, exist_ok=True)

    triplets, centers, bag = load_single_patient_eval_bag(entry, cfg)
    bag = bag.unsqueeze(0).to(device)  # [1, S, 3, H, W]

    out = model.forward_bag_features(bag)
    slice_weights = out["slice_weights"][0].detach().cpu().numpy()

    ranked = np.argsort(slice_weights)[::-1]
    top_idx = ranked[: min(cfg.n_attention_slices_to_plot, len(ranked))]

    n_rows = len(top_idx)
    fig, axes = plt.subplots(n_rows, 3, figsize=(12, 4 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_i, idx in enumerate(top_idx):
        img_tensor = bag[0, idx:idx+1]
        heat = cls_patch_attention_map(model.backbone, img_tensor)[0].detach().cpu().numpy()
        img_rgb = denormalize_tensor_image(img_tensor[0])
        overlay = overlay_heatmap_on_image(img_rgb, heat)

        axes[row_i, 0].imshow(img_rgb)
        axes[row_i, 0].set_title(f"Triplet z={centers[idx]} | slice_attn={slice_weights[idx]:.3f}")
        axes[row_i, 0].axis("off")

        axes[row_i, 1].imshow(heat, cmap="jet")
        axes[row_i, 1].set_title("CLS-to-patch attention")
        axes[row_i, 1].axis("off")

        axes[row_i, 2].imshow(overlay)
        axes[row_i, 2].set_title("Overlay")
        axes[row_i, 2].axis("off")

    plt.suptitle(f"OAC attention focus: {entry['patient_id']} [{ckpt_tag}]", y=1.02)
    plt.tight_layout()
    out_path = save_dir / f"{entry['patient_id']}_attention_{ckpt_tag}.png"
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()
    return out_path

# -----------------------------
# checkpoint load helper
# -----------------------------
def load_teacher_weights_for_visualization(base_teacher_model: PatientDinoWrapper, ckpt_path: Path):
    model = deepcopy(base_teacher_model).to(device)
    payload = torch.load(str(ckpt_path), map_location=device)
    if "teacher" not in payload:
        raise KeyError(f"'teacher' not found in checkpoint: {ckpt_path}")
    missing, unexpected = model.load_state_dict(payload["teacher"], strict=True)
    print(f"Loaded teacher for visualization from: {ckpt_path.name}")
    print(f"  missing keys: {len(missing)} | unexpected keys: {len(unexpected)}")
    model.eval()
    return model

# -----------------------------
# attention export for BOTH ckpts
# -----------------------------
attention_root = OUT_DIR / "artifacts" / "attention_maps_compare"
attention_root.mkdir(parents=True, exist_ok=True)

ckpt_map = {
    "best": BEST_CKPT_PATH,
    "last": LAST_CKPT_PATH,
}

for ckpt_tag, ckpt_path in ckpt_map.items():
    if not ckpt_path.exists():
        print(f"[WARN] missing checkpoint for attention: {ckpt_path}")
        continue

    ckpt_attention_dir = attention_root / ckpt_tag
    ckpt_attention_dir.mkdir(parents=True, exist_ok=True)

    try:
        teacher_vis = load_teacher_weights_for_visualization(teacher, ckpt_path)
    except Exception as e:
        print(f"[{ckpt_tag}] Could not load teacher checkpoint for visualization: {e}")
        continue

    if len(oac_entries) > 0:
        for entry in oac_entries[: min(3, len(oac_entries))]:
            try:
                out_path = visualize_oac_patient_attention(
                    teacher_vis,
                    entry,
                    CFG,
                    ckpt_attention_dir,
                    ckpt_tag=ckpt_tag
                )
                print(f"[{ckpt_tag}] Saved:", out_path)
            except Exception as e:
                print(f"[{ckpt_tag}] Attention visualization failed for {entry['patient_id']}: {e}")
    else:
        print("No OAC patients found.")

## Branch D/E: DINO-JEPA latent-prediction branches

These branches keep the same DINOv2 backbone but replace DINO-style augmented-view matching with JEPA-style latent prediction.

- `dino_jepa_slice`: masks block regions in PET slice-triplets and predicts the unmasked target latent representation.
- `dino_jepa_cross_slice`: uses neighbouring slice context to predict a held-out target-slice latent representation.

The default is conservative: frozen DINOv2, trainable patient pooling and predictor only. This makes it a safer embedding-capture branch for small medical cohorts.


In [ ]:

# ============================================================
# Branch D/E: DINO-JEPA-style latent-prediction training
# ============================================================

def _cosine_latent_loss(pred: torch.Tensor, target: torch.Tensor):
    pred = F.normalize(pred.float(), dim=-1)
    target = F.normalize(target.detach().float(), dim=-1)
    return 1.0 - (pred * target).sum(dim=-1).mean()

def _latent_prediction_loss(pred: torch.Tensor, target: torch.Tensor):
    if str(CFG.jepa_loss).lower() == "smooth_l1":
        return F.smooth_l1_loss(pred.float(), target.detach().float())
    return _cosine_latent_loss(pred, target)

def mask_random_blocks_5d(bag_5d: torch.Tensor, num_blocks: int, scale_range: Tuple[float, float], fill_value: float = 0.0):
    """Mask random spatial blocks in [B,S,C,H,W] tensor; same block is applied across channels."""
    x = bag_5d.clone()
    B, S, C, H, W = x.shape
    min_scale, max_scale = float(scale_range[0]), float(scale_range[1])
    for b in range(B):
        for s in range(S):
            for _ in range(max(1, int(num_blocks))):
                area = random.uniform(min_scale, max_scale) * H * W
                aspect = random.uniform(0.75, 1.33)
                h = int(round(math.sqrt(area / aspect)))
                w = int(round(math.sqrt(area * aspect)))
                h = max(1, min(H, h))
                w = max(1, min(W, w))
                y0 = random.randint(0, max(0, H - h))
                x0 = random.randint(0, max(0, W - w))
                x[b, s, :, y0:y0+h, x0:x0+w] = float(fill_value)
    return x

class PatientJEPAModel(nn.Module):
    def __init__(self, context_backbone: nn.Module, target_backbone: nn.Module, embed_dim: int, cfg: Config, mode: str):
        super().__init__()
        self.context_backbone = context_backbone
        self.target_backbone = target_backbone
        self.embed_dim = int(embed_dim)
        self.cfg = cfg
        self.mode = str(mode)
        self.pool = PatientAttentionPool(embed_dim)
        self.predictor = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, int(cfg.jepa_predictor_hidden_dim)),
            nn.GELU(),
            nn.Dropout(float(cfg.jepa_dropout)),
            nn.Linear(int(cfg.jepa_predictor_hidden_dim), embed_dim),
        )
        set_backbone_trainability(self.context_backbone, cfg.jepa_freeze_backbone, cfg.jepa_unfreeze_last_n_blocks)
        for p in self.target_backbone.parameters():
            p.requires_grad = False

    def context_backbone_is_trainable(self) -> bool:
        return any(p.requires_grad for p in self.context_backbone.parameters())

    def _encode_bag_cls(self, backbone: nn.Module, bag_5d: torch.Tensor, requires_grad: bool):
        B, S, C, H, W = bag_5d.shape
        x = bag_5d.reshape(B * S, C, H, W)
        chunks = []
        chunk_size = None if requires_grad else max(1, int(self.cfg.frozen_backbone_chunk_size))
        if chunk_size is None:
            feats = backbone.forward_features(x)
            cls = feats["x_norm_clstoken"]
        else:
            ctx = torch.enable_grad if requires_grad else torch.no_grad
            with ctx():
                for xs in x.split(chunk_size, dim=0):
                    feats = backbone.forward_features(xs)
                    chunks.append(feats["x_norm_clstoken"])
                cls = torch.cat(chunks, dim=0)
        return cls.reshape(B, S, -1)

    def forward_bag_features(self, bag_5d: torch.Tensor):
        trainable = self.context_backbone_is_trainable()
        cls_tokens = self._encode_bag_cls(self.context_backbone, bag_5d, requires_grad=trainable)
        patient_tokens, slice_weights = self.pool(cls_tokens)
        return {
            "patient_tokens": patient_tokens,
            "slice_tokens": cls_tokens,
            "slice_weights": slice_weights,
        }

    def forward_jepa_slice(self, bag_5d: torch.Tensor):
        masked = mask_random_blocks_5d(
            bag_5d,
            num_blocks=CFG.jepa_num_mask_blocks,
            scale_range=CFG.jepa_slice_mask_scale,
            fill_value=CFG.jepa_fill_value,
        )
        ctx_cls = self._encode_bag_cls(self.context_backbone, masked, requires_grad=self.context_backbone_is_trainable())
        with torch.no_grad():
            tgt_cls = self._encode_bag_cls(self.target_backbone, bag_5d, requires_grad=False)
        # Per-slice latent prediction from masked version of the same slice-triplet.
        pred = self.predictor(ctx_cls)
        loss_pred = _latent_prediction_loss(pred, tgt_cls)
        # Optional patient consistency term between masked-context patient and target patient.
        ctx_patient, ctx_weights = self.pool(ctx_cls)
        with torch.no_grad():
            tgt_patient, _ = self.pool(tgt_cls)
        loss_patient = _cosine_latent_loss(ctx_patient, tgt_patient)
        total = loss_pred + float(CFG.jepa_patient_consistency_weight) * loss_patient
        return {
            "loss_total": total,
            "loss_pred": loss_pred.detach(),
            "loss_patient": loss_patient.detach(),
            "patient_tokens": ctx_patient,
            "slice_weights": ctx_weights,
        }

    def forward_jepa_cross_slice(self, bag_5d: torch.Tensor):
        B, S, C, H, W = bag_5d.shape
        if S < 2:
            return self.forward_jepa_slice(bag_5d)
        if self.training:
            target_idx = random.randint(0, S - 1)
        else:
            target_idx = S // 2
        context_indices = [i for i in range(S) if i != target_idx]
        context_bag = bag_5d[:, context_indices]
        target_bag = bag_5d[:, target_idx:target_idx+1]
        ctx_cls = self._encode_bag_cls(self.context_backbone, context_bag, requires_grad=self.context_backbone_is_trainable())
        ctx_patient, ctx_weights = self.pool(ctx_cls)
        with torch.no_grad():
            tgt_cls = self._encode_bag_cls(self.target_backbone, target_bag, requires_grad=False)[:, 0, :]
        pred = self.predictor(ctx_patient)
        loss_pred = _latent_prediction_loss(pred, tgt_cls)
        return {
            "loss_total": loss_pred,
            "loss_pred": loss_pred.detach(),
            "loss_patient": torch.tensor(0.0, device=bag_5d.device),
            "patient_tokens": ctx_patient,
            "slice_weights": ctx_weights,
        }

    def forward(self, bag_5d: torch.Tensor):
        if self.mode == "cross_slice":
            return self.forward_jepa_cross_slice(bag_5d)
        return self.forward_jepa_slice(bag_5d)

@torch.no_grad()
def update_jepa_target_encoder(model: PatientJEPAModel, momentum: float):
    for p_ctx, p_tgt in zip(model.context_backbone.parameters(), model.target_backbone.parameters()):
        p_tgt.data.mul_(float(momentum)).add_((1.0 - float(momentum)) * p_ctx.data)

def build_jepa_model(cfg: Config, mode: str):
    context_backbone, local_embed_dim = load_backbone(cfg)
    target_backbone, _ = load_backbone(cfg)
    if local_embed_dim is None:
        local_embed_dim = getattr(context_backbone, "embed_dim", None)
    if local_embed_dim is None:
        raise ValueError("Could not infer embed_dim for JEPA model.")
    model = PatientJEPAModel(context_backbone, target_backbone, int(local_embed_dim), cfg, mode=mode)
    # initialise target from context exactly before EMA updates
    model.target_backbone.load_state_dict(model.context_backbone.state_dict(), strict=True)
    return model.to(device), int(local_embed_dim)

def run_jepa_epoch(model: PatientJEPAModel, loader: DataLoader, optimizer, train: bool, branch_name: str):
    model.train(train)
    losses_total, losses_pred, losses_patient = [], [], []
    if train:
        optimizer.zero_grad(set_to_none=True)
    iterator = _make_progress_bar(loader, desc=f"{branch_name} train" if train else f"{branch_name} val") if '_make_progress_bar' in globals() else loader
    accum_steps = max(1, int(CFG.jepa_grad_accum_steps))
    for it, batch in enumerate(iterator):
        bag = _bag_from_supervised_batch(batch, train=train).to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            with _branch_autocast_context():
                out = model(bag)
                loss = out["loss_total"]
            if train:
                (loss / accum_steps).backward()
                if ((it + 1) % accum_steps == 0) or (it == len(loader) - 1):
                    if CFG.grad_clip_norm is not None and CFG.grad_clip_norm > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    update_jepa_target_encoder(model, CFG.jepa_ema_momentum)
        losses_total.append(float(loss.detach().cpu()))
        losses_pred.append(float(out["loss_pred"].detach().cpu()))
        losses_patient.append(float(out["loss_patient"].detach().cpu()))
        if CFG.show_batch_progress and tqdm is not None and hasattr(iterator, "set_postfix"):
            if (it % max(int(CFG.progress_update_every), 1) == 0) or (it == len(loader) - 1):
                iterator.set_postfix({"loss": f"{np.mean(losses_total):.4f}"})
    return {
        "loss_total": float(np.mean(losses_total)) if losses_total else np.nan,
        "loss_pred": float(np.mean(losses_pred)) if losses_pred else np.nan,
        "loss_patient": float(np.mean(losses_patient)) if losses_patient else np.nan,
    }

def run_jepa_branch(branch_name: str, mode: str):
    print(f"\n=== Branch {'D' if mode == 'slice' else 'E'}: {branch_name} ===")
    if len(train_ssl_loader) == 0:
        print(f"[WARN] No training data found. Skipping {branch_name}.")
        return None, None
    model, local_embed_dim = build_jepa_model(CFG, mode=mode)
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"{branch_name} trainable tensors: {len(params)}")
    optimizer = torch.optim.AdamW(params, lr=CFG.jepa_lr, weight_decay=CFG.jepa_weight_decay)
    history = []
    best_monitor = float("inf")
    bad_epochs = 0
    ckpt_best = OUT_DIR / "checkpoints" / f"{branch_name}_best.pt"
    ckpt_last = OUT_DIR / "checkpoints" / f"{branch_name}_last.pt"
    for epoch in range(int(CFG.jepa_epochs)):
        progress = epoch / max(int(CFG.jepa_epochs) - 1, 1)
        lr_now = CFG.jepa_min_lr + 0.5 * (CFG.jepa_lr - CFG.jepa_min_lr) * (1 + math.cos(math.pi * progress))
        for g in optimizer.param_groups:
            g["lr"] = float(lr_now)
        train_m = run_jepa_epoch(model, train_ssl_loader, optimizer, train=True, branch_name=branch_name)
        val_m = run_jepa_epoch(model, val_ssl_loader if len(val_ssl_loader) else train_ssl_loader, optimizer, train=False, branch_name=branch_name)
        row = {"epoch": epoch + 1, "lr": lr_now}
        row.update({f"train_{k}": v for k, v in train_m.items()})
        row.update({f"val_{k}": v for k, v in val_m.items()})
        history.append(row)
        monitor = val_m["loss_total"] if np.isfinite(val_m["loss_total"]) else train_m["loss_total"]
        print(
            f"[{branch_name}] epoch {epoch+1:03d}/{CFG.jepa_epochs} | "
            f"train={train_m['loss_total']:.4f} | val={val_m['loss_total']:.4f} | "
            f"pred={train_m['loss_pred']:.4f} | patient={train_m['loss_patient']:.4f} | lr={lr_now:.2e}"
        )
        torch.save({"model": model.state_dict(), "epoch": epoch, "history": history, "cfg": asdict(CFG), "mode": mode}, str(ckpt_last))
        if np.isfinite(monitor) and monitor < best_monitor - float(CFG.early_stopping_min_delta):
            best_monitor = float(monitor)
            bad_epochs = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "history": history, "cfg": asdict(CFG), "mode": mode}, str(ckpt_best))
            print(f"  -> saved new JEPA best: {ckpt_best} (monitor={best_monitor:.4f})")
        else:
            bad_epochs += 1
            print(f"  -> no JEPA improvement for {bad_epochs}/{CFG.early_stopping_patience} epochs")
        if int(CFG.early_stopping_patience) > 0 and bad_epochs >= int(CFG.early_stopping_patience):
            print(f"[{branch_name}] early stopping triggered.")
            break
    branch_dir = BRANCH_ROOT / branch_name
    branch_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(branch_dir / "jepa_training_history.csv", index=False)
    if ckpt_best.exists():
        payload = torch.load(str(ckpt_best), map_location=device)
        model.load_state_dict(payload["model"], strict=True)
    exports = export_branch_embeddings(model, branch_name=branch_name, embed_dim_fallback=local_embed_dim)
    return model, exports

jepa_slice_model = None
jepa_cross_slice_model = None
if branch_enabled("dino_jepa_slice"):
    jepa_slice_model, jepa_slice_exports = run_jepa_branch("dino_jepa_slice", mode="slice")
    if jepa_slice_model is not None:
        del jepa_slice_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Skipping Branch D: dino_jepa_slice")

if branch_enabled("dino_jepa_cross_slice"):
    jepa_cross_slice_model, jepa_cross_slice_exports = run_jepa_branch("dino_jepa_cross_slice", mode="cross_slice")
    if jepa_cross_slice_model is not None:
        del jepa_cross_slice_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Skipping Branch E: dino_jepa_cross_slice")


In [ ]:
# # ============================================================
# # Standalone export of embeddings from fresh JEPA checkpoints
# #
# # Exports:
# #   dino_jepa_slice_best.pt
# #   dino_jepa_slice_last.pt
# #   dino_jepa_cross_slice_best.pt
# #   dino_jepa_cross_slice_last.pt
# #
# # Main branch-root outputs are promoted from "best" by default:
# #
# #   patient_dinov2_ssl_outputs/artifacts/branches/dino_jepa_slice/
# #       train_embeddings.npy
# #       val_embeddings.npy
# #       oac_embeddings.npy
# #       train_predictions.csv
# #       val_predictions.csv
# #       oac_predictions.csv
# #
# # Also keeps checkpoint-specific exports:
# #
# #   .../dino_jepa_slice/best/
# #   .../dino_jepa_slice/last/
# #   .../dino_jepa_cross_slice/best/
# #   .../dino_jepa_cross_slice/last/
# # ============================================================

# from pathlib import Path
# import json
# import shutil
# import inspect
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn.functional as F

# # -----------------------------
# # Basic paths
# # -----------------------------

# # Edit this if your run root differs.
# ROOT = Path("/mnt/d/data/patient_dinov2_ssl_outputs_branches_6")

# RUN_ROOT = ROOT
# CKPT_DIR = RUN_ROOT / "checkpoints"
# ARTIFACT_DIR = RUN_ROOT / "artifacts"
# BRANCH_ROOT = ARTIFACT_DIR / "branches"

# ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
# BRANCH_ROOT.mkdir(parents=True, exist_ok=True)

# device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# JEPA_BRANCHES = {
#     "dino_jepa_slice": {
#         "mode": "slice",
#         "best_ckpt": CKPT_DIR / "dino_jepa_slice_best.pt",
#         "last_ckpt": CKPT_DIR / "dino_jepa_slice_last.pt",
#     },
#     "dino_jepa_cross_slice": {
#         "mode": "cross_slice",
#         "best_ckpt": CKPT_DIR / "dino_jepa_cross_slice_best.pt",
#         "last_ckpt": CKPT_DIR / "dino_jepa_cross_slice_last.pt",
#     },
# }

# EXPORT_TAGS = ("best", "last")

# # This controls which checkpoint is copied into the branch root for downstream PCA cells.
# # Usually use "best".
# PROMOTE_TAG_TO_BRANCH_ROOT = "best"

# MAKE_LINEAR_PROBE_PREDICTIONS = True

# print("RUN_ROOT:", RUN_ROOT)
# print("CKPT_DIR:", CKPT_DIR)
# print("BRANCH_ROOT:", BRANCH_ROOT)
# print("device:", device)

# for branch, info in JEPA_BRANCHES.items():
#     print(f"\n{branch}")
#     print("  best:", info["best_ckpt"], info["best_ckpt"].exists())
#     print("  last:", info["last_ckpt"], info["last_ckpt"].exists())


# # -----------------------------
# # Loader discovery
# # -----------------------------

# def _find_loader(name_options, required=True):
#     for name in name_options:
#         obj = globals().get(name, None)
#         if obj is not None and hasattr(obj, "__iter__"):
#             try:
#                 n = len(obj)
#             except Exception:
#                 n = "?"
#             print(f"Using loader {name}: len={n}")
#             return obj, name

#     if required:
#         print("\nAvailable loader-like objects:")
#         for k, v in globals().items():
#             if hasattr(v, "__iter__") and ("loader" in k.lower() or "dl" in k.lower()):
#                 try:
#                     n = len(v)
#                 except Exception:
#                     n = "?"
#                 print(f"  {k}: len={n}, type={type(v)}")

#         raise RuntimeError(f"Could not find required loader from: {name_options}")

#     return None, None


# train_export_loader, train_loader_name = _find_loader(
#     [
#         "train_eval_loader",
#         "train_export_loader",
#         "train_loader_export",
#         "train_loader",
#         "ssl_train_loader",
#         "jepa_train_loader",
#         "train_dl",
#     ],
#     required=True,
# )

# val_export_loader, val_loader_name = _find_loader(
#     [
#         "val_eval_loader",
#         "valid_eval_loader",
#         "val_export_loader",
#         "val_loader_export",
#         "val_loader",
#         "valid_loader",
#         "ssl_val_loader",
#         "jepa_val_loader",
#         "val_dl",
#     ],
#     required=True,
# )

# oac_export_loader, oac_loader_name = _find_loader(
#     [
#         "oac_eval_loader",
#         "oac_export_loader",
#         "oac_loader_export",
#         "oac_loader",
#         "OAC_loader",
#         "external_oac_loader",
#         "external_loader",
#         "oac_dl",
#     ],
#     required=False,
# )


# # -----------------------------
# # Fresh JEPA model construction
# # -----------------------------

# def _infer_embed_dim(backbone, fallback=None):
#     if fallback is not None:
#         return int(fallback)

#     for attr in ["embed_dim", "num_features", "hidden_dim", "hidden_size"]:
#         if hasattr(backbone, attr):
#             val = getattr(backbone, attr)
#             if val is not None:
#                 return int(val)

#     raise ValueError("Could not infer backbone embedding dimension.")


# def build_fresh_jepa_model(mode):
#     """
#     Rebuilds a fresh PatientJEPAModel for the requested JEPA mode.

#     Preferred path:
#         build_jepa_model(CFG, mode)

#     This matches the training code and avoids needing an in-memory
#     jepa_slice_model or jepa_cross_slice_model.
#     """
#     if "CFG" not in globals():
#         raise RuntimeError("CFG is not defined. Run the config cell first.")

#     if "build_jepa_model" in globals():
#         model, local_embed_dim = build_jepa_model(CFG, mode=mode)
#         model = model.to(device)
#         model.eval()
#         return model, int(local_embed_dim)

#     # Fallback if build_jepa_model is not present.
#     required = ["load_backbone", "PatientJEPAModel"]
#     missing = [x for x in required if x not in globals()]
#     if missing:
#         raise RuntimeError(
#             f"Missing required definitions for fresh JEPA model: {missing}. "
#             "Run the JEPA model-definition cell first."
#         )

#     context_backbone, local_embed_dim = load_backbone(CFG)
#     target_backbone, _ = load_backbone(CFG)

#     if local_embed_dim is None:
#         local_embed_dim = _infer_embed_dim(context_backbone)

#     model = PatientJEPAModel(
#         context_backbone=context_backbone,
#         target_backbone=target_backbone,
#         embed_dim=int(local_embed_dim),
#         cfg=CFG,
#         mode=mode,
#     )

#     model.target_backbone.load_state_dict(
#         model.context_backbone.state_dict(),
#         strict=True,
#     )

#     model = model.to(device)
#     model.eval()

#     return model, int(local_embed_dim)


# # -----------------------------
# # Checkpoint loading
# # -----------------------------

# def _strip_prefix(state, prefix):
#     if not isinstance(state, dict):
#         return state

#     if all(k.startswith(prefix) for k in state.keys()):
#         return {k[len(prefix):]: v for k, v in state.items()}

#     return state


# def _get_jepa_state_from_payload(payload, ckpt_path):
#     """
#     JEPA training saves:
#         {"model": model.state_dict(), "epoch": ..., "history": ..., "mode": ...}

#     This also supports other common keys.
#     """
#     if not isinstance(payload, dict):
#         raise RuntimeError(f"Checkpoint is not a dict: {ckpt_path}")

#     candidate_keys = [
#         "model",
#         "model_state_dict",
#         "state_dict",
#         "jepa_model",
#         "context_encoder_state_dict",
#     ]

#     for key in candidate_keys:
#         if key in payload and isinstance(payload[key], dict):
#             return payload[key], key

#     if all(torch.is_tensor(v) for v in payload.values()):
#         return payload, "payload_is_state_dict"

#     raise KeyError(
#         f"Could not find JEPA model state dict in checkpoint: {ckpt_path}. "
#         f"Available keys: {list(payload.keys())[:30]}"
#     )


# def load_jepa_checkpoint_inplace(model, ckpt_path, expected_mode=None, strict_first=True):
#     ckpt_path = Path(ckpt_path)

#     if not ckpt_path.exists():
#         raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

#     payload = torch.load(str(ckpt_path), map_location=device)

#     if expected_mode is not None:
#         ckpt_mode = payload.get("mode", None)
#         if ckpt_mode is not None and str(ckpt_mode) != str(expected_mode):
#             print(
#                 f"[warning] checkpoint mode={ckpt_mode}, "
#                 f"but expected mode={expected_mode}"
#             )

#     state, state_key = _get_jepa_state_from_payload(payload, ckpt_path)

#     state = _strip_prefix(state, "module.")
#     state = _strip_prefix(state, "model.")

#     try:
#         missing, unexpected = model.load_state_dict(state, strict=strict_first)
#     except Exception as e:
#         print(f"[warning] strict={strict_first} JEPA load failed: {e}")
#         print("Retrying with strict=False.")
#         missing, unexpected = model.load_state_dict(state, strict=False)

#     print(f"\nLoaded JEPA checkpoint: {ckpt_path.name}")
#     print(f"  state key      : {state_key}")
#     print(f"  epoch          : {payload.get('epoch', 'NA')}")
#     print(f"  mode           : {payload.get('mode', 'NA')}")
#     print(f"  missing keys   : {len(missing)}")
#     print(f"  unexpected keys: {len(unexpected)}")

#     if len(missing) > 0:
#         print("  first missing keys:", missing[:5])
#     if len(unexpected) > 0:
#         print("  first unexpected keys:", unexpected[:5])

#     if "history" in payload:
#         h = payload["history"]
#         if isinstance(h, list) and len(h) > 0:
#             print("  history last row:", h[-1])
#         elif isinstance(h, dict):
#             print("  history keys:", list(h.keys())[:10])

#     model.eval()
#     return payload


# # -----------------------------
# # Batch parsing
# # -----------------------------

# def _to_numpy_string_list(x, n=None, default_prefix="idx"):
#     if x is None:
#         if n is None:
#             return np.array([], dtype=str)
#         return np.array([f"{default_prefix}_{i}" for i in range(n)], dtype=str)

#     if torch.is_tensor(x):
#         x = x.detach().cpu().numpy()

#     arr = np.asarray(x)

#     if arr.ndim == 0:
#         arr = arr.reshape(1)

#     arr = pd.Series(arr.reshape(-1)).astype(str).str.strip().values

#     if n is not None and len(arr) != n:
#         return np.array([f"{default_prefix}_{i}" for i in range(n)], dtype=str)

#     return arr


# def _extract_bag_from_batch(batch):
#     if isinstance(batch, dict):
#         for key in ["bag", "image", "images", "x", "input", "pet", "volume"]:
#             if key in batch and torch.is_tensor(batch[key]):
#                 return batch[key]

#     if torch.is_tensor(batch):
#         return batch

#     if isinstance(batch, (tuple, list)):
#         for item in batch:
#             if torch.is_tensor(item):
#                 return item
#             if isinstance(item, dict):
#                 out = _extract_bag_from_batch(item)
#                 if out is not None:
#                     return out

#     return None


# def _extract_metadata_from_batch(batch, batch_size, split_name, start_idx):
#     if not isinstance(batch, dict):
#         return {
#             "patient_id": np.array([f"{split_name}_idx_{start_idx+i}" for i in range(batch_size)]),
#             "path": np.array([""] * batch_size),
#             "label": np.array([-1] * batch_size),
#             "class_name": np.array(["Unknown"] * batch_size),
#         }

#     patient_id = None
#     for key in ["patient_id", "patient_ids", "Patient_ID", "PatientID", "id", "ids"]:
#         if key in batch:
#             patient_id = _to_numpy_string_list(batch[key], batch_size, f"{split_name}_idx")
#             break

#     if patient_id is None:
#         patient_id = np.array([f"{split_name}_idx_{start_idx+i}" for i in range(batch_size)])

#     path = None
#     for key in ["path", "paths", "filepath", "file_path"]:
#         if key in batch:
#             path = _to_numpy_string_list(batch[key], batch_size, "")
#             break

#     if path is None:
#         path = np.array([""] * batch_size)

#     label = None
#     for key in ["label", "labels", "target", "targets", "y"]:
#         if key in batch:
#             val = batch[key]
#             if torch.is_tensor(val):
#                 val = val.detach().cpu().numpy()
#             label = np.asarray(val).reshape(-1)
#             if len(label) != batch_size:
#                 label = None
#             break

#     if label is None:
#         label = np.array([-1] * batch_size)

#     class_name = None
#     for key in ["class_name", "class_names", "class", "classes"]:
#         if key in batch:
#             class_name = _to_numpy_string_list(batch[key], batch_size, "Unknown")
#             break

#     if class_name is None:
#         class_name = np.array([str(x) for x in label])

#     return {
#         "patient_id": patient_id,
#         "path": path,
#         "label": label,
#         "class_name": class_name,
#     }


# # -----------------------------
# # JEPA embedding extraction
# # -----------------------------

# def _infer_patient_tokens_from_jepa_output(out):
#     if isinstance(out, dict):
#         for key in [
#             "patient_tokens",
#             "patient_embedding",
#             "patient_embeddings",
#             "embedding",
#             "embeddings",
#             "features",
#             "z",
#             "repr",
#             "context_embedding",
#         ]:
#             if key in out and torch.is_tensor(out[key]):
#                 return out[key]

#         # fallback: first 2D tensor
#         for v in out.values():
#             if torch.is_tensor(v) and v.ndim >= 2:
#                 return v

#     elif torch.is_tensor(out):
#         return out

#     return None


# @torch.no_grad()
# def extract_jepa_embeddings_standalone(model, loader, split_name, embed_dim_fallback=None):
#     model.eval()

#     rows = []
#     all_embeddings = []
#     start_idx = 0

#     for batch_idx, batch in enumerate(loader):
#         bag = _extract_bag_from_batch(batch)

#         if bag is None:
#             raise RuntimeError(f"No bag/image tensor found in {split_name} batch {batch_idx}")

#         bag = bag.to(device, non_blocking=True)

#         if hasattr(model, "forward_bag_features"):
#             out = model.forward_bag_features(bag)
#         else:
#             out = model(bag)

#         patient_tokens = _infer_patient_tokens_from_jepa_output(out)

#         if patient_tokens is None:
#             raise RuntimeError(
#                 f"Could not infer patient tokens from JEPA model output for "
#                 f"{split_name} batch {batch_idx}."
#             )

#         if patient_tokens.ndim == 3:
#             patient_tokens = patient_tokens.mean(dim=1)

#         if patient_tokens.ndim > 3:
#             patient_tokens = patient_tokens.reshape(patient_tokens.shape[0], -1)

#         patient_tokens = F.normalize(patient_tokens, dim=-1)
#         emb_np = patient_tokens.detach().float().cpu().numpy().astype(np.float32)

#         batch_size = emb_np.shape[0]
#         meta = _extract_metadata_from_batch(batch, batch_size, split_name, start_idx)

#         for i in range(batch_size):
#             try:
#                 label_val = int(meta["label"][i])
#             except Exception:
#                 label_val = meta["label"][i]

#             rows.append({
#                 "split": split_name,
#                 "patient_id": str(meta["patient_id"][i]),
#                 "path": str(meta["path"][i]),
#                 "label": label_val,
#                 "class_name": str(meta["class_name"][i]),
#             })

#         all_embeddings.append(emb_np)
#         start_idx += batch_size

#     if len(rows) == 0:
#         dim = int(embed_dim_fallback) if embed_dim_fallback is not None else 0
#         return pd.DataFrame(), np.zeros((0, dim), dtype=np.float32)

#     emb = np.vstack(all_embeddings).astype(np.float32)
#     df = pd.DataFrame(rows)

#     return df, emb


# # -----------------------------
# # Optional linear-probe prediction CSVs
# # -----------------------------

# def save_linear_probe_predictions(export_dir, split_results):
#     try:
#         from sklearn.pipeline import make_pipeline
#         from sklearn.preprocessing import StandardScaler, LabelEncoder
#         from sklearn.linear_model import LogisticRegression
#     except Exception:
#         print("[info] sklearn not available; skipping prediction CSVs.")
#         return

#     if "train" not in split_results:
#         print("[info] no train split available; skipping prediction CSVs.")
#         return

#     df_train, X_train = split_results["train"]

#     if len(df_train) == 0:
#         print("[info] empty train split; skipping prediction CSVs.")
#         return

#     y_raw = df_train["class_name"].astype(str).values
#     valid = ~pd.Series(y_raw).isin(["Unknown", "Unlabelled", "None", "nan", "NaN", "", "-1"]).values

#     if valid.sum() < 4 or len(np.unique(y_raw[valid])) < 2:
#         print("[info] insufficient labelled train samples; skipping prediction CSVs.")
#         return

#     le = LabelEncoder()
#     y_train = le.fit_transform(y_raw[valid])

#     clf = make_pipeline(
#         StandardScaler(),
#         LogisticRegression(
#             max_iter=2000,
#             class_weight="balanced",
#             solver="lbfgs",
#         )
#     )

#     clf.fit(X_train[valid], y_train)

#     class_names = le.classes_.astype(str)
#     np.save(export_dir / "class_names.npy", class_names)

#     print(f"[linear probe] fitted on {valid.sum()} train samples")
#     print(f"[linear probe] classes: {list(class_names)}")

#     for split, (df, X) in split_results.items():
#         if len(df) == 0:
#             continue

#         pred_idx = clf.predict(X)
#         pred_names = class_names[pred_idx]

#         try:
#             probs = clf.predict_proba(X)
#         except Exception:
#             probs = None

#         pred_df = df.copy()
#         pred_df["pred_label"] = pred_idx
#         pred_df["pred_class_name"] = pred_names

#         if probs is not None:
#             for i, cname in enumerate(class_names):
#                 pred_df[f"prob_class_{i}"] = probs[:, i]

#         out_csv = export_dir / f"{split}_predictions.csv"
#         pred_df.to_csv(out_csv, index=False)
#         print(f"[saved] {out_csv}")


# # -----------------------------
# # Save split files
# # -----------------------------

# def _save_split_export(export_dir, split, df, X, ckpt_tag):
#     df = df.copy()
#     df["checkpoint"] = ckpt_tag

#     df.to_csv(export_dir / f"{split}_embeddings_metadata.csv", index=False)

#     np.save(export_dir / f"{split}_embeddings.npy", X)
#     np.save(export_dir / f"{split}_patient_ids.npy", df["patient_id"].astype(str).values)
#     np.save(export_dir / f"{split}_labels.npy", df["class_name"].astype(str).values)

#     print(f"[{ckpt_tag}] {split} embeddings:", X.shape)


# def _promote_checkpoint_exports_to_branch_root(tag_dir, branch_dir):
#     """
#     Copies selected checkpoint-tag files to branch root so downstream PCA cells
#     find branch_dir/train_embeddings.npy directly.
#     """
#     files_to_copy = [
#         "train_embeddings.npy",
#         "val_embeddings.npy",
#         "oac_embeddings.npy",
#         "train_patient_ids.npy",
#         "val_patient_ids.npy",
#         "oac_patient_ids.npy",
#         "train_labels.npy",
#         "val_labels.npy",
#         "oac_labels.npy",
#         "train_embeddings_metadata.csv",
#         "val_embeddings_metadata.csv",
#         "oac_embeddings_metadata.csv",
#         "train_predictions.csv",
#         "val_predictions.csv",
#         "oac_predictions.csv",
#         "class_names.npy",
#         "manifest.json",
#     ]

#     for name in files_to_copy:
#         src = tag_dir / name
#         if src.exists():
#             dst = branch_dir / name
#             shutil.copy2(src, dst)

#     print(f"[promoted] copied {tag_dir.name} exports to branch root: {branch_dir}")


# # -----------------------------
# # Export one JEPA checkpoint
# # -----------------------------

# def export_jepa_embeddings_for_checkpoint_fresh(
#     branch_name,
#     mode,
#     ckpt_path,
#     ckpt_tag,
#     train_loader,
#     val_loader,
#     oac_loader=None,
#     make_prediction_csv=True,
# ):
#     ckpt_path = Path(ckpt_path)

#     if not ckpt_path.exists():
#         print(f"[WARN] checkpoint not found: {ckpt_path}")
#         return None

#     model, local_embed_dim = build_fresh_jepa_model(mode=mode)
#     payload = load_jepa_checkpoint_inplace(
#         model=model,
#         ckpt_path=ckpt_path,
#         expected_mode=mode,
#         strict_first=True,
#     )

#     branch_dir = BRANCH_ROOT / branch_name
#     tag_dir = branch_dir / ckpt_tag

#     branch_dir.mkdir(parents=True, exist_ok=True)
#     tag_dir.mkdir(parents=True, exist_ok=True)

#     split_results = {}

#     if train_loader is not None and len(train_loader):
#         df_train, X_train = extract_jepa_embeddings_standalone(
#             model,
#             train_loader,
#             "train",
#             embed_dim_fallback=local_embed_dim,
#         )
#         _save_split_export(tag_dir, "train", df_train, X_train, ckpt_tag)
#         split_results["train"] = (df_train, X_train)
#     else:
#         print(f"[{branch_name}/{ckpt_tag}] no train loader or empty train loader")

#     if val_loader is not None and len(val_loader):
#         df_val, X_val = extract_jepa_embeddings_standalone(
#             model,
#             val_loader,
#             "val",
#             embed_dim_fallback=local_embed_dim,
#         )
#         _save_split_export(tag_dir, "val", df_val, X_val, ckpt_tag)
#         split_results["val"] = (df_val, X_val)
#     else:
#         print(f"[{branch_name}/{ckpt_tag}] no val loader or empty val loader")

#     if oac_loader is not None and len(oac_loader):
#         df_oac, X_oac = extract_jepa_embeddings_standalone(
#             model,
#             oac_loader,
#             "oac",
#             embed_dim_fallback=local_embed_dim,
#         )
#         _save_split_export(tag_dir, "oac", df_oac, X_oac, ckpt_tag)
#         split_results["oac"] = (df_oac, X_oac)
#     else:
#         print(f"[{branch_name}/{ckpt_tag}] no OAC loader or empty OAC loader")

#     if make_prediction_csv:
#         save_linear_probe_predictions(tag_dir, split_results)

#     manifest = {
#         "branch_name": branch_name,
#         "mode": mode,
#         "checkpoint_tag": ckpt_tag,
#         "checkpoint_path": str(ckpt_path),
#         "saved_dir": str(tag_dir),
#         "epoch": payload.get("epoch", None),
#         "state_keys": list(payload.keys()) if isinstance(payload, dict) else None,
#         "train_loader": train_loader_name,
#         "val_loader": val_loader_name,
#         "oac_loader": oac_loader_name,
#         "embedding_dim": int(local_embed_dim),
#         "splits_exported": list(split_results.keys()),
#         "embedding_shapes": {
#             k: list(v[1].shape) for k, v in split_results.items()
#         },
#     }

#     with open(tag_dir / "manifest.json", "w") as f:
#         json.dump(manifest, f, indent=2)

#     if ckpt_tag == PROMOTE_TAG_TO_BRANCH_ROOT:
#         _promote_checkpoint_exports_to_branch_root(tag_dir, branch_dir)

#     del model
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

#     return split_results


# # -----------------------------
# # Run JEPA exports
# # -----------------------------

# jepa_exports = {}

# for branch_name, info in JEPA_BRANCHES.items():
#     mode = info["mode"]

#     jepa_exports[branch_name] = {}

#     print("\n" + "=" * 100)
#     print(f"Exporting JEPA branch: {branch_name} | mode={mode}")
#     print("=" * 100)

#     ckpt_map = {
#         "best": info["best_ckpt"],
#         "last": info["last_ckpt"],
#     }

#     for ckpt_tag in EXPORT_TAGS:
#         ckpt_path = ckpt_map[ckpt_tag]

#         print("\n" + "-" * 80)
#         print(f"{branch_name} / {ckpt_tag}")
#         print("-" * 80)

#         jepa_exports[branch_name][ckpt_tag] = export_jepa_embeddings_for_checkpoint_fresh(
#             branch_name=branch_name,
#             mode=mode,
#             ckpt_path=ckpt_path,
#             ckpt_tag=ckpt_tag,
#             train_loader=train_export_loader,
#             val_loader=val_export_loader,
#             oac_loader=oac_export_loader,
#             make_prediction_csv=MAKE_LINEAR_PROBE_PREDICTIONS,
#         )

# print("\nDone.")
# print("JEPA embedding exports saved under:")
# for branch_name in JEPA_BRANCHES:
#     print("  ", BRANCH_ROOT / branch_name)
#     print("     best:", BRANCH_ROOT / branch_name / "best")
#     print("     last:", BRANCH_ROOT / branch_name / "last")
#     print("     promoted root files from:", PROMOTE_TAG_TO_BRANCH_ROOT)

In [ ]:
# ============================================================
# Improved Branch D/E: DINO-JEPA-style latent-prediction training
# Stabilised for PET patient embeddings
# ============================================================

import math
import random
import copy
from typing import Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from dataclasses import asdict, is_dataclass
except Exception:
    asdict = None
    is_dataclass = None


# ------------------------------------------------------------
# Recommended JEPA config overrides
# ------------------------------------------------------------
# These are intentionally conservative. They are designed to
# improve representation geometry relative to frozen DINO rather
# than aggressively alter the backbone.

def _cfg_set(name, value, overwrite=True):
    if overwrite or not hasattr(CFG, name):
        setattr(CFG, name, value)

# Core JEPA optimisation
_cfg_set("jepa_epochs", max(int(getattr(CFG, "jepa_epochs", 5)), 40), overwrite=True)
_cfg_set("jepa_lr", 3e-5, overwrite=True)
_cfg_set("jepa_min_lr", 1e-6, overwrite=True)
_cfg_set("jepa_weight_decay", 1e-4, overwrite=True)
_cfg_set("jepa_grad_accum_steps", max(1, int(getattr(CFG, "jepa_grad_accum_steps", 4))), overwrite=True)

# Backbone tuning:
# Keep frozen first. Only unfreeze after the improved head/pool-only JEPA is stable.
_cfg_set("jepa_freeze_backbone", False, overwrite=True)
_cfg_set("jepa_unfreeze_last_n_blocks", 1, overwrite=True)
_cfg_set("jepa_backbone_lr_mult", 0.10, overwrite=True)

# Predictor
_cfg_set("jepa_predictor_hidden_dim", int(getattr(CFG, "jepa_predictor_hidden_dim", 1024)), overwrite=True)
_cfg_set("jepa_dropout", 0.10, overwrite=True)
_cfg_set("jepa_use_residual_predictor", True, overwrite=True)

# Masking
_cfg_set("jepa_num_mask_blocks", 1, overwrite=True)
_cfg_set("jepa_slice_mask_scale", (0.06, 0.18), overwrite=True)
_cfg_set("jepa_fill_value", 0.0, overwrite=True)
_cfg_set("jepa_fill_mode", "mean", overwrite=True)       # "mean", "zero", or "noise"
_cfg_set("jepa_same_mask_across_slices", True, overwrite=True)
_cfg_set("jepa_deterministic_val_masks", True, overwrite=True)

# Loss weights
_cfg_set("jepa_patient_consistency_weight", 0.20, overwrite=True)
_cfg_set("jepa_anchor_weight", 0.25, overwrite=True)
_cfg_set("jepa_variance_weight", 0.03, overwrite=True)
_cfg_set("jepa_variance_target_std", 0.02, overwrite=True)

# Cross-slice JEPA
_cfg_set("jepa_cross_num_targets", 2, overwrite=True)

# EMA schedule
_cfg_set("jepa_ema_momentum_start", 0.996, overwrite=True)
_cfg_set("jepa_ema_momentum_end", 0.999, overwrite=True)

print("Improved JEPA configuration:")
for k in [
    "jepa_epochs",
    "jepa_lr",
    "jepa_min_lr",
    "jepa_freeze_backbone",
    "jepa_unfreeze_last_n_blocks",
    "jepa_backbone_lr_mult",
    "jepa_num_mask_blocks",
    "jepa_slice_mask_scale",
    "jepa_fill_mode",
    "jepa_same_mask_across_slices",
    "jepa_patient_consistency_weight",
    "jepa_anchor_weight",
    "jepa_variance_weight",
    "jepa_cross_num_targets",
]:
    print(f"  {k} = {getattr(CFG, k)}")


# ------------------------------------------------------------
# Losses and regularisation
# ------------------------------------------------------------

def _cosine_latent_loss(pred: torch.Tensor, target: torch.Tensor):
    pred = F.normalize(pred.float(), dim=-1)
    target = F.normalize(target.detach().float(), dim=-1)
    return 1.0 - (pred * target).sum(dim=-1).mean()


def _latent_prediction_loss(pred: torch.Tensor, target: torch.Tensor):
    if str(getattr(CFG, "jepa_loss", "cosine")).lower() == "smooth_l1":
        return F.smooth_l1_loss(pred.float(), target.detach().float())
    return _cosine_latent_loss(pred, target)


def _variance_latent_loss(z: torch.Tensor, target_std: float = 0.02):
    """
    VICReg-style anti-collapse term.

    Works on token collections, e.g. B*S x D or B*T x D.
    With batch size 1, this should be applied to slice/target tokens,
    not only to one patient token.
    """
    if z is None:
        return torch.tensor(0.0, device=device)

    z = z.float()

    if z.ndim > 2:
        z = z.reshape(-1, z.shape[-1])

    if z.shape[0] < 2:
        return torch.tensor(0.0, device=z.device)

    z = F.layer_norm(z, (z.shape[-1],))
    std = torch.sqrt(z.var(dim=0, unbiased=False) + 1e-4)

    return F.relu(float(target_std) - std).mean()


def _cfg_to_dict(cfg):
    if is_dataclass is not None and is_dataclass(cfg):
        try:
            return asdict(cfg)
        except Exception:
            pass
    return {k: getattr(cfg, k) for k in dir(cfg) if not k.startswith("_") and not callable(getattr(cfg, k))}


# ------------------------------------------------------------
# Improved spatial masking
# ------------------------------------------------------------

def _compute_mask_block(H, W, scale_range, deterministic=False):
    min_scale, max_scale = float(scale_range[0]), float(scale_range[1])
    scale = 0.5 * (min_scale + max_scale) if deterministic else random.uniform(min_scale, max_scale)

    area = scale * H * W
    aspect = 1.0 if deterministic else random.uniform(0.75, 1.33)

    h = int(round(math.sqrt(area / aspect)))
    w = int(round(math.sqrt(area * aspect)))

    h = max(1, min(H, h))
    w = max(1, min(W, w))

    if deterministic:
        y0 = max(0, (H - h) // 2)
        x0 = max(0, (W - w) // 2)
    else:
        y0 = random.randint(0, max(0, H - h))
        x0 = random.randint(0, max(0, W - w))

    return y0, x0, h, w


def _apply_fill(x, b, s, y0, x0, h, w, fill_mode, fill_value):
    if fill_mode == "mean":
        fill = x[b, s].mean(dim=(1, 2), keepdim=True)
        x[b, s, :, y0:y0+h, x0:x0+w] = fill
    elif fill_mode == "noise":
        mu = x[b, s].mean(dim=(1, 2), keepdim=True)
        sd = x[b, s].std(dim=(1, 2), keepdim=True).clamp_min(1e-6)
        noise = torch.randn_like(x[b, s, :, y0:y0+h, x0:x0+w]) * sd + mu
        x[b, s, :, y0:y0+h, x0:x0+w] = noise
    else:
        x[b, s, :, y0:y0+h, x0:x0+w] = float(fill_value)


def mask_random_blocks_5d(
    bag_5d: torch.Tensor,
    num_blocks: int,
    scale_range: Tuple[float, float],
    fill_value: float = 0.0,
    fill_mode: str = "mean",
    same_mask_across_slices: bool = True,
    deterministic: bool = False,
):
    """
    Improved PET-friendly masking for [B,S,C,H,W].

    Changes from original:
      - gentler default mask scale
      - mean fill instead of hard zero fill
      - optionally same spatial block across neighbouring slice triplets
      - deterministic validation masks
    """
    x = bag_5d.clone()

    B, S, C, H, W = x.shape
    num_blocks = max(1, int(num_blocks))
    fill_mode = str(fill_mode).lower()

    for b in range(B):
        for _ in range(num_blocks):
            y0, x0, h, w = _compute_mask_block(
                H,
                W,
                scale_range=scale_range,
                deterministic=deterministic,
            )

            if same_mask_across_slices:
                for s in range(S):
                    _apply_fill(x, b, s, y0, x0, h, w, fill_mode, fill_value)
            else:
                for s in range(S):
                    y0s, x0s, hs, ws = _compute_mask_block(
                        H,
                        W,
                        scale_range=scale_range,
                        deterministic=deterministic,
                    )
                    _apply_fill(x, b, s, y0s, x0s, hs, ws, fill_mode, fill_value)

    return x


# ------------------------------------------------------------
# Improved JEPA model
# ------------------------------------------------------------

class PatientJEPAModel(nn.Module):
    def __init__(
        self,
        context_backbone: nn.Module,
        target_backbone: nn.Module,
        embed_dim: int,
        cfg: Config,
        mode: str,
    ):
        super().__init__()

        self.context_backbone = context_backbone
        self.target_backbone = target_backbone
        self.embed_dim = int(embed_dim)
        self.cfg = cfg
        self.mode = str(mode)

        self.pool = PatientAttentionPool(embed_dim)

        self.predictor = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, int(cfg.jepa_predictor_hidden_dim)),
            nn.GELU(),
            nn.Dropout(float(cfg.jepa_dropout)),
            nn.Linear(int(cfg.jepa_predictor_hidden_dim), embed_dim),
        )

        set_backbone_trainability(
            self.context_backbone,
            cfg.jepa_freeze_backbone,
            cfg.jepa_unfreeze_last_n_blocks,
        )

        for p in self.target_backbone.parameters():
            p.requires_grad = False

    def context_backbone_is_trainable(self) -> bool:
        return any(p.requires_grad for p in self.context_backbone.parameters())

    def _encode_bag_cls(self, backbone: nn.Module, bag_5d: torch.Tensor, requires_grad: bool):
        B, S, C, H, W = bag_5d.shape
        x = bag_5d.reshape(B * S, C, H, W)

        chunks = []
        chunk_size = None if requires_grad else max(1, int(self.cfg.frozen_backbone_chunk_size))

        if chunk_size is None:
            feats = backbone.forward_features(x)
            cls = feats["x_norm_clstoken"]
        else:
            ctx = torch.enable_grad if requires_grad else torch.no_grad
            with ctx():
                for xs in x.split(chunk_size, dim=0):
                    feats = backbone.forward_features(xs)
                    chunks.append(feats["x_norm_clstoken"])
                cls = torch.cat(chunks, dim=0)

        return cls.reshape(B, S, -1)

    def _predict(self, tokens: torch.Tensor):
        residual = self.predictor(tokens)

        if bool(getattr(self.cfg, "jepa_use_residual_predictor", True)):
            return tokens + residual

        return residual

    def forward_bag_features(self, bag_5d: torch.Tensor):
        """
        Clean patient embedding used for export after JEPA training.
        """
        trainable = self.context_backbone_is_trainable()
        cls_tokens = self._encode_bag_cls(
            self.context_backbone,
            bag_5d,
            requires_grad=trainable,
        )
        patient_tokens, slice_weights = self.pool(cls_tokens)

        return {
            "patient_tokens": patient_tokens,
            "slice_tokens": cls_tokens,
            "slice_weights": slice_weights,
        }

    def _target_clean_tokens(self, bag_5d: torch.Tensor):
        with torch.no_grad():
            tgt_cls = self._encode_bag_cls(
                self.target_backbone,
                bag_5d,
                requires_grad=False,
            )
            tgt_patient_mean = tgt_cls.mean(dim=1)

        return tgt_cls, tgt_patient_mean

    def forward_jepa_slice(self, bag_5d: torch.Tensor):
        deterministic_mask = (
            (not self.training)
            and bool(getattr(CFG, "jepa_deterministic_val_masks", True))
        )

        masked = mask_random_blocks_5d(
            bag_5d,
            num_blocks=CFG.jepa_num_mask_blocks,
            scale_range=CFG.jepa_slice_mask_scale,
            fill_value=CFG.jepa_fill_value,
            fill_mode=CFG.jepa_fill_mode,
            same_mask_across_slices=CFG.jepa_same_mask_across_slices,
            deterministic=deterministic_mask,
        )

        trainable = self.context_backbone_is_trainable()

        # Masked context tokens
        ctx_cls = self._encode_bag_cls(
            self.context_backbone,
            masked,
            requires_grad=trainable,
        )

        # Clean context tokens, used as an anchor for exported representation
        clean_ctx_cls = self._encode_bag_cls(
            self.context_backbone,
            bag_5d,
            requires_grad=trainable,
        )

        # Frozen/EMA target tokens
        tgt_cls, tgt_patient_mean = self._target_clean_tokens(bag_5d)

        # Per-slice latent prediction
        pred = self._predict(ctx_cls)
        loss_pred = _latent_prediction_loss(pred, tgt_cls)

        # Patient-level consistency from masked context to full clean target
        ctx_patient, ctx_weights = self.pool(ctx_cls)
        loss_patient = _cosine_latent_loss(ctx_patient, tgt_patient_mean)

        # Anchor the exported clean patient representation to frozen/EMA target geometry
        clean_patient, _ = self.pool(clean_ctx_cls)
        loss_anchor = _cosine_latent_loss(clean_patient, tgt_patient_mean)

        # Anti-collapse term on predicted slice tokens and clean slice tokens
        loss_var = (
            _variance_latent_loss(
                pred,
                target_std=float(CFG.jepa_variance_target_std),
            )
            +
            _variance_latent_loss(
                clean_ctx_cls,
                target_std=float(CFG.jepa_variance_target_std),
            )
        ) * 0.5

        total = (
            loss_pred
            + float(CFG.jepa_patient_consistency_weight) * loss_patient
            + float(CFG.jepa_anchor_weight) * loss_anchor
            + float(CFG.jepa_variance_weight) * loss_var
        )

        return {
            "loss_total": total,
            "loss_pred": loss_pred.detach(),
            "loss_patient": loss_patient.detach(),
            "loss_anchor": loss_anchor.detach(),
            "loss_var": loss_var.detach(),
            "patient_tokens": clean_patient,
            "slice_weights": ctx_weights,
        }

    def _select_cross_slice_targets(self, S: int):
        n_targets = max(1, int(getattr(CFG, "jepa_cross_num_targets", 2)))
        n_targets = min(n_targets, S)

        if self.training:
            return sorted(random.sample(list(range(S)), n_targets))

        if S == 1:
            return [0]

        # Deterministic validation targets.
        if n_targets == 1:
            return [S // 2]

        grid = np.linspace(0, S - 1, n_targets)
        return sorted(set([int(round(x)) for x in grid]))

    def forward_jepa_cross_slice(self, bag_5d: torch.Tensor):
        B, S, C, H, W = bag_5d.shape

        if S < 2:
            return self.forward_jepa_slice(bag_5d)

        trainable = self.context_backbone_is_trainable()

        # Full target tokens from EMA/frozen target backbone.
        tgt_cls_all, tgt_patient_mean = self._target_clean_tokens(bag_5d)

        # Clean context anchor for exported representation.
        clean_ctx_cls = self._encode_bag_cls(
            self.context_backbone,
            bag_5d,
            requires_grad=trainable,
        )
        clean_patient, clean_weights = self.pool(clean_ctx_cls)

        target_indices = self._select_cross_slice_targets(S)

        preds = []
        targets = []
        ctx_patients = []

        for target_idx in target_indices:
            context_indices = [i for i in range(S) if i != target_idx]

            context_bag = bag_5d[:, context_indices]
            ctx_cls = self._encode_bag_cls(
                self.context_backbone,
                context_bag,
                requires_grad=trainable,
            )

            ctx_patient, _ = self.pool(ctx_cls)
            pred = self._predict(ctx_patient)

            tgt = tgt_cls_all[:, target_idx, :]

            preds.append(pred)
            targets.append(tgt)
            ctx_patients.append(ctx_patient)

        pred_stack = torch.stack(preds, dim=1)          # [B,T,D]
        target_stack = torch.stack(targets, dim=1)      # [B,T,D]
        ctx_patient_mean = torch.stack(ctx_patients, dim=1).mean(dim=1)

        loss_pred = _latent_prediction_loss(pred_stack, target_stack)
        loss_patient = _cosine_latent_loss(ctx_patient_mean, tgt_patient_mean)
        loss_anchor = _cosine_latent_loss(clean_patient, tgt_patient_mean)
        loss_var = _variance_latent_loss(
            pred_stack,
            target_std=float(CFG.jepa_variance_target_std),
        )

        total = (
            loss_pred
            + float(CFG.jepa_patient_consistency_weight) * loss_patient
            + float(CFG.jepa_anchor_weight) * loss_anchor
            + float(CFG.jepa_variance_weight) * loss_var
        )

        return {
            "loss_total": total,
            "loss_pred": loss_pred.detach(),
            "loss_patient": loss_patient.detach(),
            "loss_anchor": loss_anchor.detach(),
            "loss_var": loss_var.detach(),
            "patient_tokens": clean_patient,
            "slice_weights": clean_weights,
        }

    def forward(self, bag_5d: torch.Tensor):
        if self.mode == "cross_slice":
            return self.forward_jepa_cross_slice(bag_5d)
        return self.forward_jepa_slice(bag_5d)


# ------------------------------------------------------------
# EMA and optimiser
# ------------------------------------------------------------

@torch.no_grad()
def update_jepa_target_encoder(model: PatientJEPAModel, momentum: float):
    for p_ctx, p_tgt in zip(model.context_backbone.parameters(), model.target_backbone.parameters()):
        p_tgt.data.mul_(float(momentum)).add_((1.0 - float(momentum)) * p_ctx.data)


def _jepa_ema_momentum(step: int, total_steps: int):
    if total_steps <= 1:
        return float(CFG.jepa_ema_momentum_end)

    t = min(max(step / float(total_steps - 1), 0.0), 1.0)

    m0 = float(CFG.jepa_ema_momentum_start)
    m1 = float(CFG.jepa_ema_momentum_end)

    # Cosine ramp from m0 to m1.
    return m1 - (m1 - m0) * 0.5 * (1.0 + math.cos(math.pi * t))


def build_jepa_optimizer(model: PatientJEPAModel, cfg: Config):
    backbone_params = []
    head_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if name.startswith("context_backbone."):
            backbone_params.append(p)
        else:
            head_params.append(p)

    param_groups = []

    if len(head_params) > 0:
        param_groups.append({
            "params": head_params,
            "lr": float(cfg.jepa_lr),
            "base_lr": float(cfg.jepa_lr),
            "name": "jepa_head_pool_predictor",
        })

    if len(backbone_params) > 0:
        backbone_lr = float(cfg.jepa_lr) * float(getattr(cfg, "jepa_backbone_lr_mult", 0.1))
        param_groups.append({
            "params": backbone_params,
            "lr": backbone_lr,
            "base_lr": backbone_lr,
            "name": "jepa_context_backbone",
        })

    print("JEPA parameter groups:")
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"  {g['name']}: tensors={len(g['params'])}, params={n:,}, base_lr={g['base_lr']:.2e}")

    return torch.optim.AdamW(
        param_groups,
        lr=float(cfg.jepa_lr),
        weight_decay=float(cfg.jepa_weight_decay),
    )


def _jepa_lr_scale_for_epoch(epoch: int, total_epochs: int):
    warmup_epochs = max(1, int(getattr(CFG, "lr_warmup_epochs", 2)))

    if epoch < warmup_epochs:
        return float(epoch + 1) / float(warmup_epochs)

    denom = max(1, total_epochs - warmup_epochs)
    progress = float(epoch - warmup_epochs) / float(denom)
    return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))


def set_jepa_epoch_lr(optimizer, epoch: int, total_epochs: int):
    scale = _jepa_lr_scale_for_epoch(epoch, total_epochs)

    # Keep at least min_lr / jepa_lr fraction for all groups.
    min_scale = float(CFG.jepa_min_lr) / max(float(CFG.jepa_lr), 1e-12)
    scale = max(scale, min_scale)

    for g in optimizer.param_groups:
        g["lr"] = float(g["base_lr"]) * scale

    return optimizer.param_groups[0]["lr"]


# ------------------------------------------------------------
# Model construction
# ------------------------------------------------------------

def build_jepa_model(cfg: Config, mode: str):
    context_backbone, local_embed_dim = load_backbone(cfg)
    target_backbone, _ = load_backbone(cfg)

    if local_embed_dim is None:
        local_embed_dim = getattr(context_backbone, "embed_dim", None)

    if local_embed_dim is None:
        raise ValueError("Could not infer embed_dim for JEPA model.")

    model = PatientJEPAModel(
        context_backbone,
        target_backbone,
        int(local_embed_dim),
        cfg,
        mode=mode,
    )

    # Initialise target from context exactly before EMA updates.
    model.target_backbone.load_state_dict(
        model.context_backbone.state_dict(),
        strict=True,
    )

    return model.to(device), int(local_embed_dim)


# ------------------------------------------------------------
# JEPA training loop
# ------------------------------------------------------------

def run_jepa_epoch(
    model: PatientJEPAModel,
    loader: DataLoader,
    optimizer,
    train: bool,
    branch_name: str,
    step_state=None,
    total_train_steps: int = 1,
):
    model.train(train)

    losses_total = []
    losses_pred = []
    losses_patient = []
    losses_anchor = []
    losses_var = []

    if train:
        optimizer.zero_grad(set_to_none=True)

    iterator = (
        _make_progress_bar(loader, desc=f"{branch_name} train" if train else f"{branch_name} val")
        if "_make_progress_bar" in globals()
        else loader
    )

    accum_steps = max(1, int(CFG.jepa_grad_accum_steps))

    for it, batch in enumerate(iterator):
        bag = _bag_from_supervised_batch(batch, train=train).to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with _branch_autocast_context():
                out = model(bag)
                loss = out["loss_total"]

            if train:
                (loss / accum_steps).backward()

                if ((it + 1) % accum_steps == 0) or (it == len(loader) - 1):
                    if CFG.grad_clip_norm is not None and CFG.grad_clip_norm > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)

                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

                    if step_state is not None:
                        m = _jepa_ema_momentum(step_state["step"], total_train_steps)
                        step_state["ema_momentum"] = m
                        step_state["step"] += 1
                    else:
                        m = float(CFG.jepa_ema_momentum_end)

                    update_jepa_target_encoder(model, m)

        losses_total.append(float(out["loss_total"].detach().cpu()))
        losses_pred.append(float(out["loss_pred"].detach().cpu()))
        losses_patient.append(float(out["loss_patient"].detach().cpu()))
        losses_anchor.append(float(out["loss_anchor"].detach().cpu()))
        losses_var.append(float(out["loss_var"].detach().cpu()))

        if CFG.show_batch_progress and tqdm is not None and hasattr(iterator, "set_postfix"):
            if (it % max(int(CFG.progress_update_every), 1) == 0) or (it == len(loader) - 1):
                iterator.set_postfix({
                    "loss": f"{np.mean(losses_total):.4f}",
                    "pred": f"{np.mean(losses_pred):.4f}",
                    "anch": f"{np.mean(losses_anchor):.4f}",
                })

    return {
        "loss_total": float(np.mean(losses_total)) if losses_total else np.nan,
        "loss_pred": float(np.mean(losses_pred)) if losses_pred else np.nan,
        "loss_patient": float(np.mean(losses_patient)) if losses_patient else np.nan,
        "loss_anchor": float(np.mean(losses_anchor)) if losses_anchor else np.nan,
        "loss_var": float(np.mean(losses_var)) if losses_var else np.nan,
    }


def run_jepa_branch(branch_name: str, mode: str):
    print(f"\n=== Improved Branch {'D' if mode == 'slice' else 'E'}: {branch_name} ===")

    if len(train_ssl_loader) == 0:
        print(f"[WARN] No training data found. Skipping {branch_name}.")
        return None, None

    model, local_embed_dim = build_jepa_model(CFG, mode=mode)

    params = [p for p in model.parameters() if p.requires_grad]
    print(f"{branch_name} trainable tensors: {len(params)}")

    optimizer = build_jepa_optimizer(model, CFG)

    history = []
    best_monitor = float("inf")
    bad_epochs = 0

    ckpt_best = OUT_DIR / "checkpoints" / f"{branch_name}_best.pt"
    ckpt_last = OUT_DIR / "checkpoints" / f"{branch_name}_last.pt"

    n_updates_per_epoch = max(1, math.ceil(len(train_ssl_loader) / max(1, int(CFG.jepa_grad_accum_steps))))
    total_train_steps = max(1, int(CFG.jepa_epochs) * n_updates_per_epoch)
    step_state = {"step": 0, "ema_momentum": float(CFG.jepa_ema_momentum_start)}

    for epoch in range(int(CFG.jepa_epochs)):
        lr_now = set_jepa_epoch_lr(optimizer, epoch, int(CFG.jepa_epochs))

        train_m = run_jepa_epoch(
            model,
            train_ssl_loader,
            optimizer,
            train=True,
            branch_name=branch_name,
            step_state=step_state,
            total_train_steps=total_train_steps,
        )

        val_loader_for_jepa = val_ssl_loader if len(val_ssl_loader) else train_ssl_loader

        val_m = run_jepa_epoch(
            model,
            val_loader_for_jepa,
            optimizer,
            train=False,
            branch_name=branch_name,
            step_state=None,
            total_train_steps=total_train_steps,
        )

        row = {
            "epoch": epoch + 1,
            "lr": lr_now,
            "ema_momentum": step_state.get("ema_momentum", np.nan),
        }

        row.update({f"train_{k}": v for k, v in train_m.items()})
        row.update({f"val_{k}": v for k, v in val_m.items()})

        history.append(row)

        monitor = val_m["loss_total"] if np.isfinite(val_m["loss_total"]) else train_m["loss_total"]

        print(
            f"[{branch_name}] epoch {epoch+1:03d}/{CFG.jepa_epochs} | "
            f"train={train_m['loss_total']:.4f} | val={val_m['loss_total']:.4f} | "
            f"pred={train_m['loss_pred']:.4f} | patient={train_m['loss_patient']:.4f} | "
            f"anchor={train_m['loss_anchor']:.4f} | var={train_m['loss_var']:.4f} | "
            f"lr={lr_now:.2e} | ema={step_state.get('ema_momentum', np.nan):.5f}"
        )

        torch.save(
            {
                "model": model.state_dict(),
                "epoch": epoch,
                "history": history,
                "cfg": _cfg_to_dict(CFG),
                "mode": mode,
                "improved_jepa": True,
            },
            str(ckpt_last),
        )

        if np.isfinite(monitor) and monitor < best_monitor - float(CFG.early_stopping_min_delta):
            best_monitor = float(monitor)
            bad_epochs = 0

            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "history": history,
                    "cfg": _cfg_to_dict(CFG),
                    "mode": mode,
                    "improved_jepa": True,
                },
                str(ckpt_best),
            )

            print(f"  -> saved new improved JEPA best: {ckpt_best} (monitor={best_monitor:.4f})")
        else:
            bad_epochs += 1
            print(f"  -> no improved JEPA gain for {bad_epochs}/{CFG.early_stopping_patience} epochs")

        if int(CFG.early_stopping_patience) > 0 and bad_epochs >= int(CFG.early_stopping_patience):
            print(f"[{branch_name}] early stopping triggered.")
            break

    branch_dir = BRANCH_ROOT / branch_name
    branch_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(history).to_csv(
        branch_dir / "jepa_training_history_improved.csv",
        index=False,
    )

    if ckpt_best.exists():
        payload = torch.load(str(ckpt_best), map_location=device)
        model.load_state_dict(payload["model"], strict=True)

    exports = export_branch_embeddings(
        model,
        branch_name=branch_name,
        embed_dim_fallback=local_embed_dim,
    )

    return model, exports


# ------------------------------------------------------------
# Run improved JEPA branches
# ------------------------------------------------------------

jepa_slice_model = None
jepa_cross_slice_model = None

if branch_enabled("dino_jepa_slice"):
    jepa_slice_model, jepa_slice_exports = run_jepa_branch("dino_jepa_slice", mode="slice")

    if jepa_slice_model is not None:
        del jepa_slice_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Skipping improved Branch D: dino_jepa_slice")


if branch_enabled("dino_jepa_cross_slice"):
    jepa_cross_slice_model, jepa_cross_slice_exports = run_jepa_branch("dino_jepa_cross_slice", mode="cross_slice")

    if jepa_cross_slice_model is not None:
        del jepa_cross_slice_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Skipping improved Branch E: dino_jepa_cross_slice")

## Post-training important-region analysis for extracted embeddings

Class activation maps require a supervised class head. The SSL model here primarily produces **patient embeddings**, so the added post-training analysis asks a slightly different question: **which image regions most affect, or most align with, the extracted embedding?**

This cell adds two complementary methods:

1. **Patch-token similarity map** — highlights patches whose DINOv2 patch tokens are most aligned with the patient embedding.
2. **Embedding occlusion map** — masks small windows and measures how much the patient embedding changes. A large cosine-distance change means the region is important for the final embedding.

These maps complement the existing class/attention-style maps and are more directly aligned with downstream clustering or retrieval based on extracted embeddings.


In [ ]:
# ============================================================
# Embedding-conditioned important-region maps after SSL training
# ============================================================
@torch.no_grad()
def _normalise_heatmap_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = x - np.nanmin(x)
    denom = np.nanmax(x)
    if np.isfinite(denom) and denom > 0:
        x = x / denom
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

@torch.no_grad()
def patient_embedding_and_slice_weights(model: PatientDinoWrapper, bag_5d: torch.Tensor):
    model.eval()
    out = model.forward_bag_features(bag_5d)
    emb = F.normalize(out["patient_tokens"].float(), dim=-1)
    slice_weights = out["slice_weights"].detach().float()
    return emb, slice_weights

@torch.no_grad()
def embedding_patch_similarity_map(
    model: PatientDinoWrapper,
    image_4d: torch.Tensor,
    reference_embedding: torch.Tensor,
):
    """Map DINOv2 patch-token cosine similarity to a patient/reference embedding."""
    model.eval()
    feats = model.backbone.forward_features(image_4d)
    patch_tokens = feats["x_norm_patchtokens"].float()  # [B, N, D]
    patch_tokens = F.normalize(patch_tokens, dim=-1)
    ref = F.normalize(reference_embedding.float(), dim=-1)
    if ref.ndim == 1:
        ref = ref.unsqueeze(0)
    if ref.shape[0] == 1 and patch_tokens.shape[0] > 1:
        ref = ref.expand(patch_tokens.shape[0], -1)

    scores = torch.einsum("bnd,bd->bn", patch_tokens, ref)
    patch_size = int(getattr(model.backbone, "patch_size", 14))
    if not isinstance(patch_size, int):
        patch_size = patch_size[0] if isinstance(patch_size, (tuple, list)) else 14
    H = image_4d.shape[-2] // patch_size
    W = image_4d.shape[-1] // patch_size
    heat = scores.reshape(image_4d.shape[0], H, W)
    heat = F.interpolate(heat.unsqueeze(1), size=image_4d.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)
    return heat

@torch.no_grad()
def embedding_occlusion_importance_map(
    model: PatientDinoWrapper,
    bag_5d: torch.Tensor,
    slice_idx: int,
    window_size: int = 56,
    stride: int = 28,
    fill_value: float = 0.0,
    max_windows: Optional[int] = None,
):
    """Occlude windows in one slice and score cosine-distance change in the patient embedding."""
    model.eval()
    bag_5d = bag_5d.clone()
    base_emb, _ = patient_embedding_and_slice_weights(model, bag_5d)
    _, _, _, H, W = bag_5d.shape

    windows = []
    for y0 in range(0, H, stride):
        y1 = min(y0 + window_size, H)
        if y1 - y0 < max(4, window_size // 3):
            continue
        for x0 in range(0, W, stride):
            x1 = min(x0 + window_size, W)
            if x1 - x0 < max(4, window_size // 3):
                continue
            windows.append((y0, y1, x0, x1))

    if max_windows is not None and len(windows) > int(max_windows):
        # deterministic sub-sampling across the image grid
        keep_idx = np.linspace(0, len(windows) - 1, int(max_windows)).round().astype(int)
        windows = [windows[i] for i in keep_idx]

    heat_sum = np.zeros((H, W), dtype=np.float32)
    heat_count = np.zeros((H, W), dtype=np.float32)
    rows = []

    for y0, y1, x0, x1 in windows:
        occ = bag_5d.clone()
        occ[:, slice_idx, :, y0:y1, x0:x1] = fill_value
        occ_emb, _ = patient_embedding_and_slice_weights(model, occ)
        cosine_sim = F.cosine_similarity(base_emb, occ_emb, dim=-1).mean().item()
        cosine_distance = float(1.0 - cosine_sim)
        positive_change = max(cosine_distance, 0.0)
        heat_sum[y0:y1, x0:x1] += positive_change
        heat_count[y0:y1, x0:x1] += 1.0
        rows.append({
            "slice_idx": int(slice_idx),
            "y0": int(y0), "y1": int(y1), "x0": int(x0), "x1": int(x1),
            "cosine_similarity_after_occlusion": float(cosine_sim),
            "cosine_distance_from_baseline": cosine_distance,
        })

    heat = heat_sum / np.maximum(heat_count, 1.0)
    heat = _normalise_heatmap_np(heat)
    return heat, pd.DataFrame(rows)

@torch.no_grad()
def visualize_embedding_important_regions_for_patient(
    model: PatientDinoWrapper,
    entry: Dict,
    cfg: Config,
    save_dir: Path,
    ckpt_tag: str = "best",
    top_k_slices: int = 3,
    occlusion_window_size: int = 56,
    occlusion_stride: int = 28,
):
    save_dir.mkdir(parents=True, exist_ok=True)
    triplets, centers, bag = load_single_patient_eval_bag(entry, cfg)
    bag_5d = bag.unsqueeze(0).to(device)

    patient_emb, slice_weights = patient_embedding_and_slice_weights(model, bag_5d)
    slice_weights_np = slice_weights[0].detach().cpu().numpy()
    ranked = np.argsort(slice_weights_np)[::-1]
    top_idx = ranked[: min(int(top_k_slices), len(ranked))]

    all_rows = []
    for idx in top_idx:
        img_tensor = bag_5d[0, idx:idx+1]
        img_rgb = denormalize_tensor_image(img_tensor[0])

        # Existing attention-style signal.
        try:
            cls_heat = cls_patch_attention_map(model.backbone, img_tensor)[0].detach().cpu().numpy()
            cls_heat = _normalise_heatmap_np(cls_heat)
        except Exception as e:
            print(f"CLS attention map failed for {entry['patient_id']} z={centers[idx]}: {e}")
            cls_heat = np.zeros(img_rgb.shape[:2], dtype=np.float32)

        # Embedding-conditioned patch similarity.
        sim_heat = embedding_patch_similarity_map(model, img_tensor, patient_emb[0])[0].detach().cpu().numpy()
        sim_heat = _normalise_heatmap_np(sim_heat)

        # Direct embedding perturbation sensitivity.
        occ_heat, occ_df = embedding_occlusion_importance_map(
            model,
            bag_5d,
            slice_idx=int(idx),
            window_size=int(occlusion_window_size),
            stride=int(occlusion_stride),
            fill_value=0.0,  # zero in normalised space is approximately ImageNet-mean fill
        )
        occ_df.insert(0, "patient_id", entry["patient_id"])
        occ_df.insert(1, "checkpoint", ckpt_tag)
        occ_df.insert(2, "z_center", int(centers[idx]))
        occ_df.insert(3, "slice_attention_weight", float(slice_weights_np[idx]))
        all_rows.append(occ_df)

        fig, axes = plt.subplots(1, 5, figsize=(18, 4))
        axes[0].imshow(img_rgb)
        axes[0].set_title(f"Image z={centers[idx]}\nslice_w={slice_weights_np[idx]:.3f}")
        axes[0].axis("off")

        axes[1].imshow(cls_heat, cmap="jet")
        axes[1].set_title("CLS attention")
        axes[1].axis("off")

        axes[2].imshow(overlay_heatmap_on_image(img_rgb, sim_heat))
        axes[2].set_title("Patch similarity\nto patient embedding")
        axes[2].axis("off")

        axes[3].imshow(occ_heat, cmap="jet")
        axes[3].set_title("Embedding occlusion\nimportance")
        axes[3].axis("off")

        axes[4].imshow(overlay_heatmap_on_image(img_rgb, occ_heat))
        axes[4].set_title("Occlusion overlay")
        axes[4].axis("off")

        plt.suptitle(f"Embedding-important regions: {entry['patient_id']} [{ckpt_tag}]", y=1.05)
        plt.tight_layout()
        out_png = save_dir / f"{entry['patient_id']}_z{int(centers[idx])}_embedding_regions_{ckpt_tag}.png"
        plt.savefig(out_png, dpi=220, bbox_inches="tight")
        plt.show()
        print("Saved:", out_png)

    if all_rows:
        all_df = pd.concat(all_rows, ignore_index=True)
        out_csv = save_dir / f"{entry['patient_id']}_embedding_occlusion_windows_{ckpt_tag}.csv"
        all_df.to_csv(out_csv, index=False)
        print("Saved:", out_csv)
        return all_df
    return pd.DataFrame()

# -----------------------------
# Run examples on OAC patients when available; otherwise validation patients.
# -----------------------------
embedding_region_root = OUT_DIR / "artifacts" / "embedding_region_importance"
embedding_region_root.mkdir(parents=True, exist_ok=True)

if BEST_CKPT_PATH.exists():
    try:
        teacher_region = load_teacher_weights_for_visualization(teacher, BEST_CKPT_PATH)
        example_entries = oac_entries if len(oac_entries) > 0 else val_entries
        for entry in example_entries[: min(2, len(example_entries))]:
            try:
                visualize_embedding_important_regions_for_patient(
                    teacher_region,
                    entry,
                    CFG,
                    save_dir=embedding_region_root / "best",
                    ckpt_tag="best",
                    top_k_slices=min(2, CFG.n_attention_slices_to_plot),
                    occlusion_window_size=56,
                    occlusion_stride=28,
                )
            except Exception as e:
                print(f"Embedding-region analysis failed for {entry['patient_id']}: {e}")
    except Exception as e:
        print("Could not load best checkpoint for embedding-region analysis:", e)
else:
    print("Best checkpoint not found yet. Run training first, then re-run this cell.")


In [ ]:
# ============================================================
# Standalone script:
# Branch-wise train/val 3D PCA panel + centroid-distance analysis
#
# PCA basis:
#   fitted ONCE using A_frozen_dino_baseline train + val embeddings
#
# Projection:
#   all branches projected into the same frozen-baseline 3D PCA space
#
# Outputs:
#   - high-resolution 3D multi-panel PCA figure
#   - high-resolution individual 3D PCA figures
#   - projected PCA scores
#   - class centroid distances in:
#       1. original L2-normalised embedding space
#       2. frozen-baseline 3D PCA space
#   - grayscale centroid-distance heatmaps
#   - manifest JSON
# ============================================================

from pathlib import Path
from itertools import combinations
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


# ============================================================
# 0. Paths and global settings
# ============================================================

ROOT = Path("/mnt/d/data/patient_dinov2_ssl_outputs_branches_6")
BRANCH_ROOT = ROOT / "artifacts" / "branches"

BRANCHES = {
    "A_frozen_dino_baseline": BRANCH_ROOT / "frozen_dino_baseline",
    "B_supervised_frozen_classifier": BRANCH_ROOT / "supervised_frozen_classifier",
    "C_dino_ssl_last": ROOT / "artifacts" / "last",
    "D_dino_jepa_slice": BRANCH_ROOT / "dino_jepa_slice",
    "E_dino_jepa_cross_slice": BRANCH_ROOT / "dino_jepa_cross_slice",
}

BRANCH_DISPLAY_NAMES = {
    "A_frozen_dino_baseline": "A. Frozen baseline",
    "B_supervised_frozen_classifier": "B. Supervised classifier",
    "C_dino_ssl_last": "C. SSL",
    "D_dino_jepa_slice": "D. JEPA",
    "E_dino_jepa_cross_slice": "E. DINO-JEPA cross-slice",
}

BASELINE_BRANCH = "A_frozen_dino_baseline"

OUT_DIR = ROOT / "artifacts" / "branch_train_val_PANEL_3D_pca_FIXED_TO_FROZEN_BASELINE_highres_centroids"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CENTROID_OUT_DIR = OUT_DIR / "centroid_distance_analysis"
CENTROID_OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. File names
# ============================================================

TRAIN_EMB_NAMES = [
    "train_embeddings.npy",
    "training_embeddings.npy",
    "embeddings_train.npy",
    "train_features.npy",
]

VAL_EMB_NAMES = [
    "val_embeddings.npy",
    "validation_embeddings.npy",
    "embeddings_val.npy",
    "val_features.npy",
]

TRAIN_LABEL_NAMES = [
    "train_labels.npy",
    "train_targets.npy",
    "train_classes.npy",
    "train_y.npy",
    "train_class_labels.npy",
]

VAL_LABEL_NAMES = [
    "val_labels.npy",
    "validation_labels.npy",
    "val_targets.npy",
    "val_classes.npy",
    "val_y.npy",
    "val_class_labels.npy",
]

PREFERRED_LABEL_COLUMNS = [
    "class_name",
    "label",
    "class",
    "target",
    "Clinical_Stage",
    "Treatment_Group",
    "Sex",
    "Response",
]


# ============================================================
# 2. Analysis settings
# ============================================================

SPLITS_TO_USE = ("train", "val")
BASELINE_SPLITS_FOR_PCA = ("train", "val")

L2_NORMALISE_BEFORE_PCA = True

OVERLAY_BASELINE_REFERENCE = True
LEGEND_PANEL_BRANCH = "A_frozen_dino_baseline"

# Internal train/val lung is not removed.
# This only matters if an external split exists in projected_records.
SKIP_EXTERNAL_LUNG = True

# Centroid settings
CENTROID_SPLIT_SCOPES = {
    "train_val": ("train", "val"),
    "train_only": ("train",),
    "val_only": ("val",),
}

MIN_CLASS_N_FOR_CENTROID = 2

EMBEDDING_DISTANCE_METRICS = [
    "euclidean",
    "cosine",
]


# ============================================================
# 3. Plot settings
# ============================================================

# High-resolution figure settings
N_COLS = 2

# Larger figure, more horizontal room, less wasted vertical space.
FIGSIZE_PANEL = (8.5, 7.5)

# Higher DPI for PNG export
SAVE_DPI = 600

# Reduce white space between panels
SUBPLOT_WSPACE = 0.02
SUBPLOT_HSPACE = 0.02

# Reduce outer margins
FIG_LEFT = 0.01
FIG_RIGHT = 0.99
FIG_BOTTOM = 0.01
FIG_TOP = 0.06

# Camera
VIEW_ELEV = 22
VIEW_AZIM = 42

SPLIT_MARKERS = {
    "train": "o",
    "val": "^",
    "external": "X",
}

SPLIT_SIZES = {
    "train": 24,
    "val": 42,
    "external": 72,
}

SPLIT_ALPHA = {
    "train": 0.70,
    "val": 0.95,
    "external": 1.00,
}

CLASS_ORDER = [
    "oesophagus",
    "melanoma",
    "lymphoma",
    "lung",
]

FIXED_CLASS_COLOURS = {
    "oesophagus": "pink",
    "melanoma": "purple",
    "lymphoma": "tab:blue",
    "lung": "green",
}


# ============================================================
# 4. Helper functions
# ============================================================

def find_first_existing(folder, names):
    folder = Path(folder)

    for name in names:
        hits = list(folder.glob(f"**/{name}"))
        if len(hits) > 0:
            return hits[0]

    return None


def load_embedding_array(path):
    path = Path(path)

    X = np.load(path, allow_pickle=True)
    X = np.asarray(X)

    if X.dtype == object:
        X = np.vstack(X)

    if X.ndim == 2:
        X = X.astype(np.float32)

    elif X.ndim == 3:
        print(f"[info] {path}: 3D array {X.shape}; mean-pooling axis=1.")
        X = X.mean(axis=1).astype(np.float32)

    elif X.ndim > 3:
        print(f"[info] {path}: {X.ndim}D array {X.shape}; flattening to N x D.")
        X = X.reshape(X.shape[0], -1).astype(np.float32)

    else:
        raise ValueError(f"Unsupported embedding shape for {path}: {X.shape}")

    return X


def load_label_array(path, n_expected=None):
    path = Path(path)

    y = np.load(path, allow_pickle=True)
    y = np.asarray(y).astype(str)

    if y.ndim > 1:
        y = y.reshape(-1)

    if n_expected is not None and len(y) != n_expected:
        print(
            f"[warning] label length mismatch for {path}: "
            f"{len(y)} vs expected {n_expected}"
        )
        return None

    return y


def find_label_from_csv(folder, split, n_expected):
    folder = Path(folder)

    candidate_csvs = []
    candidate_csvs += list(folder.glob(f"**/{split}*.csv"))
    candidate_csvs += list(folder.glob("**/*metadata*.csv"))
    candidate_csvs += list(folder.glob("**/*labels*.csv"))
    candidate_csvs += list(folder.glob("**/*predictions*.csv"))

    for csv_path in candidate_csvs:
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue

        if len(df) != n_expected:
            continue

        for col in PREFERRED_LABEL_COLUMNS:
            if col in df.columns:
                y = df[col].astype(str).values
                return y, csv_path, col

    return None, None, None


def load_split_labels(folder, split, n_expected):
    if split == "train":
        label_path = find_first_existing(folder, TRAIN_LABEL_NAMES)
        default_label = "Unlabelled"

    elif split == "val":
        label_path = find_first_existing(folder, VAL_LABEL_NAMES)
        default_label = "Unlabelled"

    else:
        raise ValueError(f"Unsupported split: {split}")

    if label_path is not None:
        y = load_label_array(label_path, n_expected=n_expected)

        if y is not None:
            return y, str(label_path)

    y, csv_path, col = find_label_from_csv(folder, split, n_expected)

    if y is not None:
        return y, f"{csv_path}:{col}"

    return np.array([default_label] * n_expected), None


def l2_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)


def get_split_paths(branch_dir):
    branch_dir = Path(branch_dir)

    return {
        "train": find_first_existing(branch_dir, TRAIN_EMB_NAMES),
        "val": find_first_existing(branch_dir, VAL_EMB_NAMES),
    }


def clean_label_string(x):
    xs = str(x).strip()
    xl = xs.lower()

    xl = re.sub(r"\.nii(\.gz)?$", "", xl)

    xl = (
        xl.replace("-", " ")
          .replace("_", " ")
          .replace("/", " ")
          .replace("\\", " ")
          .replace(".", " ")
          .strip()
    )

    xl = re.sub(r"\s+", " ", xl)

    return xl


def standardise_class_label(x):
    xl = clean_label_string(x)
    tokens = xl.split()

    # Numeric labels
    if xl in ["0", "0 0", "0.0"]:
        return "oesophagus"

    if xl in ["1", "1 0", "1.0"]:
        return "melanoma"

    if xl in ["2", "2 0", "2.0"]:
        return "lymphoma"

    if xl in ["3", "3 0", "3.0"]:
        return "lung"

    # Oesophagus / OAC variants
    if (
        xl in [
            "oc",
            "oac",
            "oesophagus",
            "esophagus",
            "oesophageal cancer",
            "esophageal cancer",
        ]
        or "oesoph" in xl
        or "esoph" in xl
        or "oac" in xl
    ):
        return "oesophagus"

    # Melanoma variants
    if (
        xl in [
            "mela",
            "mel",
            "melanoma",
            "external melanoma",
            "mela cancer",
            "mel cancer",
            "melanoma cancer",
        ]
        or any(tok.startswith("mela") for tok in tokens)
        or any(tok.startswith("melr") for tok in tokens)
        or "melanoma" in xl
        or "mela" in xl
    ):
        return "melanoma"

    # Lung variants
    if (
        xl in [
            "lung",
            "lung cancer",
            "external lung",
            "nsclc",
            "non small cell lung cancer",
            "non-small cell lung cancer",
        ]
        or any(tok.startswith("lung") for tok in tokens)
        or "lung" in xl
    ):
        return "lung"

    # Lymphoma variants
    if (
        xl in [
            "lym",
            "lymphoma",
            "external lymphoma",
            "lymphoma cancer",
        ]
        or any(tok.startswith("lym") for tok in tokens)
        or "lymphoma" in xl
    ):
        return "lymphoma"

    return str(x).strip()


def standardise_label_array(labels):
    return np.array([standardise_class_label(x) for x in labels]).astype(str)


def load_branch_train_val(branch_name, branch_dir):
    branch_dir = Path(branch_dir)
    paths = get_split_paths(branch_dir)

    train_path = paths["train"]
    val_path = paths["val"]

    if train_path is None or val_path is None:
        print(f"[skip] {branch_name}: train or val embedding file not found.")
        print(f"       searched under: {branch_dir}")
        return None

    X_train = load_embedding_array(train_path)
    X_val = load_embedding_array(val_path)

    y_train, train_label_source = load_split_labels(
        branch_dir,
        "train",
        len(X_train),
    )

    y_val, val_label_source = load_split_labels(
        branch_dir,
        "val",
        len(X_val),
    )

    y_train = standardise_label_array(y_train)
    y_val = standardise_label_array(y_val)

    print(f"[loaded] {branch_name}")
    print(f"         train: {X_train.shape} | {train_path}")
    print(f"         val  : {X_val.shape} | {val_path}")
    print(f"         train labels: {train_label_source}")
    print(f"         val labels  : {val_label_source}")
    print(f"         train class counts: {pd.Series(y_train).value_counts().to_dict()}")
    print(f"         val class counts  : {pd.Series(y_val).value_counts().to_dict()}")

    return {
        "branch": branch_name,
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val,
        "train_path": train_path,
        "val_path": val_path,
    }


def combine_branch_embeddings(rec, splits=("train", "val")):
    X_list = []
    split_list = []
    label_list = []

    for split in splits:
        X = rec.get(f"X_{split}", None)
        y = rec.get(f"y_{split}", None)

        if X is None:
            continue

        if y is None:
            y = np.array(["Unlabelled"] * len(X)).astype(str)
        else:
            y = standardise_label_array(np.asarray(y).astype(str))

        X_list.append(X)
        split_list.append(np.array([split] * len(X)))
        label_list.append(y)

    if len(X_list) == 0:
        return None, None, None

    X_all = np.vstack(X_list).astype(np.float32)
    split_all = np.concatenate(split_list)
    label_all = np.concatenate(label_list)

    finite_rows = np.isfinite(X_all).all(axis=1)

    return X_all[finite_rows], split_all[finite_rows], label_all[finite_rows]


# ============================================================
# 5. Load all branches
# ============================================================

branch_records = {}

for branch_name, branch_dir in BRANCHES.items():
    print("\n" + "=" * 90)
    print(branch_name)
    print("=" * 90)

    rec = load_branch_train_val(branch_name, branch_dir)

    if rec is not None:
        branch_records[branch_name] = rec


if BASELINE_BRANCH not in branch_records:
    raise RuntimeError(
        f"Baseline branch not loaded: {BASELINE_BRANCH}. "
        "Cannot fit fixed PCA basis."
    )


# ============================================================
# 6. Fit 3D PCA on frozen baseline train + val
# ============================================================

baseline_rec = branch_records[BASELINE_BRANCH]

X_base, split_base, label_base = combine_branch_embeddings(
    baseline_rec,
    splits=BASELINE_SPLITS_FOR_PCA,
)

if X_base is None:
    raise RuntimeError("Could not construct baseline matrix for PCA.")

if X_base.shape[0] < 4:
    raise RuntimeError(
        f"Need enough baseline samples for 3D PCA. Got only {X_base.shape[0]} samples."
    )

if L2_NORMALISE_BEFORE_PCA:
    X_base_for_pca = l2_normalize(X_base)
else:
    X_base_for_pca = X_base.copy()

baseline_mean = X_base_for_pca.mean(axis=0, keepdims=True)
X_base_centered = X_base_for_pca - baseline_mean

fixed_pca = PCA(n_components=3)
Z_base = fixed_pca.fit_transform(X_base_centered)

pc1_var = fixed_pca.explained_variance_ratio_[0]
pc2_var = fixed_pca.explained_variance_ratio_[1]
pc3_var = fixed_pca.explained_variance_ratio_[2]

print("\nFixed 3D PCA fitted on frozen DINO baseline only")
print(f"  baseline branch       : {BASELINE_BRANCH}")
print(f"  baseline splits used  : {BASELINE_SPLITS_FOR_PCA}")
print(f"  baseline matrix shape : {X_base.shape}")
print(f"  PC1 variance          : {pc1_var:.4f}")
print(f"  PC2 variance          : {pc2_var:.4f}")
print(f"  PC3 variance          : {pc3_var:.4f}")
print(f"  cumulative PC1-3      : {(pc1_var + pc2_var + pc3_var):.4f}")


def transform_with_fixed_baseline_pca_3d(X):
    X = np.asarray(X, dtype=np.float32)

    if X.shape[1] != baseline_mean.shape[1]:
        raise ValueError(
            f"Embedding dimension mismatch: branch dim={X.shape[1]}, "
            f"baseline dim={baseline_mean.shape[1]}. "
            "Cannot project this branch onto frozen-baseline PCA."
        )

    finite_rows = np.isfinite(X).all(axis=1)
    X = X[finite_rows]

    if L2_NORMALISE_BEFORE_PCA:
        Xp = l2_normalize(X)
    else:
        Xp = X.copy()

    Xp = Xp - baseline_mean
    Z = fixed_pca.transform(Xp)

    return Z, finite_rows


# ============================================================
# 7. Project all branches into frozen-baseline 3D PCA space
# ============================================================

projected_records = {}
all_Z_for_limits = []

for branch_name, rec in branch_records.items():
    X_all, split_all, label_all = combine_branch_embeddings(
        rec,
        splits=SPLITS_TO_USE,
    )

    if X_all is None:
        print(f"[skip] {branch_name}: no train/val embeddings.")
        continue

    try:
        Z, finite_rows = transform_with_fixed_baseline_pca_3d(X_all)
    except Exception as e:
        print(f"[projection skip] {branch_name}: {e}")
        continue

    split_all = split_all[finite_rows]
    label_all = standardise_label_array(label_all[finite_rows])

    projected_records[branch_name] = {
        "Z": Z,
        "split": split_all,
        "label": label_all,
    }

    all_Z_for_limits.append(Z)


if len(projected_records) == 0:
    raise RuntimeError("No projected branch records available for plotting.")


# ============================================================
# 8. Optional external lung filter
# ============================================================

if SKIP_EXTERNAL_LUNG:
    for branch_name in list(projected_records.keys()):
        rec = projected_records[branch_name]

        if "external" in set(rec["split"]):
            keep_mask = ~(
                (rec["split"] == "external") &
                (rec["label"] == "lung")
            )

            removed = int((~keep_mask).sum())

            projected_records[branch_name]["Z"] = rec["Z"][keep_mask]
            projected_records[branch_name]["split"] = rec["split"][keep_mask]
            projected_records[branch_name]["label"] = rec["label"][keep_mask]

            print(
                f"[external filter] {branch_name}: "
                f"removed {removed} external lung cases"
            )


# ============================================================
# 9. Consistent class colour map
# ============================================================

all_labels_for_color = []

for branch_name, rec in projected_records.items():
    all_labels_for_color.extend(rec["label"].astype(str).tolist())

unique_observed = sorted(set(all_labels_for_color))

unique_labels_global = [
    c for c in CLASS_ORDER
    if c in unique_observed
] + [
    c for c in unique_observed
    if c not in CLASS_ORDER
]

print("\nFinal labels used for colour map:")
print(unique_labels_global)

fallback_cmap = plt.get_cmap("tab20")

LABEL_TO_COLOR = {}

for i, lab in enumerate(unique_labels_global):
    if lab in FIXED_CLASS_COLOURS:
        LABEL_TO_COLOR[lab] = FIXED_CLASS_COLOURS[lab]
    else:
        LABEL_TO_COLOR[lab] = fallback_cmap(i % fallback_cmap.N)

print("\nFinal class colour map:")
for lab in unique_labels_global:
    print(f"  {lab}: {LABEL_TO_COLOR[lab]}")


# ============================================================
# 10. Consistent 3D axis limits
# ============================================================

Z_lim = np.vstack([rec["Z"] for rec in projected_records.values()])

x_min, x_max = np.nanpercentile(Z_lim[:, 0], [1, 99])
y_min, y_max = np.nanpercentile(Z_lim[:, 1], [1, 99])
z_min, z_max = np.nanpercentile(Z_lim[:, 2], [1, 99])

x_pad = 0.08 * (x_max - x_min + 1e-12)
y_pad = 0.08 * (y_max - y_min + 1e-12)
z_pad = 0.08 * (z_max - z_min + 1e-12)

FIXED_XLIM = (x_min - x_pad, x_max + x_pad)
FIXED_YLIM = (y_min - y_pad, y_max + y_pad)
FIXED_ZLIM = (z_min - z_pad, z_max + z_pad)



# ============================================================
# 11. High-resolution multi-panel 3D plot
#     PATCHED: one central legend for the whole panel grid
# ============================================================

branch_names_to_plot = [
    b for b in BRANCHES.keys()
    if b in projected_records
]

n_panels = len(branch_names_to_plot)
n_cols = min(N_COLS, n_panels)
n_rows = int(np.ceil(n_panels / n_cols))

fig = plt.figure(figsize=FIGSIZE_PANEL, constrained_layout=False)

axes = []

for panel_idx in range(n_rows * n_cols):
    if panel_idx < n_panels:
        ax = fig.add_subplot(n_rows, n_cols, panel_idx + 1, projection="3d")
    else:
        ax = fig.add_subplot(n_rows, n_cols, panel_idx + 1)
        ax.axis("off")

    axes.append(ax)


for panel_idx, branch_name in enumerate(branch_names_to_plot):
    ax = axes[panel_idx]

    rec = projected_records[branch_name]
    Z = rec["Z"]
    split_all = rec["split"]
    label_all = rec["label"]

    # --------------------------------------------------------
    # Optional frozen-baseline reference cloud
    # --------------------------------------------------------
    if OVERLAY_BASELINE_REFERENCE and branch_name != BASELINE_BRANCH:
        ax.scatter(
            Z_base[:, 0],
            Z_base[:, 1],
            Z_base[:, 2],
            s=7,
            alpha=0.10,
            marker=".",
            color="gray",
            linewidths=0,
            depthshade=False,
            label="Frozen baseline reference",
        )

    # --------------------------------------------------------
    # Plot train/val embeddings by class
    # --------------------------------------------------------
    for lab in unique_labels_global:
        for split in SPLITS_TO_USE:
            mask = (label_all == lab) & (split_all == split)

            if mask.sum() == 0:
                continue

            ax.scatter(
                Z[mask, 0],
                Z[mask, 1],
                Z[mask, 2],
                s=SPLIT_SIZES.get(split, 40),
                alpha=SPLIT_ALPHA.get(split, 0.9),
                marker=SPLIT_MARKERS.get(split, "o"),
                color=LABEL_TO_COLOR[lab],
                edgecolors="black" if split == "val" else "none",
                linewidths=0.45 if split == "val" else 0.0,
                depthshade=False,
            )

    # --------------------------------------------------------
    # Panel formatting
    # --------------------------------------------------------
    ax.set_title(
        BRANCH_DISPLAY_NAMES.get(branch_name, branch_name),
        fontsize=9,
        fontweight="bold",
        pad=2,
    )

    ax.set_xlim(FIXED_XLIM)
    ax.set_ylim(FIXED_YLIM)
    ax.set_zlim(FIXED_ZLIM)

    ax.set_xlabel(
        f"PC1\n({pc1_var * 100:.1f}%)",
        fontsize=8,
        labelpad=2,
    )

    ax.set_ylabel(
        f"PC2\n({pc2_var * 100:.1f}%)",
        fontsize=8,
        labelpad=2,
    )

    ax.set_zlabel(
        f"PC3\n({pc3_var * 100:.1f}%)",
        fontsize=8,
        labelpad=2,
    )

    ax.tick_params(axis="both", which="major", labelsize=6, pad=0)
    ax.tick_params(axis="z", which="major", labelsize=6, pad=0)

    ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)

    ax.xaxis.pane.set_alpha(0.04)
    ax.yaxis.pane.set_alpha(0.04)
    ax.zaxis.pane.set_alpha(0.04)

    ax.grid(True, linestyle="--", linewidth=0.35, alpha=0.22)

    # Reduce 3D box whitespace visually
    try:
        ax.set_box_aspect((1.0, 1.0, 0.85))
    except Exception:
        pass


# ============================================================
# PATCH: central figure-level legend for the whole panel grid
# ============================================================

class_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=str(lab),
        markerfacecolor=LABEL_TO_COLOR[lab],
        markeredgecolor="black",
        markersize=7,
        linestyle="None",
    )
    for lab in unique_labels_global
]

split_handles = [
    Line2D(
        [0],
        [0],
        marker=SPLIT_MARKERS["train"],
        color="black",
        label="train",
        markerfacecolor="white",
        markeredgecolor="black",
        linestyle="None",
        markersize=7,
    ),
    Line2D(
        [0],
        [0],
        marker=SPLIT_MARKERS["val"],
        color="black",
        label="val",
        markerfacecolor="white",
        markeredgecolor="black",
        linestyle="None",
        markersize=8,
    ),
]

baseline_handle = [
    Line2D(
        [0],
        [0],
        marker=".",
        color="gray",
        label="baseline reference",
        linestyle="None",
        markersize=8,
        alpha=0.5,
    )
]

handles = split_handles + class_handles

if OVERLAY_BASELINE_REFERENCE:
    handles = handles + baseline_handle

# Central legend position.
# Increase y slightly, e.g. 0.53, if it overlaps the central lower panels.
fig.legend(
    handles=handles,
    loc="center",
    bbox_to_anchor=(0.83, 0.45),
    ncol=1,
    fontsize=9,
    frameon=False,
    framealpha=0.92,
    fancybox=False,
    edgecolor="none",
    title="",
    title_fontsize=8,
    borderpad=0.20,
    handletextpad=0.35,
    labelspacing=0.25,
    markerscale=0.90,
)


# fig.suptitle(
#     "",
#     fontsize=8,
#     fontweight="bold",
#     y=0.975,
# )

# plt.subplots_adjust(
#     left=FIG_LEFT,
#     right=FIG_RIGHT,
#     bottom=FIG_BOTTOM,
#     top=FIG_TOP,
#     wspace=SUBPLOT_WSPACE,
#     hspace=SUBPLOT_HSPACE,
# )

plt.subplots_adjust(
    left=0.40,
    right=1.165,
    bottom=0.015,
    top=0.90,
    wspace=0.08,
    hspace=0.05,
)

panel_png = OUT_DIR / "PANEL_train_val_all_branches_projected_on_frozen_baseline_3D_PCA_SSL_LAST_HIGHRES.png"
panel_pdf = OUT_DIR / "PANEL_train_val_all_branches_projected_on_frozen_baseline_3D_PCA_SSL_LAST_HIGHRES.pdf"
panel_svg = OUT_DIR / "PANEL_train_val_all_branches_projected_on_frozen_baseline_3D_PCA_SSL_LAST_HIGHRES.svg"

plt.savefig(panel_png, dpi=SAVE_DPI, bbox_inches="tight", pad_inches=0.02)
plt.savefig(panel_pdf, bbox_inches="tight", pad_inches=0.02)
plt.savefig(panel_svg, bbox_inches="tight", pad_inches=0.02)
plt.show()

print(f"Saved high-resolution 3D panel PNG: {panel_png}")
print(f"Saved high-resolution 3D panel PDF: {panel_pdf}")
print(f"Saved high-resolution 3D panel SVG: {panel_svg}")

# ============================================================
# 12. Save individual high-resolution 3D plots
# ============================================================

SAVE_INDIVIDUAL_3D_BRANCH_PLOTS = True

if SAVE_INDIVIDUAL_3D_BRANCH_PLOTS:
    for branch_name in branch_names_to_plot:
        rec = projected_records[branch_name]
        Z = rec["Z"]
        split_all = rec["split"]
        label_all = rec["label"]

        fig_ind = plt.figure(figsize=(4.5, 4.5), constrained_layout=False)
        ax = fig_ind.add_subplot(111, projection="3d")

        if OVERLAY_BASELINE_REFERENCE and branch_name != BASELINE_BRANCH:
            ax.scatter(
                Z_base[:, 0],
                Z_base[:, 1],
                Z_base[:, 2],
                s=8,
                alpha=0.10,
                marker=".",
                color="gray",
                linewidths=0,
                depthshade=False,
            )

        for lab in unique_labels_global:
            for split in SPLITS_TO_USE:
                mask = (label_all == lab) & (split_all == split)

                if mask.sum() == 0:
                    continue

                ax.scatter(
                    Z[mask, 0],
                    Z[mask, 1],
                    Z[mask, 2],
                    s=SPLIT_SIZES.get(split, 40),
                    alpha=SPLIT_ALPHA.get(split, 0.9),
                    marker=SPLIT_MARKERS.get(split, "o"),
                    color=LABEL_TO_COLOR[lab],
                    edgecolors="black" if split == "val" else "none",
                    linewidths=0.45 if split == "val" else 0.0,
                    depthshade=False,
                )

        ax.set_title(
            BRANCH_DISPLAY_NAMES.get(branch_name, branch_name),
            fontsize=8,
            fontweight="bold",
            pad=3,
        )

        ax.set_xlabel(f"PC1 ({pc1_var * 100:.1f}%)", labelpad=3)
        ax.set_ylabel(f"PC2 ({pc2_var * 100:.1f}%)", labelpad=3)
        ax.set_zlabel(f"PC3 ({pc3_var * 100:.1f}%)", labelpad=3)

        ax.set_xlim(FIXED_XLIM)
        ax.set_ylim(FIXED_YLIM)
        ax.set_zlim(FIXED_ZLIM)

        ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)

        ax.xaxis.pane.set_alpha(0.04)
        ax.yaxis.pane.set_alpha(0.04)
        ax.zaxis.pane.set_alpha(0.04)

        ax.grid(True, linestyle="--", linewidth=0.35, alpha=0.22)

        try:
            ax.set_box_aspect((1.0, 1.0, 0.85))
        except Exception:
            pass

        safe_name = branch_name.replace("/", "_").replace(" ", "_")

        ind_png = OUT_DIR / f"{safe_name}_projected_on_frozen_baseline_3D_PCA_HIGHRES.png"
        ind_pdf = OUT_DIR / f"{safe_name}_projected_on_frozen_baseline_3D_PCA_HIGHRES.pdf"
        ind_svg = OUT_DIR / f"{safe_name}_projected_on_frozen_baseline_3D_PCA_HIGHRES.svg"

        plt.subplots_adjust(left=0.02, right=0.99, bottom=0.01, top=0.93)

        plt.savefig(ind_png, dpi=SAVE_DPI, bbox_inches="tight", pad_inches=0.02)
        plt.savefig(ind_pdf, bbox_inches="tight", pad_inches=0.02)
        plt.savefig(ind_svg, bbox_inches="tight", pad_inches=0.02)
        plt.close(fig_ind)

        print(f"Saved individual high-resolution 3D plot: {ind_png}")


# ============================================================
# 13. Save projected 3D PCA scores
# ============================================================

all_projected_scores = []

for branch_name, rec in projected_records.items():
    df = pd.DataFrame({
        "branch": branch_name,
        "split": rec["split"],
        "label": rec["label"],
        "frozen_baseline_PC1": rec["Z"][:, 0],
        "frozen_baseline_PC2": rec["Z"][:, 1],
        "frozen_baseline_PC3": rec["Z"][:, 2],
    })

    safe_name = branch_name.replace("/", "_").replace(" ", "_")

    branch_csv = OUT_DIR / f"{safe_name}_train_val_projected_on_frozen_baseline_3D_PCA_scores.csv"
    df.to_csv(branch_csv, index=False)

    all_projected_scores.append(df)

    print(f"Saved branch 3D PCA scores: {branch_csv}")


combined_df = pd.concat(all_projected_scores, ignore_index=True)

combined_csv = OUT_DIR / "ALL_BRANCHES_train_val_projected_on_frozen_baseline_3D_PCA_scores_SSL_LAST.csv"
combined_df.to_csv(combined_csv, index=False)

print(f"\nSaved combined 3D PCA scores: {combined_csv}")


# ============================================================
# 14. Centroid-distance functions
# ============================================================

def safe_standardise_labels(y):
    if "standardise_label_array" in globals():
        return standardise_label_array(y)

    return np.asarray(y).astype(str)


def prepare_original_embedding_records_for_branch(branch_name, rec, splits=("train", "val")):
    X_all, split_all, label_all = combine_branch_embeddings(
        rec,
        splits=splits,
    )

    if X_all is None:
        return None, None, None

    label_all = safe_standardise_labels(label_all)

    finite_rows = np.isfinite(X_all).all(axis=1)

    X_all = X_all[finite_rows].astype(np.float32)
    split_all = split_all[finite_rows].astype(str)
    label_all = label_all[finite_rows].astype(str)

    return X_all, split_all, label_all


def get_projected_3d_records_for_branch(branch_name, splits=("train", "val")):
    if branch_name not in projected_records:
        return None, None, None

    rec = projected_records[branch_name]

    Z = rec["Z"]
    split_all = rec["split"].astype(str)
    label_all = safe_standardise_labels(rec["label"])

    keep = np.isin(split_all, list(splits))

    Z = Z[keep].astype(np.float32)
    split_all = split_all[keep].astype(str)
    label_all = label_all[keep].astype(str)

    finite_rows = np.isfinite(Z).all(axis=1)

    return Z[finite_rows], split_all[finite_rows], label_all[finite_rows]


def compute_class_centroids(X, labels, branch_name, split_scope_name, space_name):
    labels = np.asarray(labels).astype(str)
    X = np.asarray(X, dtype=np.float32)

    rows_centroid = []
    rows_dispersion = []

    for lab in sorted(np.unique(labels)):
        mask = labels == lab
        n = int(mask.sum())

        if n < MIN_CLASS_N_FOR_CENTROID:
            print(
                f"[centroid skip] {branch_name} | {split_scope_name} | "
                f"{space_name} | {lab}: n={n} < {MIN_CLASS_N_FOR_CENTROID}"
            )
            continue

        Xc = X[mask]
        centroid = Xc.mean(axis=0)

        d_to_centroid = np.linalg.norm(Xc - centroid[None, :], axis=1)

        rows_centroid.append({
            "branch": branch_name,
            "split_scope": split_scope_name,
            "space": space_name,
            "class_name": lab,
            "n": n,
            **{
                f"centroid_dim_{i:04d}": float(v)
                for i, v in enumerate(centroid)
            },
        })

        rows_dispersion.append({
            "branch": branch_name,
            "split_scope": split_scope_name,
            "space": space_name,
            "class_name": lab,
            "n": n,
            "mean_distance_to_centroid": float(np.mean(d_to_centroid)),
            "median_distance_to_centroid": float(np.median(d_to_centroid)),
            "std_distance_to_centroid": float(np.std(d_to_centroid)),
            "max_distance_to_centroid": float(np.max(d_to_centroid)),
        })

    df_centroids = pd.DataFrame(rows_centroid)
    df_dispersion = pd.DataFrame(rows_dispersion)

    return df_centroids, df_dispersion


def centroid_matrix_from_df(df_centroids):
    if df_centroids.empty:
        return np.zeros((0, 0), dtype=np.float32), []

    centroid_cols = [
        c for c in df_centroids.columns
        if c.startswith("centroid_dim_")
    ]

    centroid_cols = sorted(centroid_cols)

    classes = df_centroids["class_name"].astype(str).values.tolist()
    C = df_centroids[centroid_cols].values.astype(np.float32)

    return C, classes


def compute_pairwise_centroid_distances(
    df_centroids,
    branch_name,
    split_scope_name,
    space_name,
    metric="euclidean",
):
    C, classes = centroid_matrix_from_df(df_centroids)

    if C.shape[0] < 2:
        return pd.DataFrame(), pd.DataFrame()

    D = cdist(C, C, metric=metric)

    rows_long = []

    for i, j in combinations(range(len(classes)), 2):
        rows_long.append({
            "branch": branch_name,
            "split_scope": split_scope_name,
            "space": space_name,
            "metric": metric,
            "class_a": classes[i],
            "class_b": classes[j],
            "centroid_distance": float(D[i, j]),
        })

    df_long = pd.DataFrame(rows_long)

    df_square = pd.DataFrame(
        D,
        index=classes,
        columns=classes,
    )

    df_square.insert(0, "class_name", classes)
    df_square.insert(0, "metric", metric)
    df_square.insert(0, "space", space_name)
    df_square.insert(0, "split_scope", split_scope_name)
    df_square.insert(0, "branch", branch_name)

    return df_long, df_square


def compute_nearest_centroid_summary(df_pairwise_long):
    if df_pairwise_long.empty:
        return pd.DataFrame()

    rows = []

    for branch in sorted(df_pairwise_long["branch"].unique()):
        df_b = df_pairwise_long[df_pairwise_long["branch"] == branch]

        for split_scope in sorted(df_b["split_scope"].unique()):
            df_bs = df_b[df_b["split_scope"] == split_scope]

            for space in sorted(df_bs["space"].unique()):
                df_bss = df_bs[df_bs["space"] == space]

                for metric in sorted(df_bss["metric"].unique()):
                    df_m = df_bss[df_bss["metric"] == metric]

                    classes = sorted(
                        set(df_m["class_a"].astype(str).tolist()) |
                        set(df_m["class_b"].astype(str).tolist())
                    )

                    for cls in classes:
                        d_cls = df_m[
                            (df_m["class_a"] == cls) |
                            (df_m["class_b"] == cls)
                        ].copy()

                        if len(d_cls) == 0:
                            continue

                        other_class = []

                        for _, r in d_cls.iterrows():
                            if r["class_a"] == cls:
                                other_class.append(r["class_b"])
                            else:
                                other_class.append(r["class_a"])

                        d_cls["other_class"] = other_class

                        nearest = d_cls.loc[d_cls["centroid_distance"].idxmin()]
                        farthest = d_cls.loc[d_cls["centroid_distance"].idxmax()]

                        rows.append({
                            "branch": branch,
                            "split_scope": split_scope,
                            "space": space,
                            "metric": metric,
                            "class_name": cls,
                            "nearest_class": nearest["other_class"],
                            "nearest_centroid_distance": float(nearest["centroid_distance"]),
                            "farthest_class": farthest["other_class"],
                            "farthest_centroid_distance": float(farthest["centroid_distance"]),
                            "mean_distance_to_other_centroids": float(d_cls["centroid_distance"].mean()),
                        })

    return pd.DataFrame(rows)


def plot_centroid_distance_heatmap(
    df_square,
    out_png,
    title,
    cmap_name="Greys",
    mask_diagonal=True,
    annotate=True,
):
    """
    Grayscale centroid-distance matrix for publication readability.

    - Uses Greys colour map.
    - Diagonal is masked and shown as white.
    - Larger distances are darker.
    - Diagonal cells are marked with "—".
    """
    if df_square.empty:
        return

    meta_cols = ["branch", "split_scope", "space", "metric", "class_name"]
    class_names = df_square["class_name"].astype(str).tolist()

    D = df_square.drop(columns=meta_cols).values.astype(float)

    D_plot = D.copy()

    if mask_diagonal:
        np.fill_diagonal(D_plot, np.nan)

    off_diag_values = D_plot[~np.isnan(D_plot)]

    if len(off_diag_values) > 0:
        vmin = float(np.nanmin(off_diag_values))
        vmax = float(np.nanmax(off_diag_values))
    else:
        vmin, vmax = 0.0, 1.0

    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-6

    fig, ax = plt.subplots(figsize=(6.8, 5.8))

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color="white")

    im = ax.imshow(
        D_plot,
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))

    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)

    ax.set_xticks(np.arange(-0.5, len(class_names), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(class_names), 1), minor=True)

    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=1.6,
    )

    ax.tick_params(which="minor", bottom=False, left=False)

    if annotate:
        for i in range(D.shape[0]):
            for j in range(D.shape[1]):

                if i == j:
                    txt = "—"
                    colour = "black"
                    weight = "bold"
                else:
                    txt = f"{D[i, j]:.2f}"
                    norm_val = (D[i, j] - vmin) / (vmax - vmin + 1e-12)
                    colour = "white" if norm_val > 0.55 else "black"
                    weight = "normal"

                ax.text(
                    j,
                    i,
                    txt,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=colour,
                    fontweight=weight,
                )

    ax.set_title(
        title,
        fontsize=11,
        fontweight="bold",
        pad=10,
    )

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    metric = str(df_square["metric"].iloc[0]).lower()

    if metric == "cosine":
        cbar.set_label("Cosine centroid distance", fontsize=9)
    else:
        cbar.set_label("Euclidean centroid distance", fontsize=9)

    cbar.ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)


# ============================================================
# 15. Run centroid-distance analysis
# ============================================================

all_centroids_embedding = []
all_centroids_pca3d = []
all_dispersion_embedding = []
all_dispersion_pca3d = []
all_pairwise_long = []
all_pairwise_square = []

for branch_name, rec in branch_records.items():
    if branch_name not in projected_records:
        print(f"[skip centroid] {branch_name}: not present in projected_records")
        continue

    safe_branch = branch_name.replace("/", "_").replace(" ", "_")

    for split_scope_name, split_scope in CENTROID_SPLIT_SCOPES.items():
        print("\n" + "-" * 90)
        print(f"Centroid analysis: {branch_name} | {split_scope_name}")
        print("-" * 90)

        # ----------------------------------------------------
        # Original embedding space
        # ----------------------------------------------------
        X_emb, split_emb, label_emb = prepare_original_embedding_records_for_branch(
            branch_name,
            rec,
            splits=split_scope,
        )

        if X_emb is not None and len(X_emb) > 0:
            label_emb = safe_standardise_labels(label_emb)

            if L2_NORMALISE_BEFORE_PCA:
                X_for_centroid = l2_normalize(X_emb)
                emb_space_name = "embedding_l2_normalised"
            else:
                X_for_centroid = X_emb.astype(np.float32)
                emb_space_name = "embedding_raw"

            df_cent_emb, df_disp_emb = compute_class_centroids(
                X_for_centroid,
                label_emb,
                branch_name=branch_name,
                split_scope_name=split_scope_name,
                space_name=emb_space_name,
            )

            all_centroids_embedding.append(df_cent_emb)
            all_dispersion_embedding.append(df_disp_emb)

            for metric in EMBEDDING_DISTANCE_METRICS:
                df_long, df_square = compute_pairwise_centroid_distances(
                    df_cent_emb,
                    branch_name=branch_name,
                    split_scope_name=split_scope_name,
                    space_name=emb_space_name,
                    metric=metric,
                )

                all_pairwise_long.append(df_long)
                all_pairwise_square.append(df_square)

                if not df_square.empty:
                    square_csv = (
                        CENTROID_OUT_DIR /
                        f"{safe_branch}_{split_scope_name}_embedding_centroid_distance_matrix_{metric}.csv"
                    )
                    df_square.to_csv(square_csv, index=False)

                    heatmap_png = (
                        CENTROID_OUT_DIR /
                        f"{safe_branch}_{split_scope_name}_embedding_centroid_distance_heatmap_{metric}_GREY.png"
                    )

                    plot_centroid_distance_heatmap(
                        df_square,
                        heatmap_png,
                        title=f"{BRANCH_DISPLAY_NAMES.get(branch_name, branch_name)}\n"
                              f"{split_scope_name} centroid distances | embedding | {metric}",
                        cmap_name="Greys",
                    )

        # ----------------------------------------------------
        # Frozen-baseline 3D PCA space
        # ----------------------------------------------------
        Z_pca, split_pca, label_pca = get_projected_3d_records_for_branch(
            branch_name,
            splits=split_scope,
        )

        if Z_pca is not None and len(Z_pca) > 0:
            label_pca = safe_standardise_labels(label_pca)

            df_cent_pca, df_disp_pca = compute_class_centroids(
                Z_pca,
                label_pca,
                branch_name=branch_name,
                split_scope_name=split_scope_name,
                space_name="frozen_baseline_3D_PCA",
            )

            all_centroids_pca3d.append(df_cent_pca)
            all_dispersion_pca3d.append(df_disp_pca)

            df_long, df_square = compute_pairwise_centroid_distances(
                df_cent_pca,
                branch_name=branch_name,
                split_scope_name=split_scope_name,
                space_name="frozen_baseline_3D_PCA",
                metric="euclidean",
            )

            all_pairwise_long.append(df_long)
            all_pairwise_square.append(df_square)

            if not df_square.empty:
                square_csv = (
                    CENTROID_OUT_DIR /
                    f"{safe_branch}_{split_scope_name}_3D_PCA_centroid_distance_matrix_euclidean.csv"
                )
                df_square.to_csv(square_csv, index=False)

                heatmap_png = (
                    CENTROID_OUT_DIR /
                    f"{safe_branch}_{split_scope_name}_3D_PCA_centroid_distance_heatmap_euclidean_GREY.png"
                )

                plot_centroid_distance_heatmap(
                    df_square,
                    heatmap_png,
                    title=f"{BRANCH_DISPLAY_NAMES.get(branch_name, branch_name)}\n"
                          f"{split_scope_name} centroid distances | frozen 3D PCA",
                    cmap_name="Greys",
                )


# ============================================================
# 16. Save combined centroid outputs
# ============================================================

def concat_or_empty(dfs):
    dfs = [df for df in dfs if df is not None and len(df) > 0]

    if len(dfs) == 0:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


df_centroids_embedding_all = concat_or_empty(all_centroids_embedding)
df_centroids_pca3d_all = concat_or_empty(all_centroids_pca3d)

df_dispersion_embedding_all = concat_or_empty(all_dispersion_embedding)
df_dispersion_pca3d_all = concat_or_empty(all_dispersion_pca3d)

df_pairwise_long_all = concat_or_empty(all_pairwise_long)
df_pairwise_square_all = concat_or_empty(all_pairwise_square)

df_centroid_summary = compute_nearest_centroid_summary(df_pairwise_long_all)

centroids_embedding_csv = CENTROID_OUT_DIR / "ALL_BRANCHES_class_centroids_embedding_space.csv"
centroids_pca_csv = CENTROID_OUT_DIR / "ALL_BRANCHES_class_centroids_frozen_baseline_3D_PCA_space.csv"

dispersion_embedding_csv = CENTROID_OUT_DIR / "ALL_BRANCHES_class_dispersion_embedding_space.csv"
dispersion_pca_csv = CENTROID_OUT_DIR / "ALL_BRANCHES_class_dispersion_frozen_baseline_3D_PCA_space.csv"

pairwise_long_csv = CENTROID_OUT_DIR / "ALL_BRANCHES_pairwise_class_centroid_distances_long.csv"
pairwise_square_csv = CENTROID_OUT_DIR / "ALL_BRANCHES_pairwise_class_centroid_distances_square.csv"

summary_csv = CENTROID_OUT_DIR / "ALL_BRANCHES_centroid_distance_summary.csv"

df_centroids_embedding_all.to_csv(centroids_embedding_csv, index=False)
df_centroids_pca3d_all.to_csv(centroids_pca_csv, index=False)

df_dispersion_embedding_all.to_csv(dispersion_embedding_csv, index=False)
df_dispersion_pca3d_all.to_csv(dispersion_pca_csv, index=False)

df_pairwise_long_all.to_csv(pairwise_long_csv, index=False)
df_pairwise_square_all.to_csv(pairwise_square_csv, index=False)

df_centroid_summary.to_csv(summary_csv, index=False)

print("\nSaved centroid outputs:")
print("  embedding centroids :", centroids_embedding_csv)
print("  3D PCA centroids    :", centroids_pca_csv)
print("  embedding dispersion:", dispersion_embedding_csv)
print("  3D PCA dispersion   :", dispersion_pca_csv)
print("  pairwise long       :", pairwise_long_csv)
print("  pairwise square     :", pairwise_square_csv)
print("  summary             :", summary_csv)


# ============================================================
# 17. Save manifest
# ============================================================

manifest = {
    "pca_reference": "A_frozen_dino_baseline train + val",
    "pca_components": 3,
    "baseline_branch": BASELINE_BRANCH,
    "baseline_splits_for_pca": list(BASELINE_SPLITS_FOR_PCA),
    "splits_plotted": list(SPLITS_TO_USE),
    "l2_normalise_before_pca": bool(L2_NORMALISE_BEFORE_PCA),
    "pc1_variance_ratio": float(pc1_var),
    "pc2_variance_ratio": float(pc2_var),
    "pc3_variance_ratio": float(pc3_var),
    "pc1_pc2_pc3_cumulative_variance_ratio": float(pc1_var + pc2_var + pc3_var),
    "branches_projected": list(projected_records.keys()),
    "class_order": CLASS_ORDER,
    "class_colour_map": {str(k): str(v) for k, v in LABEL_TO_COLOR.items()},
    "fixed_xlim": [float(FIXED_XLIM[0]), float(FIXED_XLIM[1])],
    "fixed_ylim": [float(FIXED_YLIM[0]), float(FIXED_YLIM[1])],
    "fixed_zlim": [float(FIXED_ZLIM[0]), float(FIXED_ZLIM[1])],
    "plot_settings": {
        "save_dpi": SAVE_DPI,
        "figsize_panel": list(FIGSIZE_PANEL),
        "subplot_wspace": SUBPLOT_WSPACE,
        "subplot_hspace": SUBPLOT_HSPACE,
        "centroid_heatmap_cmap": "Greys",
    },
    "outputs": {
        "panel_png": str(panel_png),
        "panel_pdf": str(panel_pdf),
        "panel_svg": str(panel_svg),
        "combined_pca_scores_csv": str(combined_csv),
        "centroid_summary_csv": str(summary_csv),
        "pairwise_centroid_distances_csv": str(pairwise_long_csv),
    },
}

manifest_json = OUT_DIR / "PANEL_train_val_all_branches_projected_on_frozen_baseline_3D_PCA_HIGHRES_manifest.json"

with open(manifest_json, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved manifest: {manifest_json}")

if len(df_centroid_summary) > 0:
    display_cols = [
        "branch",
        "split_scope",
        "space",
        "metric",
        "class_name",
        "nearest_class",
        "nearest_centroid_distance",
        "farthest_class",
        "farthest_centroid_distance",
    ]

    print("\nNearest/farthest centroid summary:")
    print(df_centroid_summary[display_cols].head(60).to_string(index=False))

print("\nAll high-resolution 3D PCA and centroid-distance outputs saved to:")
print(OUT_DIR)
print("\nDone.")

In [ ]:
# ============================================================
# Standalone script:
# External cohort centroid distance against train+validation cohort
#
# Purpose:
#   1. Load SSL-last train, validation and external embeddings.
#   2. Fit frozen-DINO 3D PCA using frozen baseline train+val only.
#   3. Project SSL train/val/external into the same frozen-baseline 3D PCA space.
#   4. Compute centroid distances:
#        external class centroid  ->  internal train+val class centroids
#   5. Save grayscale centroid-distance matrices.
#
# Distances computed in:
#   A. Original L2-normalised embedding space
#   B. Frozen-baseline 3D PCA space
#
# Outputs:
#   OUT_DIR/
#       SSL_LAST_train_val_external_PROJECTED_ON_FROZEN_BASELINE_3D_PCA.png
#       SSL_LAST_train_val_external_PROJECTED_ON_FROZEN_BASELINE_3D_PCA_scores.csv
#
#   OUT_DIR/external_vs_trainval_centroid_distance/
#       external_vs_trainval_centroid_distances_embedding_euclidean.csv
#       external_vs_trainval_centroid_distances_embedding_cosine.csv
#       external_vs_trainval_centroid_distances_3D_PCA_euclidean.csv
#       external_vs_trainval_centroid_summary.csv
#       *_GREY.png
# ============================================================

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


# ============================================================
# 0. User settings
# ============================================================

ROOT = Path("/mnt/d/data/patient_dinov2_ssl_outputs_branches_6")

FROZEN_BASELINE_DIR = ROOT / "artifacts" / "branches" / "frozen_dino_baseline"
SSL_LAST_DIR = ROOT / "artifacts" / "last"
EXTERNAL_DIR = ROOT / "artifacts" / "external_lung_melanoma_nii_ssl_last"

OUT_DIR = ROOT / "artifacts" / "ssl_last_external_vs_trainval_centroid_distance_frozen_3d_pca"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CENTROID_OUT_DIR = OUT_DIR / "external_vs_trainval_centroid_distance"
CENTROID_OUT_DIR.mkdir(parents=True, exist_ok=True)

# External classes to include.
# Examples:
#   ["melanoma"]          -> external melanoma only
#   ["lung"]              -> external lung only
#   ["lung", "melanoma"]  -> both
EXTERNAL_CLASSES_TO_KEEP = ["melanoma"]

# L2-normalise before PCA and before embedding-space centroid analysis.
L2_NORMALISE_BEFORE_PCA = True

# Minimum number of samples required for a class centroid.
# Set to 1 if you want centroid distance even for a single external case.
MIN_CLASS_N_FOR_CENTROID = 1

# Plot settings
FIGSIZE_3D = (4.5, 4.5)
SAVE_DPI = 600

SPLIT_MARKERS = {
    "train": "o",
    "val": "^",
    "external": "X",
}

SPLIT_SIZES = {
    "train": 32,
    "val": 68,
    "external": 125,
}

SPLIT_ALPHA = {
    "train": 0.65,
    "val": 0.95,
    "external": 1.00,
}

CLASS_ORDER = [
    "oesophagus",
    "melanoma",
    "lymphoma",
    "lung",
]

FIXED_CLASS_COLOURS = {
    "oesophagus": "pink",
    "melanoma": "purple",
    "lymphoma": "tab:blue",
    "lung": "green",
}


# ============================================================
# 1. Expected file names
# ============================================================

FROZEN_TRAIN_EMB_PATHS = [
    FROZEN_BASELINE_DIR / "train_embeddings.npy",
    FROZEN_BASELINE_DIR / "training_embeddings.npy",
    FROZEN_BASELINE_DIR / "embeddings_train.npy",
    FROZEN_BASELINE_DIR / "train_features.npy",
]

FROZEN_VAL_EMB_PATHS = [
    FROZEN_BASELINE_DIR / "val_embeddings.npy",
    FROZEN_BASELINE_DIR / "validation_embeddings.npy",
    FROZEN_BASELINE_DIR / "embeddings_val.npy",
    FROZEN_BASELINE_DIR / "val_features.npy",
]

SSL_TRAIN_EMB_PATHS = [
    SSL_LAST_DIR / "train_embeddings.npy",
    SSL_LAST_DIR / "training_embeddings.npy",
    SSL_LAST_DIR / "embeddings_train.npy",
]

SSL_VAL_EMB_PATHS = [
    SSL_LAST_DIR / "val_embeddings.npy",
    SSL_LAST_DIR / "validation_embeddings.npy",
    SSL_LAST_DIR / "embeddings_val.npy",
]

SSL_TRAIN_METADATA_PATHS = [
    SSL_LAST_DIR / "train_embeddings_metadata.csv",
    SSL_LAST_DIR / "train_metadata.csv",
    SSL_LAST_DIR / "train_predictions.csv",
]

SSL_VAL_METADATA_PATHS = [
    SSL_LAST_DIR / "val_embeddings_metadata.csv",
    SSL_LAST_DIR / "val_metadata.csv",
    SSL_LAST_DIR / "val_predictions.csv",
]

SSL_TRAIN_LABEL_PATHS = [
    SSL_LAST_DIR / "train_labels.npy",
    SSL_LAST_DIR / "train_targets.npy",
    SSL_LAST_DIR / "train_classes.npy",
    SSL_LAST_DIR / "train_y.npy",
]

SSL_VAL_LABEL_PATHS = [
    SSL_LAST_DIR / "val_labels.npy",
    SSL_LAST_DIR / "val_targets.npy",
    SSL_LAST_DIR / "val_classes.npy",
    SSL_LAST_DIR / "val_y.npy",
]

EXTERNAL_EMB_PATHS = [
    EXTERNAL_DIR / "external_lung_melanoma_nii_embeddings.npy",
    EXTERNAL_DIR / "external_lung_melanoma_embeddings.npy",
    EXTERNAL_DIR / "external_embeddings.npy",
]

EXTERNAL_METADATA_PATHS = [
    EXTERNAL_DIR / "external_lung_melanoma_nii_metadata.csv",
    EXTERNAL_DIR / "external_lung_melanoma_metadata.csv",
    EXTERNAL_DIR / "external_lung_melanoma_nii_embeddings.csv",
    EXTERNAL_DIR / "external_lung_melanoma_embeddings.csv",
]

PREFERRED_LABEL_COLUMNS = [
    "class_name",
    "class",
    "label",
    "target",
    "Clinical_Stage",
    "Treatment_Group",
    "Sex",
    "Response",
]

PREFERRED_ID_COLUMNS = [
    "patient_id",
    "Patient_ID",
    "PatientID",
    "id",
    "ids",
]

PREFERRED_PATH_COLUMNS = [
    "path",
    "file_name",
    "filepath",
    "file_path",
]


# ============================================================
# 2. Helper functions
# ============================================================

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def load_embedding_array(path):
    path = Path(path)
    X = np.load(path, allow_pickle=True)
    X = np.asarray(X)

    if X.dtype == object:
        X = np.vstack(X)

    if X.ndim == 2:
        X = X.astype(np.float32)

    elif X.ndim == 3:
        print(f"[info] {path}: 3D embeddings {X.shape}; mean-pooling axis=1.")
        X = X.mean(axis=1).astype(np.float32)

    elif X.ndim > 3:
        print(f"[info] {path}: {X.ndim}D embeddings {X.shape}; flattening to N x D.")
        X = X.reshape(X.shape[0], -1).astype(np.float32)

    else:
        raise ValueError(f"Unsupported embedding shape for {path}: {X.shape}")

    return X


def load_1d_npy(path, n_expected=None):
    path = Path(path)
    arr = np.load(path, allow_pickle=True)
    arr = np.asarray(arr)

    if arr.ndim > 1:
        arr = arr.reshape(-1)

    arr = pd.Series(arr).astype(str).str.strip().values

    if n_expected is not None and len(arr) != n_expected:
        print(f"[warning] length mismatch for {path}: {len(arr)} vs expected {n_expected}")
        return None

    return arr


def load_metadata_csv(paths, n_expected=None):
    for p in paths:
        p = Path(p)

        if not p.exists():
            continue

        try:
            df = pd.read_csv(p)
        except Exception as e:
            print(f"[warning] could not read metadata CSV {p}: {e}")
            continue

        if n_expected is not None and len(df) != n_expected:
            print(f"[warning] metadata length mismatch for {p}: {len(df)} vs expected {n_expected}")
            continue

        return df, p

    return None, None


def extract_labels_from_metadata(df, n_expected):
    if df is None:
        return None, None

    for col in PREFERRED_LABEL_COLUMNS:
        if col in df.columns:
            y = df[col].astype(str).str.strip().values

            if len(y) == n_expected:
                return y, col

    return None, None


def extract_ids_from_metadata(df, n_expected, default_prefix):
    if df is not None:
        for col in PREFERRED_ID_COLUMNS:
            if col in df.columns:
                ids = df[col].astype(str).str.strip().values

                if len(ids) == n_expected:
                    return ids, col

    ids = np.array([f"{default_prefix}_{i}" for i in range(n_expected)])
    return ids, None


def extract_paths_from_metadata(df, n_expected):
    if df is not None:
        for col in PREFERRED_PATH_COLUMNS:
            if col in df.columns:
                paths = df[col].astype(str).str.strip().values

                if len(paths) == n_expected:
                    return paths, col

    paths = np.array([""] * n_expected)
    return paths, None


def l2_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)


def clean_label_string(x):
    xs = str(x).strip()
    xl = xs.lower()

    xl = re.sub(r"\.nii(\.gz)?$", "", xl)

    xl = (
        xl.replace("-", " ")
          .replace("_", " ")
          .replace("/", " ")
          .replace("\\", " ")
          .replace(".", " ")
          .strip()
    )

    xl = re.sub(r"\s+", " ", xl)

    return xl


def standardise_class_labels(labels):
    out = []

    for x in labels:
        xs = str(x).strip()
        xl = clean_label_string(xs)
        tokens = xl.split()

        if xl in ["0", "0 0", "0.0"]:
            out.append("oesophagus")
            continue

        if xl in ["1", "1 0", "1.0"]:
            out.append("melanoma")
            continue

        if xl in ["2", "2 0", "2.0"]:
            out.append("lymphoma")
            continue

        if xl in ["3", "3 0", "3.0"]:
            out.append("lung")
            continue

        if (
            xl in [
                "oc",
                "oac",
                "oesophagus",
                "esophagus",
                "oesophageal",
                "esophageal",
                "oesophageal cancer",
                "esophageal cancer",
                "oesoph",
                "esoph",
            ]
            or "oesoph" in xl
            or "esoph" in xl
            or "oac" in xl
        ):
            out.append("oesophagus")
            continue

        if (
            xl in [
                "mela",
                "mel",
                "melanoma",
                "melanoma cancer",
                "mela cancer",
                "mel cancer",
                "external melanoma",
                "external mela",
                "external mel",
            ]
            or any(tok.startswith("mela") for tok in tokens)
            or any(tok.startswith("melr") for tok in tokens)
            or "melanoma" in xl
            or "mela" in xl
        ):
            out.append("melanoma")
            continue

        if (
            xl in [
                "lung",
                "lung cancer",
                "external lung",
                "nsclc",
                "non small cell lung cancer",
                "non-small cell lung cancer",
            ]
            or any(tok.startswith("lung") for tok in tokens)
            or "lung" in xl
        ):
            out.append("lung")
            continue

        if (
            xl in [
                "lym",
                "lymphoma",
                "external lymphoma",
                "lymphoma cancer",
            ]
            or any(tok.startswith("lym") for tok in tokens)
            or "lymphoma" in xl
        ):
            out.append("lymphoma")
            continue

        out.append(xs)

    return np.array(out).astype(str)


def infer_label_from_path_or_patient_id(df):
    df = df.copy()

    candidate_cols = []
    for col in ["class_name", "label", "patient_id", "file_name", "path"]:
        if col in df.columns:
            candidate_cols.append(col)

    inferred_strings = []

    for _, row in df.iterrows():
        joined = " ".join([str(row.get(c, "")) for c in candidate_cols])
        inferred_strings.append(joined)

    inferred = standardise_class_labels(inferred_strings)

    final = []

    for current, inf in zip(df["class_name"].astype(str).values, inferred):
        cur_std = standardise_class_labels([current])[0]

        if cur_std in CLASS_ORDER:
            final.append(cur_std)

        elif inf in CLASS_ORDER:
            final.append(inf)

        else:
            final.append(cur_std)

    df["class_name"] = np.array(final).astype(str)
    df["label"] = df["class_name"].values

    return df


# ============================================================
# 3. Loading functions
# ============================================================

def load_ssl_split(split):
    if split == "train":
        emb_path = first_existing(SSL_TRAIN_EMB_PATHS)
        meta_paths = SSL_TRAIN_METADATA_PATHS
        label_paths = SSL_TRAIN_LABEL_PATHS

    elif split == "val":
        emb_path = first_existing(SSL_VAL_EMB_PATHS)
        meta_paths = SSL_VAL_METADATA_PATHS
        label_paths = SSL_VAL_LABEL_PATHS

    else:
        raise ValueError(f"Unsupported split: {split}")

    if emb_path is None:
        raise FileNotFoundError(f"Could not find SSL-last {split} embeddings in {SSL_LAST_DIR}")

    X = load_embedding_array(emb_path)
    n = len(X)

    df_meta, meta_path = load_metadata_csv(meta_paths, n_expected=n)

    labels, label_source = extract_labels_from_metadata(df_meta, n)

    if labels is None:
        label_path = first_existing(label_paths)

        if label_path is not None:
            labels = load_1d_npy(label_path, n_expected=n)
            label_source = str(label_path)

    if labels is None:
        labels = np.array(["Unknown"] * n)
        label_source = None

    labels = standardise_class_labels(labels)

    patient_ids, id_source = extract_ids_from_metadata(
        df_meta,
        n_expected=n,
        default_prefix=split,
    )

    paths, path_source = extract_paths_from_metadata(
        df_meta,
        n_expected=n,
    )

    df = pd.DataFrame({
        "split": split,
        "patient_id": patient_ids,
        "path": paths,
        "label": labels,
        "class_name": labels,
        "source": "ssl_last_internal",
    })

    df = infer_label_from_path_or_patient_id(df)

    print(f"\n[loaded SSL {split}]")
    print(f"  embeddings : {X.shape} | {emb_path}")
    print(f"  metadata   : {meta_path}")
    print(f"  label src  : {label_source}")
    print(f"  id src     : {id_source}")
    print(f"  path src   : {path_source}")
    print("  class counts:")
    print(df["class_name"].value_counts())

    return df, X


def load_external_lung_mel():
    emb_path = first_existing(EXTERNAL_EMB_PATHS)

    if emb_path is None:
        raise FileNotFoundError(
            f"Could not find external lung/melanoma embeddings in {EXTERNAL_DIR}. "
            "Run the external NIfTI SSL-last embedding export first."
        )

    X = load_embedding_array(emb_path)
    n = len(X)

    df_meta, meta_path = load_metadata_csv(EXTERNAL_METADATA_PATHS, n_expected=n)

    if df_meta is None:
        raise FileNotFoundError(
            f"Could not find external metadata CSV in {EXTERNAL_DIR}."
        )

    labels, label_source = extract_labels_from_metadata(df_meta, n)

    if labels is None:
        labels = np.array(["external"] * n)
        label_source = None

    labels = standardise_class_labels(labels)

    patient_ids, id_source = extract_ids_from_metadata(
        df_meta,
        n_expected=n,
        default_prefix="external",
    )

    paths, path_source = extract_paths_from_metadata(
        df_meta,
        n_expected=n,
    )

    if "file_name" in df_meta.columns:
        file_names = df_meta["file_name"].astype(str).str.strip().values
    else:
        file_names = np.array([Path(p).name for p in paths])

    df = pd.DataFrame({
        "split": "external",
        "patient_id": patient_ids,
        "file_name": file_names,
        "path": paths,
        "label": labels,
        "class_name": labels,
        "source": "external_lung_melanoma",
    })

    df = infer_label_from_path_or_patient_id(df)

    before_n = len(df)

    keep_classes = [str(c).lower().strip() for c in EXTERNAL_CLASSES_TO_KEEP]
    keep_mask = df["class_name"].astype(str).str.lower().str.strip().isin(keep_classes)

    df = df.loc[keep_mask].reset_index(drop=True)
    X = X[keep_mask.values]

    after_n = len(df)

    print("\n[loaded external cohort]")
    print(f"  embeddings : {X.shape} | {emb_path}")
    print(f"  metadata   : {meta_path}")
    print(f"  label src  : {label_source}")
    print(f"  id src     : {id_source}")
    print(f"  path src   : {path_source}")
    print(f"  external classes kept: {EXTERNAL_CLASSES_TO_KEEP}")
    print(f"  skipped external cases: {before_n - after_n}")
    print("  class counts after filtering:")
    print(df["class_name"].value_counts())

    if len(df) == 0:
        raise RuntimeError(
            "No external cases remain after filtering. "
            f"Requested EXTERNAL_CLASSES_TO_KEEP={EXTERNAL_CLASSES_TO_KEEP}. "
            "Check class labels in the external metadata."
        )

    return df, X


def load_frozen_baseline_for_pca():
    frozen_train_path = first_existing(FROZEN_TRAIN_EMB_PATHS)
    frozen_val_path = first_existing(FROZEN_VAL_EMB_PATHS)

    if frozen_train_path is None or frozen_val_path is None:
        raise FileNotFoundError(
            "Could not find frozen baseline train/val embeddings. Expected under:\n"
            f"{FROZEN_BASELINE_DIR}"
        )

    X_frozen_train = load_embedding_array(frozen_train_path)
    X_frozen_val = load_embedding_array(frozen_val_path)

    print("\n[loaded frozen baseline for PCA]")
    print("  train:", X_frozen_train.shape, "|", frozen_train_path)
    print("  val  :", X_frozen_val.shape, "|", frozen_val_path)

    return X_frozen_train, X_frozen_val, frozen_train_path, frozen_val_path


# ============================================================
# 4. Load data
# ============================================================

print("ROOT:", ROOT)
print("FROZEN_BASELINE_DIR:", FROZEN_BASELINE_DIR)
print("SSL_LAST_DIR:", SSL_LAST_DIR)
print("EXTERNAL_DIR:", EXTERNAL_DIR)
print("OUT_DIR:", OUT_DIR)

df_train, X_train = load_ssl_split("train")
df_val, X_val = load_ssl_split("val")
df_ext, X_ext = load_external_lung_mel()

X_frozen_train, X_frozen_val, frozen_train_path, frozen_val_path = load_frozen_baseline_for_pca()


# ============================================================
# 5. Dimension check
# ============================================================

dims = {
    "frozen_train": X_frozen_train.shape[1],
    "frozen_val": X_frozen_val.shape[1],
    "ssl_train": X_train.shape[1],
    "ssl_val": X_val.shape[1],
    "external": X_ext.shape[1],
}

print("\nEmbedding dimensions:")
for k, v in dims.items():
    print(f"  {k}: {v}")

if len(set(dims.values())) != 1:
    raise ValueError(
        f"Embedding dimensions do not match: {dims}. "
        "Frozen baseline, SSL-last and external embeddings must have the same dimensionality."
    )


# ============================================================
# 6. Fit frozen-baseline 3D PCA
# ============================================================

X_frozen_for_pca = np.vstack([X_frozen_train, X_frozen_val]).astype(np.float32)

if X_frozen_for_pca.shape[0] < 4:
    raise RuntimeError(
        f"Need at least 4 frozen-baseline samples for 3D PCA. "
        f"Got {X_frozen_for_pca.shape[0]}."
    )

if L2_NORMALISE_BEFORE_PCA:
    X_frozen_pca = l2_normalize(X_frozen_for_pca)
else:
    X_frozen_pca = X_frozen_for_pca.copy()

frozen_baseline_mean = X_frozen_pca.mean(axis=0, keepdims=True)
X_frozen_centered = X_frozen_pca - frozen_baseline_mean

fixed_pca = PCA(n_components=3)
Z_frozen_reference = fixed_pca.fit_transform(X_frozen_centered)

pc1_var = fixed_pca.explained_variance_ratio_[0]
pc2_var = fixed_pca.explained_variance_ratio_[1]
pc3_var = fixed_pca.explained_variance_ratio_[2]

print("\n3D PCA fitted on frozen DINO baseline train + val only")
print("  frozen PCA matrix:", X_frozen_for_pca.shape)
print(f"  PC1 variance     : {pc1_var:.4f}")
print(f"  PC2 variance     : {pc2_var:.4f}")
print(f"  PC3 variance     : {pc3_var:.4f}")
print(f"  cumulative PC1-3 : {(pc1_var + pc2_var + pc3_var):.4f}")


def project_on_frozen_baseline_pca_3d(X):
    X = np.asarray(X, dtype=np.float32)
    finite = np.isfinite(X).all(axis=1)

    Xf = X[finite]

    if L2_NORMALISE_BEFORE_PCA:
        Xp = l2_normalize(Xf)
    else:
        Xp = Xf.copy()

    Xp = Xp - frozen_baseline_mean
    Z = fixed_pca.transform(Xp)

    return Z, finite


# ============================================================
# 7. Project train, val and external into 3D PCA
# ============================================================

Z_train, keep_train = project_on_frozen_baseline_pca_3d(X_train)
Z_val, keep_val = project_on_frozen_baseline_pca_3d(X_val)
Z_ext, keep_ext = project_on_frozen_baseline_pca_3d(X_ext)

df_train = df_train.loc[keep_train].reset_index(drop=True)
df_val = df_val.loc[keep_val].reset_index(drop=True)
df_ext = df_ext.loc[keep_ext].reset_index(drop=True)

X_train_kept = X_train[keep_train]
X_val_kept = X_val[keep_val]
X_ext_kept = X_ext[keep_ext]

df_train["PC1"] = Z_train[:, 0]
df_train["PC2"] = Z_train[:, 1]
df_train["PC3"] = Z_train[:, 2]

df_val["PC1"] = Z_val[:, 0]
df_val["PC2"] = Z_val[:, 1]
df_val["PC3"] = Z_val[:, 2]

df_ext["PC1"] = Z_ext[:, 0]
df_ext["PC2"] = Z_ext[:, 1]
df_ext["PC3"] = Z_ext[:, 2]

pca_df = pd.concat([df_train, df_val, df_ext], ignore_index=True)

pca_df = infer_label_from_path_or_patient_id(pca_df)
pca_df["class_name"] = standardise_class_labels(pca_df["class_name"].values)
pca_df["label"] = pca_df["class_name"].values

print("\nClass counts by split after final standardisation:")
print(pd.crosstab(pca_df["class_name"], pca_df["split"]))


# ============================================================
# 8. Class colour map for 3D scatter
# ============================================================

unique_labels_observed = sorted(pca_df["class_name"].astype(str).unique())

ordered_labels = [
    c for c in CLASS_ORDER
    if c in unique_labels_observed
] + [
    c for c in unique_labels_observed
    if c not in CLASS_ORDER
]

LABEL_TO_COLOR = {}

for c in CLASS_ORDER:
    if c in ordered_labels:
        LABEL_TO_COLOR[c] = FIXED_CLASS_COLOURS[c]

fallback_cmap = plt.get_cmap("tab20")

for i, lab in enumerate(ordered_labels):
    if lab not in LABEL_TO_COLOR:
        LABEL_TO_COLOR[lab] = fallback_cmap(i % fallback_cmap.N)

unique_labels_global = ordered_labels

print("\nFinal class colour map:")
for lab in unique_labels_global:
    print(f"  {lab}: {LABEL_TO_COLOR[lab]}")


# ============================================================
# 9. 3D PCA plot
# ============================================================

Z_all = pca_df[["PC1", "PC2", "PC3"]].values

x_min, x_max = np.nanpercentile(Z_all[:, 0], [1, 99])
y_min, y_max = np.nanpercentile(Z_all[:, 1], [1, 99])
z_min, z_max = np.nanpercentile(Z_all[:, 2], [1, 99])

x_pad = 0.10 * (x_max - x_min + 1e-12)
y_pad = 0.10 * (y_max - y_min + 1e-12)
z_pad = 0.10 * (z_max - z_min + 1e-12)

FIXED_XLIM = (x_min - x_pad, x_max + x_pad)
FIXED_YLIM = (y_min - y_pad, y_max + y_pad)
FIXED_ZLIM = (z_min - z_pad, z_max + z_pad)

fig = plt.figure(figsize=FIGSIZE_3D, constrained_layout=False)
ax = fig.add_subplot(111, projection="3d")

for split in ["train", "val", "external"]:
    for lab in unique_labels_global:
        g = pca_df[
            (pca_df["split"] == split) &
            (pca_df["class_name"].astype(str) == str(lab))
        ]

        if len(g) == 0:
            continue

        ax.scatter(
            g["PC1"],
            g["PC2"],
            g["PC3"],
            s=SPLIT_SIZES[split],
            alpha=SPLIT_ALPHA[split],
            marker=SPLIT_MARKERS[split],
            color=LABEL_TO_COLOR[lab],
            edgecolors="black" if split in ["val", "external"] else "none",
            linewidths=1.0 if split == "external" else (0.6 if split == "val" else 0.0),
            depthshade=False,
        )

ax.set_xlabel(f"PC1 ({pc1_var * 100:.1f}%)", fontsize=9, labelpad=8)
ax.set_ylabel(f"PC2 ({pc2_var * 100:.1f}%)", fontsize=9, labelpad=8)
ax.set_zlabel(f"PC3 ({pc3_var * 100:.1f}%)", fontsize=9, labelpad=8)

# ax.set_title(
#     "SSL-last train/validation and external cohort projected into frozen-DINO 3D PCA space",
#     fontsize=11,
#     fontweight="bold",
# )

ax.set_xlim(FIXED_XLIM)
ax.set_ylim(FIXED_YLIM)
ax.set_zlim(FIXED_ZLIM)

ax.view_init(elev=22, azim=42)

ax.xaxis.pane.set_alpha(0.05)
ax.yaxis.pane.set_alpha(0.05)
ax.zaxis.pane.set_alpha(0.05)
ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.25)

try:
    ax.set_box_aspect((1.0, 1.0, 0.85))
except Exception:
    pass

split_handles = [
    Line2D([0], [0], marker="o", color="black", label="train",
           markerfacecolor="white", markeredgecolor="black", linestyle="None", markersize=7),
    Line2D([0], [0], marker="^", color="black", label="val",
           markerfacecolor="white", markeredgecolor="black", linestyle="None", markersize=7),
    Line2D([0], [0], marker="X", color="black", label="external",
           markerfacecolor="white", markeredgecolor="black", linestyle="None", markersize=8),
]

class_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=str(lab),
        markerfacecolor=LABEL_TO_COLOR[lab],
        markeredgecolor="black",
        markersize=8,
        linestyle="None",
    )
    for lab in unique_labels_global
]

# ax.legend(
#     handles=split_handles + class_handles,
#     loc="upper right",
#     fontsize=8,
#     frameon=True,
#     title="Legend",
#     title_fontsize=9,
# )

plt.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.92)

out_png = OUT_DIR / "SSL_LAST_train_val_external_PROJECTED_ON_FROZEN_BASELINE_3D_PCA.png"
out_pdf = OUT_DIR / "SSL_LAST_train_val_external_PROJECTED_ON_FROZEN_BASELINE_3D_PCA.pdf"
out_svg = OUT_DIR / "SSL_LAST_train_val_external_PROJECTED_ON_FROZEN_BASELINE_3D_PCA.svg"

plt.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", pad_inches=0.02)
plt.savefig(out_pdf, bbox_inches="tight", pad_inches=0.02)
plt.savefig(out_svg, bbox_inches="tight", pad_inches=0.02)
plt.show()

print("\nSaved 3D PNG:", out_png)
print("Saved 3D PDF:", out_pdf)
print("Saved 3D SVG:", out_svg)


# ============================================================
# 10. Save PCA scores
# ============================================================

pca_scores_csv = OUT_DIR / "SSL_LAST_train_val_external_PROJECTED_ON_FROZEN_BASELINE_3D_PCA_scores.csv"
pca_df.to_csv(pca_scores_csv, index=False)

print("Saved 3D PCA scores:", pca_scores_csv)


# ============================================================
# 11. External vs train+val centroid-distance analysis
# ============================================================

def euclidean_distance_matrix(A, B):
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)
    diff = A[:, None, :] - B[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))


def cosine_distance_matrix(A, B, eps=1e-12):
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)

    A = A / (np.linalg.norm(A, axis=1, keepdims=True) + eps)
    B = B / (np.linalg.norm(B, axis=1, keepdims=True) + eps)

    sim = A @ B.T
    return 1.0 - sim


def build_original_embedding_df():
    X_original = np.vstack([
        X_train_kept,
        X_val_kept,
        X_ext_kept,
    ]).astype(np.float32)

    df_original = pca_df.copy().reset_index(drop=True)

    if len(df_original) != len(X_original):
        raise RuntimeError(
            f"Embedding dataframe length mismatch: df={len(df_original)}, X={len(X_original)}"
        )

    return df_original, X_original


def ordered_classes_from_labels(labels):
    labels = np.asarray(labels).astype(str)
    observed = sorted(set(labels))

    return [
        c for c in CLASS_ORDER
        if c in observed
    ] + [
        c for c in observed
        if c not in CLASS_ORDER
    ]


def compute_group_class_centroids(df, X, split_values, group_name, space_name):
    """
    Computes class centroids for a specified group, e.g.
      train+val or external.
    """
    keep = df["split"].astype(str).isin(split_values).values
    df_g = df.loc[keep].reset_index(drop=True)
    X_g = X[keep]

    if len(df_g) == 0:
        return pd.DataFrame(), np.zeros((0, X.shape[1]), dtype=np.float32), []

    labels = df_g["class_name"].astype(str).values
    classes = ordered_classes_from_labels(labels)

    rows = []
    centroid_vectors = []
    centroid_classes = []

    for cls in classes:
        mask = labels == cls
        n = int(mask.sum())

        if n < MIN_CLASS_N_FOR_CENTROID:
            print(
                f"[centroid skip] {group_name} | {space_name} | {cls}: "
                f"n={n} < {MIN_CLASS_N_FOR_CENTROID}"
            )
            continue

        X_cls = X_g[mask]
        centroid = X_cls.mean(axis=0)

        centroid_vectors.append(centroid)
        centroid_classes.append(cls)

        row = {
            "group": group_name,
            "space": space_name,
            "class_name": cls,
            "n": n,
        }

        for j, val in enumerate(centroid):
            row[f"centroid_dim_{j:04d}"] = float(val)

        rows.append(row)

    df_centroids = pd.DataFrame(rows)

    if len(centroid_vectors) == 0:
        C = np.zeros((0, X.shape[1]), dtype=np.float32)
    else:
        C = np.vstack(centroid_vectors).astype(np.float32)

    return df_centroids, C, centroid_classes


def compute_external_vs_trainval_distances(
    df,
    X,
    space_name,
    metric,
):
    """
    Computes external class centroid distance to train+val class centroids.

    Rows:
      external class centroids

    Columns:
      train+val class centroids
    """
    df_trainval_cent, C_trainval, classes_trainval = compute_group_class_centroids(
        df=df,
        X=X,
        split_values=["train", "val"],
        group_name="train_val",
        space_name=space_name,
    )

    df_external_cent, C_external, classes_external = compute_group_class_centroids(
        df=df,
        X=X,
        split_values=["external"],
        group_name="external",
        space_name=space_name,
    )

    if C_trainval.shape[0] == 0 or C_external.shape[0] == 0:
        return (
            df_trainval_cent,
            df_external_cent,
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
        )

    if metric == "euclidean":
        D = euclidean_distance_matrix(C_external, C_trainval)

    elif metric == "cosine":
        D = cosine_distance_matrix(C_external, C_trainval)

    else:
        raise ValueError(f"Unsupported metric: {metric}")

    rows_long = []

    for i, ext_cls in enumerate(classes_external):
        for j, tv_cls in enumerate(classes_trainval):
            rows_long.append({
                "space": space_name,
                "metric": metric,
                "external_class": ext_cls,
                "train_val_class": tv_cls,
                "external_n": int(
                    df_external_cent.loc[
                        df_external_cent["class_name"] == ext_cls,
                        "n",
                    ].iloc[0]
                ),
                "train_val_n": int(
                    df_trainval_cent.loc[
                        df_trainval_cent["class_name"] == tv_cls,
                        "n",
                    ].iloc[0]
                ),
                "centroid_distance": float(D[i, j]),
                "same_class": bool(ext_cls == tv_cls),
            })

    df_long = pd.DataFrame(rows_long)

    df_matrix = pd.DataFrame(
        D,
        index=classes_external,
        columns=classes_trainval,
    )

    df_matrix.insert(0, "external_class", classes_external)
    df_matrix.insert(0, "metric", metric)
    df_matrix.insert(0, "space", space_name)

    summary_rows = []

    for ext_cls in classes_external:
        d_ext = df_long[df_long["external_class"] == ext_cls].copy()

        nearest = d_ext.loc[d_ext["centroid_distance"].idxmin()]
        farthest = d_ext.loc[d_ext["centroid_distance"].idxmax()]

        same = d_ext[d_ext["same_class"] == True]

        if len(same) > 0:
            same_class_distance = float(same["centroid_distance"].iloc[0])
        else:
            same_class_distance = np.nan

        summary_rows.append({
            "space": space_name,
            "metric": metric,
            "external_class": ext_cls,
            "external_n": int(nearest["external_n"]),
            "nearest_train_val_class": nearest["train_val_class"],
            "nearest_centroid_distance": float(nearest["centroid_distance"]),
            "farthest_train_val_class": farthest["train_val_class"],
            "farthest_centroid_distance": float(farthest["centroid_distance"]),
            "same_class_train_val_distance": same_class_distance,
            "mean_distance_to_train_val_centroids": float(d_ext["centroid_distance"].mean()),
        })

    df_summary = pd.DataFrame(summary_rows)

    return (
        df_trainval_cent,
        df_external_cent,
        df_long,
        df_matrix,
        df_summary,
    )


def plot_external_vs_trainval_heatmap(
    df_long,
    out_png,
    title,
    cmap_name="Greys",
):
    """
    Grayscale heatmap.

    Rows:
      external class centroids

    Columns:
      train+val class centroids

    Darker = larger centroid distance.
    """
    if df_long.empty:
        return

    external_classes = ordered_classes_from_labels(df_long["external_class"].values)
    trainval_classes = ordered_classes_from_labels(df_long["train_val_class"].values)

    D = np.full(
        (len(external_classes), len(trainval_classes)),
        np.nan,
        dtype=float,
    )

    for _, row in df_long.iterrows():
        i = external_classes.index(row["external_class"])
        j = trainval_classes.index(row["train_val_class"])
        D[i, j] = float(row["centroid_distance"])

    values = D[~np.isnan(D)]

    if len(values) > 0:
        vmin = float(np.nanmin(values))
        vmax = float(np.nanmax(values))
    else:
        vmin, vmax = 0.0, 1.0

    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-6

    fig_h, ax_h = plt.subplots(figsize=(7.2, 4.8))

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color="white")

    im = ax_h.imshow(
        D,
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
    )

    ax_h.set_xticks(np.arange(len(trainval_classes)))
    ax_h.set_yticks(np.arange(len(external_classes)))

    ax_h.set_xticklabels(
        [f"train+val\n{c}" for c in trainval_classes],
        rotation=45,
        ha="right",
        fontsize=9,
    )

    ax_h.set_yticklabels(
        [f"external\n{c}" for c in external_classes],
        fontsize=9,
    )

    ax_h.set_xticks(np.arange(-0.5, len(trainval_classes), 1), minor=True)
    ax_h.set_yticks(np.arange(-0.5, len(external_classes), 1), minor=True)

    ax_h.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=1.6,
    )

    ax_h.tick_params(which="minor", bottom=False, left=False)

    for i in range(D.shape[0]):
        for j in range(D.shape[1]):
            val = D[i, j]

            if np.isnan(val):
                txt = ""
                colour = "black"
            else:
                txt = f"{val:.3f}"
                norm_val = (val - vmin) / (vmax - vmin + 1e-12)
                colour = "white" if norm_val > 0.55 else "black"

            ax_h.text(
                j,
                i,
                txt,
                ha="center",
                va="center",
                fontsize=8,
                color=colour,
            )

    ax_h.set_title(
        title,
        fontsize=11,
        fontweight="bold",
        pad=10,
    )

    cbar = plt.colorbar(im, ax=ax_h, fraction=0.046, pad=0.04)

    metric = str(df_long["metric"].iloc[0]).lower()

    if metric == "cosine":
        cbar.set_label("Cosine centroid distance", fontsize=9)
    else:
        cbar.set_label("Euclidean centroid distance", fontsize=9)

    cbar.ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig_h)


# ============================================================
# 12. Run external vs train+val centroid analysis
# ============================================================

embedding_df, X_original_all = build_original_embedding_df()

if L2_NORMALISE_BEFORE_PCA:
    X_embedding_for_centroids = l2_normalize(X_original_all)
    embedding_space_name = "embedding_l2_normalised"
else:
    X_embedding_for_centroids = X_original_all.astype(np.float32)
    embedding_space_name = "embedding_raw"

Z_pca_all = pca_df[["PC1", "PC2", "PC3"]].values.astype(np.float32)

all_centroids = []
all_long = []
all_matrix = []
all_summary = []

# ------------------------------------------------------------
# A. Original embedding space
# ------------------------------------------------------------

for metric in ["euclidean", "cosine"]:
    (
        df_trainval_cent,
        df_external_cent,
        df_long,
        df_matrix,
        df_summary,
    ) = compute_external_vs_trainval_distances(
        df=embedding_df,
        X=X_embedding_for_centroids,
        space_name=embedding_space_name,
        metric=metric,
    )

    all_centroids.extend([df_trainval_cent, df_external_cent])
    all_long.append(df_long)
    all_matrix.append(df_matrix)
    all_summary.append(df_summary)

    long_csv = CENTROID_OUT_DIR / f"external_vs_trainval_centroid_distances_{embedding_space_name}_{metric}.csv"
    matrix_csv = CENTROID_OUT_DIR / f"external_vs_trainval_centroid_matrix_{embedding_space_name}_{metric}.csv"
    summary_csv = CENTROID_OUT_DIR / f"external_vs_trainval_centroid_summary_{embedding_space_name}_{metric}.csv"

    df_long.to_csv(long_csv, index=False)
    df_matrix.to_csv(matrix_csv, index=False)
    df_summary.to_csv(summary_csv, index=False)

    heatmap_png = CENTROID_OUT_DIR / f"external_vs_trainval_centroid_matrix_{embedding_space_name}_{metric}_GREY.png"

    plot_external_vs_trainval_heatmap(
        df_long,
        heatmap_png,
        title=f"External cohort vs train+validation centroids\n{embedding_space_name} | {metric}",
        cmap_name="Greys",
    )

    print("\nSaved:")
    print(" ", long_csv)
    print(" ", matrix_csv)
    print(" ", summary_csv)
    print(" ", heatmap_png)


# ------------------------------------------------------------
# B. Frozen-baseline 3D PCA space
# ------------------------------------------------------------

(
    df_trainval_cent,
    df_external_cent,
    df_long,
    df_matrix,
    df_summary,
) = compute_external_vs_trainval_distances(
    df=pca_df,
    X=Z_pca_all,
    space_name="frozen_baseline_3D_PCA",
    metric="euclidean",
)

all_centroids.extend([df_trainval_cent, df_external_cent])
all_long.append(df_long)
all_matrix.append(df_matrix)
all_summary.append(df_summary)

long_csv = CENTROID_OUT_DIR / "external_vs_trainval_centroid_distances_frozen_baseline_3D_PCA_euclidean.csv"
matrix_csv = CENTROID_OUT_DIR / "external_vs_trainval_centroid_matrix_frozen_baseline_3D_PCA_euclidean.csv"
summary_csv = CENTROID_OUT_DIR / "external_vs_trainval_centroid_summary_frozen_baseline_3D_PCA_euclidean.csv"

df_long.to_csv(long_csv, index=False)
df_matrix.to_csv(matrix_csv, index=False)
df_summary.to_csv(summary_csv, index=False)

heatmap_png = CENTROID_OUT_DIR / "external_vs_trainval_centroid_matrix_frozen_baseline_3D_PCA_euclidean_GREY.png"

plot_external_vs_trainval_heatmap(
    df_long,
    heatmap_png,
    title="External cohort vs train+validation centroids\nfrozen-baseline 3D PCA | euclidean",
    cmap_name="Greys",
)

print("\nSaved:")
print(" ", long_csv)
print(" ", matrix_csv)
print(" ", summary_csv)
print(" ", heatmap_png)


# ============================================================
# 13. Save combined centroid outputs
# ============================================================

def concat_nonempty(dfs):
    dfs = [df for df in dfs if df is not None and len(df) > 0]

    if len(dfs) == 0:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


df_centroids_all = concat_nonempty(all_centroids)
df_long_all = concat_nonempty(all_long)
df_matrix_all = concat_nonempty(all_matrix)
df_summary_all = concat_nonempty(all_summary)

centroids_all_csv = CENTROID_OUT_DIR / "ALL_external_and_trainval_centroids.csv"
long_all_csv = CENTROID_OUT_DIR / "ALL_external_vs_trainval_centroid_distances_long.csv"
matrix_all_csv = CENTROID_OUT_DIR / "ALL_external_vs_trainval_centroid_matrices.csv"
summary_all_csv = CENTROID_OUT_DIR / "ALL_external_vs_trainval_centroid_summary.csv"

df_centroids_all.to_csv(centroids_all_csv, index=False)
df_long_all.to_csv(long_all_csv, index=False)
df_matrix_all.to_csv(matrix_all_csv, index=False)
df_summary_all.to_csv(summary_all_csv, index=False)

print("\nSaved combined centroid outputs:")
print("  centroids:", centroids_all_csv)
print("  long     :", long_all_csv)
print("  matrix   :", matrix_all_csv)
print("  summary  :", summary_all_csv)


# ============================================================
# 14. Save manifest
# ============================================================

manifest = {
    "analysis": "external cohort centroid distance against internal train+validation cohort",
    "pca_reference": "frozen_dino_baseline_train_val",
    "pca_components": 3,
    "frozen_train_path": str(frozen_train_path),
    "frozen_val_path": str(frozen_val_path),
    "ssl_last_dir": str(SSL_LAST_DIR),
    "external_dir": str(EXTERNAL_DIR),
    "external_classes_kept": EXTERNAL_CLASSES_TO_KEEP,
    "l2_normalise_before_pca": bool(L2_NORMALISE_BEFORE_PCA),
    "min_class_n_for_centroid": int(MIN_CLASS_N_FOR_CENTROID),
    "pc1_variance_ratio": float(pc1_var),
    "pc2_variance_ratio": float(pc2_var),
    "pc3_variance_ratio": float(pc3_var),
    "pc1_pc2_pc3_cumulative_variance_ratio": float(pc1_var + pc2_var + pc3_var),
    "n_train": int((pca_df["split"] == "train").sum()),
    "n_val": int((pca_df["split"] == "val").sum()),
    "n_external": int((pca_df["split"] == "external").sum()),
    "class_counts_by_split": pd.crosstab(
        pca_df["class_name"],
        pca_df["split"],
    ).to_dict(),
    "class_colour_map": {
        str(k): str(v) for k, v in LABEL_TO_COLOR.items()
    },
    "centroid_heatmap_cmap": "Greys",
    "outputs": {
        "pca_png": str(out_png),
        "pca_pdf": str(out_pdf),
        "pca_svg": str(out_svg),
        "pca_scores_csv": str(pca_scores_csv),
        "combined_centroids_csv": str(centroids_all_csv),
        "combined_external_vs_trainval_long_csv": str(long_all_csv),
        "combined_external_vs_trainval_matrix_csv": str(matrix_all_csv),
        "combined_external_vs_trainval_summary_csv": str(summary_all_csv),
    },
}

manifest_json = OUT_DIR / "external_vs_trainval_centroid_distance_manifest.json"

with open(manifest_json, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_json)

if len(df_summary_all) > 0:
    print("\nExternal cohort centroid summary:")
    display_cols = [
        "space",
        "metric",
        "external_class",
        "external_n",
        "nearest_train_val_class",
        "nearest_centroid_distance",
        "farthest_train_val_class",
        "farthest_centroid_distance",
        "same_class_train_val_distance",
        "mean_distance_to_train_val_centroids",
    ]

    print(df_summary_all[display_cols].to_string(index=False))

print("\nDone.")